# AI Lab Project: Hypoxia Prediction from Single-Cell RNA-seq Data

This notebook merges the original EDA, Unsupervised Learning and Supervised Learning notebooks into one coherent workflow. The objective is to understand whether hypoxic cancer cells can be recognized from single-cell RNA-seq gene expression profiles, and whether the biological signal is consistent across the HCC1806 and MCF7 breast cancer cell lines.

## Project outline

The workflow follows the original structure of the project:

- Exploratory Data Analysis: inspect raw SmartSeq data, metadata, sparsity, quality-control metrics, filtering choices and the comparison with the professor-provided filtered files.
- Unsupervised Learning: study whether hypoxia appears as a spontaneous structure in the data, while checking normalization, confounders, clustering stability, NMF programs, pathway enrichment and cross-line projection.
- Supervised Learning: build classification pipelines for SmartSeq normalised data, our QCd data and DropSeq data, then evaluate within-line, combined and transfer scenarios.
- Biological interpretation and conclusions: connect model performance to hypoxia biology, pathway-level evidence, limitations and possible improvements.

## Libraries and global settings

In [ ]:
# Core Python utilities
import os
import math
import re
import json
import warnings
from pathlib import Path

# Data handling and numerical computing
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Scientific statistics
from scipy.stats import (
    median_abs_deviation,
    skew,
    kurtosis,
    ttest_ind,
    hypergeom,
)
from statsmodels.stats.multitest import multipletests

# Single-cell analysis
import scanpy as sc
import anndata as ad

# Machine learning: preprocessing, decomposition and feature selection
from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    RandomizedSearchCV,
    learning_curve,
)
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, SparsePCA, NMF
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.manifold import TSNE

# Machine learning: clustering and neighbours
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors

# Machine learning: classifiers
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    StackingClassifier,
)
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

# Machine learning: metrics and diagnostics
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    adjusted_rand_score,
    adjusted_mutual_info_score,
    silhouette_score,
)

# Hyperparameter tuning and enrichment analysis
import optuna
import gseapy as gp

# Reproducibility and global style
RANDOM_STATE = 42
UNSUPERVISED_RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)
RNG = np.random.RandomState(UNSUPERVISED_RANDOM_STATE)

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="Features .* are constant.*")
warnings.filterwarnings("ignore", message="invalid value encountered.*")

optuna.logging.set_verbosity(optuna.logging.WARNING)
sc.settings.verbosity = 0
sns.set_context("notebook")
plt.rcParams["figure.dpi"] = 110


def safe_f_classif(X, y):
    """ANOVA F-test wrapper used by SelectKBest.
    Constant genes inside CV folds are non-informative and can generate
    repeated warnings or NaN scores, so their scores are set to 0 and
    their p-values to 1.
    """
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning)
        warnings.filterwarnings("ignore", category=RuntimeWarning)
        warnings.filterwarnings("ignore", message="Features .* are constant.*")
        warnings.filterwarnings("ignore", message="invalid value encountered.*")
        with np.errstate(divide="ignore", invalid="ignore"):
            scores, pvalues = f_classif(X, y)

    scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
    pvalues = np.nan_to_num(pvalues, nan=1.0, posinf=1.0, neginf=1.0)
    return scores, pvalues


## Data loading

All datasets needed by the three sections are loaded here, once. This prevents the EDA, Unsupervised Learning and Supervised Learning sections from repeating the same path definitions and loading logic.

The SmartSeq part uses the unfiltered matrices, the professor-provided filtered matrices, the filtered/normalised labelled matrices, the anonymous test matrices and the QCd matrices produced by our filtering. The DropSeq files are loaded for the supervised-learning comparison only.

In [ ]:
# ---------------------------------------------------------------------
# File paths
# ---------------------------------------------------------------------
PROJECT_DIR = Path(".")
SMARTSEQ_DIR = PROJECT_DIR
DROPSEQ_DIR = PROJECT_DIR / "DropSeq"

# SmartSeq unfiltered matrices and metadata, used mainly in EDA and Unsupervised Learning
HCC_UNFILTERED_FILE = SMARTSEQ_DIR / "HCC1806_SmartS_Unfiltered_Data.txt"
MCF_UNFILTERED_FILE = SMARTSEQ_DIR / "MCF7_SmartS_Unfiltered_Data.txt"
HCC_METADATA_FILE = SMARTSEQ_DIR / "HCC1806_SmartS_MetaData.tsv"
MCF_METADATA_FILE = SMARTSEQ_DIR / "MCF7_SmartS_MetaData.tsv"

# Professor-provided filtered SmartSeq matrices, used for the filtering comparison in EDA
HCC_FILTERED_FILE = SMARTSEQ_DIR / "HCC1806_SmartS_Filtered_Data.txt"
MCF_FILTERED_FILE = SMARTSEQ_DIR / "MCF7_SmartS_Filtered_Data.txt"

# SmartSeq filtered/normalised train and anonymous test matrices, used in Supervised Learning
DATA_DIR = str(SMARTSEQ_DIR) + "/"
HCC_TRAIN_FILE = SMARTSEQ_DIR / "HCC1806_SmartS_Filtered_Normalised_3000_Data_train.txt"
MCF_TRAIN_FILE = SMARTSEQ_DIR / "MCF7_SmartS_Filtered_Normalised_3000_Data_train.txt"
HCC_ANON_FILE = SMARTSEQ_DIR / "HCC1806_SmartS_Filtered_Normalised_3000_Data_test_anonim.txt"
MCF_ANON_FILE = SMARTSEQ_DIR / "MCF7_SmartS_Filtered_Normalised_3000_Data_test_anonim.txt"

# SmartSeq QCd matrices produced by our filtering, used in Supervised Learning
QCD_DATA_DIR = str(PROJECT_DIR) + "/"
HCC_QCD_FILE = PROJECT_DIR / "HCC1806_counts_QCd.csv"
MCF_QCD_FILE = PROJECT_DIR / "MCF7_counts_QCd.csv"

# DropSeq filtered/normalised matrices, used only in Supervised Learning
DROPSEQ_DATA_DIR = str(DROPSEQ_DIR) + "/"
DROP_HCC_TRAIN_FILE = DROPSEQ_DIR / "HCC1806_Filtered_Normalised_3000_Data_train.txt"
DROP_MCF_TRAIN_FILE = DROPSEQ_DIR / "MCF7_Filtered_Normalised_3000_Data_train.txt"
DROP_HCC_ANON_FILE = DROPSEQ_DIR / "HCC1806_Filtered_Normalised_3000_Data_test_anonim.txt"
DROP_MCF_ANON_FILE = DROPSEQ_DIR / "MCF7_Filtered_Normalised_3000_Data_test_anonim.txt"

# ---------------------------------------------------------------------
# General helper functions
# ---------------------------------------------------------------------
LABEL_MAP = {"Normoxia": 0, "Hypoxia": 1}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}


def read_gene_by_cell_matrix(path: Path) -> pd.DataFrame:
    """Read a whitespace-separated expression matrix with genes in rows and cells in columns."""
    df = pd.read_csv(path, sep=r"\s+", engine="python", header=0, index_col=0, quotechar='"')
    df.index = df.index.astype(str).str.strip('"')
    df.columns = df.columns.astype(str).str.strip('"')
    df.index.name = "Gene"
    df.columns.name = "Filename"
    return df.apply(pd.to_numeric, errors="coerce")


def extract_condition_from_cell_name(cell_name: str) -> str:
    """Extract Normoxia/Hypoxia label from a cell/sample name."""
    name = str(cell_name).lower()
    if "hypoxia" in name or "hypo" in name:
        return "Hypoxia"
    if "normoxia" in name or "norm" in name:
        return "Normoxia"
    raise ValueError(f"Cannot extract condition from cell name: {cell_name}")


def read_expression_matrix(path: Path) -> pd.DataFrame:
    """Read a whitespace-separated gene expression matrix with genes in rows and cells in columns."""
    return read_gene_by_cell_matrix(path)


def prepare_labelled_dataset(path: Path, cell_line: str):
    """Return X, y_numeric, y_label and metadata for one labelled expression file."""
    expr = read_expression_matrix(path)
    X = expr.T.copy()
    X.index.name = "cell_id"
    X.columns.name = "gene"
    y_label = pd.Series(
        [extract_condition_from_cell_name(cell_id) for cell_id in X.index],
        index=X.index,
        name="condition_label",
    )
    y = y_label.map(LABEL_MAP).astype(int)
    metadata = pd.DataFrame({
        "cell_id": X.index,
        "cell_line": cell_line,
        "condition_label": y_label.values,
        "condition": y.values,
    }).set_index("cell_id")
    return X, y, y_label, metadata


def prepare_anonymous_dataset(path: Path):
    """Return X for an anonymous expression file. No labels are available."""
    expr = read_expression_matrix(path)
    X = expr.T.copy()
    X.index.name = "cell_id"
    X.columns.name = "gene"
    return X


def read_qcd_counts_matrix(path: Path) -> pd.DataFrame:
    """Read our QCd count matrix with cells in rows and genes in columns."""
    df = pd.read_csv(path)
    cell_id_col = "Unnamed: 0" if "Unnamed: 0" in df.columns else df.columns[0]
    cell_ids = df[cell_id_col].astype(str).str.strip('"').str.strip()
    X = df.drop(columns=[cell_id_col]).copy()
    X.index = cell_ids
    X.index.name = "cell_id"
    X.columns = X.columns.astype(str).str.strip('"').str.strip()
    X.columns.name = "gene"
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.loc[:, X.notna().any(axis=0)]
    return X


def prepare_qcd_labelled_dataset(path: Path, cell_line: str):
    """Return X, y_numeric, y_label and metadata for one QCd count file."""
    X = read_qcd_counts_matrix(path)
    y_label = pd.Series(
        [extract_condition_from_cell_name(cell_id) for cell_id in X.index],
        index=X.index,
        name="condition_label",
    )
    y = y_label.map(LABEL_MAP).astype(int)
    metadata = pd.DataFrame({
        "cell_id": X.index,
        "cell_line": cell_line,
        "dataset_version": "QCd",
        "condition_label": y_label.values,
        "condition": y.values,
    }).set_index("cell_id")
    return X, y, y_label, metadata

# ---------------------------------------------------------------------
# EDA datasets: unfiltered matrices, metadata, merged matrices and filtered matrices
# ---------------------------------------------------------------------
matrix_U1 = read_gene_by_cell_matrix(HCC_UNFILTERED_FILE)
matrix_U1_T = matrix_U1.T.copy()
metadata1 = pd.read_csv(HCC_METADATA_FILE, sep="\t")
final_U1 = metadata1.merge(matrix_U1_T.reset_index(), on="Filename", how="inner")
final_U1.to_csv("HCC1806_final_unfiltered.csv", index=False)

matrix_U2 = read_gene_by_cell_matrix(MCF_UNFILTERED_FILE)
matrix_U2_T = matrix_U2.T.copy()
metadata2 = pd.read_csv(MCF_METADATA_FILE, sep="\t")
final_U2 = metadata2.merge(matrix_U2_T.reset_index(), on="Filename", how="inner")
final_U2.to_csv("MCF7_final_unfiltered.csv", index=False)

EDAdraft_df = final_U1.copy()
metadata_cols = ["Filename", "Cell Line", "PCR Plate", "Pos", "Condition", "Hours", "Cell name", "PreprocessingTag", "ProcessingComments"]
gene_cols = [col for col in EDAdraft_df.columns if col not in metadata_cols]
eda_metadata_cols = ["PCR Plate", "Pos", "Condition", "Cell name"]
EDA_df = EDAdraft_df[eda_metadata_cols + list(gene_cols)]

EDAdraft_df_MCF7 = final_U2.copy()
metadata_cols_MCF7_initial = ["Filename", "Cell Line", "Lane", "Pos", "Condition", "Hours", "Cell name", "PreprocessingTag", "ProcessingComments"]
gene_cols_MCF7_initial = [col for col in EDAdraft_df_MCF7.columns if col not in metadata_cols_MCF7_initial]
eda_metadata_cols_MCF7 = ["Lane", "Pos", "Condition", "Cell name"]
EDA_df_MCF7 = EDAdraft_df_MCF7[eda_metadata_cols_MCF7 + list(gene_cols_MCF7_initial)]

filtered_data_Buffa = read_gene_by_cell_matrix(HCC_FILTERED_FILE)
filtered_data_Buffa_T = filtered_data_Buffa.T.copy()
filtered_data_Buffa_MCF7 = read_gene_by_cell_matrix(MCF_FILTERED_FILE)
filtered_data_Buffa_T_MCF7 = filtered_data_Buffa_MCF7.T.copy()

# ---------------------------------------------------------------------
# Unsupervised datasets: raw unfiltered merged count tables converted to AnnData
# ---------------------------------------------------------------------
META_COLS = {
    "Cell", "Filename", "Cell Line", "PCR Plate", "Lane", "Pos", "Condition",
    "Hours", "Cell name", "PreprocessingTag", "ProcessingComments",
}

BUFFA = (
    "NDRG1 LDHA PGK1 TPI1 PGAM1 PFKP ENO1 SLC2A1 PNP BNIP3 PSRC1 SLC16A1 DDIT4 "
    "ADM HK2 SEC61G AK4 CA9 P4HA1 ACOT7 GPI TUBB6 TUBA1C CDKN3 CTSV VEGFA TUBA1A "
    "LRRC42 PSMA7 GAPDH CHCHD2 YKT6 MIF MAP7D1 MRPL15 MRPL13 MCTS1 UTP11 KIF4A "
    "HILPDA MRGBP KIF20A MRPS17 ESRP1 SHCBP1 SLC25A32 CORO1C ANLN MAD2L2 ANKRD37"
).split()
BUFFA_SET = set(BUFFA)

S_GENES = (
    "MCM5 PCNA TYMS FEN1 MCM2 MCM4 RRM1 UNG GINS2 MCM6 CDCA7 DTL PRIM1 UHRF1 HELLS "
    "RFC2 RPA2 NASP RAD51AP1 GMNN WDR76 SLBP CCNE2 UBR7 POLD3 MSH2 ATAD2 RAD51 RRM2 "
    "CDC45 CDC6 EXO1 TIPIN DSCC1 BLM CASP8AP2 USP1 CLSPN POLA1 CHAF1B BRIP1 E2F8"
).split()
G2M_GENES = (
    "HMGB2 CDK1 NUSAP1 UBE2C BIRC5 TPX2 TOP2A NDC80 CKS2 NUF2 CKS1B MKI67 TMPO CENPF "
    "TACC3 FAM64A SMC4 CCNB2 CKAP2L CKAP2 AURKB BUB1 KIF11 ANP32E TUBB4B GTSE1 KIF20B "
    "HJURP CDCA3 HN1 CDC20 TTK CDC25C KIF2C RANGAP1 NCAPD2 DLGAP5 CDCA2 CDCA8 ECT2 "
    "KIF23 HMMR AURKA PSRC1 ANLN LBR CKAP5 CENPE CTCF NEK2 G2E3 GAS2L3 CBX5 CENPA"
).split()

QC = dict(min_total_counts=1000, min_detected_genes=500, max_pct_mito=0.06)


def make_unsupervised_table(final_df: pd.DataFrame) -> pd.DataFrame:
    """Convert the EDA merged table into the cells x genes format used by the unsupervised loader."""
    df = final_df.copy()
    df = df.rename(columns={"Filename": "Cell"})
    return df


def load_counts_from_dataframe(df: pd.DataFrame, n_hvg: int = 3000):
    """Raw merged counts -> QC'd AnnData with counts and log-normalised layers."""
    df = df.copy()
    df["Cell"] = df["Cell"].astype(str).str.replace('"', '', regex=False).str.strip()
    genes = np.array([c for c in df.columns if c not in META_COLS])
    cond_raw = df["Condition"].astype(str).str.strip()
    cond = np.where(
        cond_raw.str.startswith("Hypo"),
        "Hypoxia",
        np.where(cond_raw.str.startswith("Norm"), "Normoxia", "NA"),
    )
    G = df[genes].to_numpy(np.float32)

    total_counts = G.sum(axis=1)
    n_detected = (G > 0).sum(axis=1)
    mito_idx = [i for i, g in enumerate(genes) if str(g).startswith("MT-")]
    pct_mito = G[:, mito_idx].sum(axis=1) / np.where(total_counts <= 0, 1, total_counts)
    keep = (
        (total_counts >= QC["min_total_counts"])
        & (n_detected >= QC["min_detected_genes"])
        & (pct_mito <= QC["max_pct_mito"])
    )

    n_total = int(len(keep))
    G, cond, cells = G[keep], cond[keep], df["Cell"].values[keep]
    min_cells = max(1, int(np.ceil(0.01 * G.shape[0])))
    gene_mask = (G >= 1).sum(axis=0) >= min_cells
    G, genes = G[:, gene_mask], genes[gene_mask]

    libsize = np.where(G.sum(axis=1, keepdims=True) <= 0, 1, G.sum(axis=1, keepdims=True))
    lognorm = np.log1p(G / libsize * 1e4)

    hvg_order = np.argsort(lognorm.var(axis=0))[::-1][:n_hvg]
    selected = sorted(set(hvg_order.tolist()) | {i for i, g in enumerate(genes) if g in BUFFA_SET})

    a = ad.AnnData(G[:, selected].copy())
    a.var_names = list(genes[selected])
    a.obs_names = list(cells)
    a.obs["condition"] = pd.Categorical(cond, categories=["Normoxia", "Hypoxia"])
    a.layers["counts"] = a.X.copy()
    a.layers["lognorm"] = lognorm[:, selected].copy()
    a.obs["total_counts"] = G.sum(axis=1)
    a.obs["n_genes"] = (G > 0).sum(axis=1)
    a.uns["n_cells_total"] = n_total
    return a

unfiltered_merged_tables = {
    "HCC1806": make_unsupervised_table(final_U1),
    "MCF7": make_unsupervised_table(final_U2),
}
raw = {name: load_counts_from_dataframe(df) for name, df in unfiltered_merged_tables.items()}

summary = pd.DataFrame({
    "n_cells_QC": [a.n_obs for a in raw.values()],
    "n_cells_total": [a.uns["n_cells_total"] for a in raw.values()],
    "n_genes (HVG U Buffa)": [a.n_vars for a in raw.values()],
    "% Hypoxia": [round(100 * (a.obs.condition == "Hypoxia").mean(), 1) for a in raw.values()],
    "% zeros": [round(100 * float((np.asarray(a.layers["counts"]) == 0).mean()), 1) for a in raw.values()],
    "Buffa present": [sum(g in set(a.var_names) for g in BUFFA) for a in raw.values()],
}, index=list(raw))

# ---------------------------------------------------------------------
# Supervised datasets: SmartSeq normalised, SmartSeq QCd and DropSeq normalised
# ---------------------------------------------------------------------
X_hcc, y_hcc, y_hcc_label, meta_hcc = prepare_labelled_dataset(HCC_TRAIN_FILE, "HCC1806")
X_mcf, y_mcf, y_mcf_label, meta_mcf = prepare_labelled_dataset(MCF_TRAIN_FILE, "MCF7")
X_hcc_anon = prepare_anonymous_dataset(HCC_ANON_FILE)
X_mcf_anon = prepare_anonymous_dataset(MCF_ANON_FILE)

X_hcc_qcd, y_hcc_qcd, y_hcc_qcd_label, meta_hcc_qcd = prepare_qcd_labelled_dataset(HCC_QCD_FILE, "HCC1806")
X_mcf_qcd, y_mcf_qcd, y_mcf_qcd_label, meta_mcf_qcd = prepare_qcd_labelled_dataset(MCF_QCD_FILE, "MCF7")

X_hcc_dropseq, y_hcc_dropseq, y_hcc_dropseq_label, meta_hcc_dropseq = prepare_labelled_dataset(DROP_HCC_TRAIN_FILE, "HCC1806")
X_mcf_dropseq, y_mcf_dropseq, y_mcf_dropseq_label, meta_mcf_dropseq = prepare_labelled_dataset(DROP_MCF_TRAIN_FILE, "MCF7")
X_hcc_dropseq_anon = prepare_anonymous_dataset(DROP_HCC_ANON_FILE)
X_mcf_dropseq_anon = prepare_anonymous_dataset(DROP_MCF_ANON_FILE)

print("EDA HCC1806 unfiltered merged:", EDA_df.shape)
print("EDA MCF7 unfiltered merged:", EDA_df_MCF7.shape)
print("Professor filtered HCC1806:", filtered_data_Buffa_T.shape)
print("Professor filtered MCF7:", filtered_data_Buffa_T_MCF7.shape)
print("Unsupervised raw objects:")
display(summary)
print("SmartSeq normalised:", X_hcc.shape, X_mcf.shape)
print("SmartSeq QCd:", X_hcc_qcd.shape, X_mcf_qcd.shape)
print("DropSeq normalised:", X_hcc_dropseq.shape, X_mcf_dropseq.shape)


---

## Exploratory Data Analysis

## Introduction

This project lies at the intersection of **machine learning** and **health sciences**.    
    
In particular, it focuses on **cancer cells** and on how changes in oxygen availability can affect **cellular behaviour** and **gene expression**.   
   
   
A key concept in this context is **hypoxia**, the condition in which cells receive insufficient oxygen to maintain normal aerobic metabolism.   
    
With low levels of O₂, mitochondria slow down energy production, forcing the cell to reprogram its functioning which can lead to cell death if the stress continues.    
    
Cancer cells, on the other hand, grows best when oxygen is lacking. In fact, hypoxia in tumor cells stimulates the expression of certain genes capable of suppressing the immune response.   
    
   
Machine learning methods can be used, therefore, to explore whether **gene expression profiles** distinguish cells grown under normal oxygen conditions from cells exposed to hypoxia.    
  
   
The analysis is performed on breast cancer cells cultured under:   
   
| Condition | Oxygen level |
|---|---:|
| <span style="background-color:#dbeafe; padding:2px 6px; border-radius:4px;">**Normoxia**</span> | 21% O₂ |
| <span style="background-color:#fee2e2; padding:2px 6px; border-radius:4px;">**Hypoxia**</span>| 1% O₂ |

## Dataset

The project is based on **14 datasets**, divided according to the sequencing platform and the breast cancer cell line.

Two sequencing platforms are considered:

- `Smart-seq`: a single-cell sequencing protocol with improved read coverage across transcripts

| Pros | Cons |
|---|---|
| mRNA sequence does not have to be known | Not strand-specific |
| As little as 50 pg of starting material can be used | No early multiplexing |
| Improved coverage across transcripts | Applicable only to poly(A)+ RNA |
| High level of mappable reads |  |

- `Drop-seq`: analyzes mRNA transcripts from droplets of individual cells in a highly parallel fashion. 

| Pros | Cons |
|---|---|
| Analysis of single-cell sequences in a highly parallel manner | Requires a custom microfluidics device |
| Unique molecular and cell barcodes enable cell- and gene-specific identification of mRNA strands | Lower gene-per-cell sensitivity compared to other scRNA-seq methods |
| Template-switching PCR produces high-yield reads from single cells | Limited to mRNA transcripts |
| Low cost and fast library prep|  |
   
   
Two breast cancer cell lines are analysed:

- `HCC1806`: human epithelial cells derived from breast/mammary gland tissue established from a 60-year-old black female patient.   
From a molecular point of view, HCC1806 is **negative** for the expression of Estrogen Receptor *ER*
- `MCF7`: human epithelial cells derived from breast/mammary gland tissue established from a 69-year-old white female patient.   
From a molecular point of view, MCF7 is **positive** for the expression of Estrogen Receptor *ER*

#### Dataset Summary


| Platform | Cell line | Dataset type |
|---|---|---|
| Smart-seq | HCC1806 | Unfiltered |
| Smart-seq | HCC1806 | Metadata |
| Smart-seq | HCC1806 | Filtered |
| Smart-seq | HCC1806 | Filtered and Normalised Train |
| Smart-seq | HCC1806 | Filtered and Normalised Test |
| Smart-seq | MCF7 | Unfiltered |
| Smart-seq | MCF7 | Metadata |
| Smart-seq | MCF7 | Filtered |
| Smart-seq | MCF7 | Filtered and Normalised Train |
| Smart-seq | MCF7 | Filtered and Normalised Test |
| Drop-seq | HCC1806 | Filtered and Normalised Train |
| Drop-seq | HCC1806 | Filtered and Normalised Test |
| Drop-seq | MCF7 | Filtered and Normalised Train |
| Drop-seq | MCF7 | Filtered and Normalised Test |

### HCC1806 SmartS

#### Unfiltered Dataset

This unfiltered matrix report gene expression values, with genes as rows and cells as columns.

The column format is `output.STAR.Position_Condition_Name_Aligned.sortedByCoord.out.bam`

The rows correspond to a very large number of genes, confirming the high-dimensional nature of the dataset.   
Since no filtering or normalization has been applied in this version, it is important to observe the following gene categories.

**MT-genes — mitochondrial genes**

These are genes encoded by mitochondrial DNA, linked to cellular respiration and energy production.  
In tumor cells, especially under hypoxic conditions, energy metabolism changes significantly. Furthermore, high expression of mitochondrial genes may indicate cellular stress or sample quality issues.

**RPS and RPL — ribosomal genes**

RPS genes encode proteins of the small subunit of the ribosome, while RPL genes encode proteins of the large subunit. They are involved in protein synthesis.  
Cancer cells often have high proliferation and intense protein production. These genes may therefore reflect high biosynthetic activity, but they also dominate the variability of the dataset because they are often highly expressed.

**P — Pseudogenes**

These are sequences similar to functional genes, but often do not encode complete or functional proteins.  
They can have an ambiguous role: some pseudogenes may participate in the regulation of gene expression, but they can also create technical interpretation or mapping problems because they are very similar to the genes from which they derive.

#### Metadata

Each of the 243 rows corresponds to a single cell, while the columns provide information describing the experimental context in which each cell was processed.

The row format is `output.STAR.Position_Condition_Name_Aligned.sortedByCoord.out.bam`

Metadata includes:
- Cell Line: breast cancer cell line

- PCR Plate and Pos: technical information related to sample preparation

- <span style="background-color:green; padding:2px 6px; border-radius:4px;">**Condition**</span>: whether the cell was cultured under Normoxia or Hypoxia

- Hours: 24 hours under hypoxic or normoxic conditions

- Cell name, PreprocessingTag and PreprocessingComment

#### Final Unfiltered Dataset

Merging gene expression data with metadata allows us to associate each cell with its biological condition and other relevant attributes, improving interpretability and enabling condition-specific analyses.

To achieve this, we transpose the unfiltered gene expression matrix and merge it with the metadata using an inner join.

### MCF7 SmartS

The same procedure is applied to the MCF7 cell line, since the structure of the unfiltered matrix and the metadata integration process remain unchanged.

#### Unfiltered Dataset

#### Metadata

Note that here the exposure time is 72 hours for all samples

#### Final Unfiltered Dataset

## Exploratory Data Analysis

Exploratory Data Analysis was performed as a preliminary step to:

1. Understand the dataset structure

    Number of cells, number of genes, metadata variables, biological condition, presence of zero values, and very high dimensionality. 

2. Evaluate cell quality with metrics such as:

    total counts per cell; number of detected genes; mitochondrial percentage; ribosomal genes; pseudogenes.

    This is important because in RNA-seq data, the workflow always includes quality checks before analysis, preprocessing, and classification.

3. Identify outliers or low-quality cells

    Some cells may have very few reads, few expressed genes, or high mitochondrial percentages. These cells can bias subsequent analyses.

4. Understand data distribution

    Gene counts are often very sparse, full of zeros, and with highly skewed distributions. For this reason, EDA helps determine whether transformations, normalization, or filters are needed.

5. Compare Normoxia and Hypoxia

    EDA allows you to see whether the two conditions are balanced, whether they have similar distributions, or whether they already show differences in quality metrics or the expression of certain genes.

6. Guide subsequent decisions

    After EDA, you can decide: which cells to remove; which genes to filter; whether to use raw, filtered, or normalized data; which features to use in the model; whether to apply PCA, clustering, or classification.

### HCC1806 SmartS

#### Preliminaries

In [ ]:
EDA_df.info()

In [ ]:
# Check type
EDA_df.dtypes

In [ ]:
EDA_df.describe(include="all")

In [ ]:
# Check for NaN
EDA_df.isna().sum()

In [ ]:
# Check for duplicates
EDA_df.duplicated().sum()

#### Analysis

##### Target Variable 

In [ ]:
print(EDA_df["Condition"].value_counts())
EDA_df["Condition"].value_counts(normalize=True) * 100

In [ ]:
condition_colors = {
    "Normo": "blue",
    "Hypo": "red"
}

In [ ]:
plt.figure(figsize=(5,4))

ax = sns.countplot(
    data=EDA_df,
    x="Condition",
    hue="Condition",
    palette=condition_colors,
    legend=False
)

for container in ax.containers:
    ax.bar_label(container, padding=3)

plt.title("Distribution of Target Variable")
plt.xlabel("Condition")
plt.ylabel("Count")
plt.ylim(0, EDA_df["Condition"].value_counts().max() + 15)

plt.show()

The target variable <span style="background-color:green; padding:2px 6px; border-radius:4px;">**Condition**</span> shows a balanced distribution between the two experimental groups.

In particular, the dataset contains:

- **117 cells** under **normoxic conditions**
- **126 cells** under **hypoxic conditions**

The difference between the two classes is relatively small, suggesting that there is **no significant imbalance** in the target variable.

In [ ]:
pd.crosstab(EDA_df["PCR Plate"], EDA_df["Condition"])

In [ ]:
plate_condition = pd.crosstab(EDA_df["PCR Plate"], EDA_df["Condition"])

ax = plate_condition[["Normo", "Hypo"]].plot(
    kind="bar",
    figsize=(10, 5),
    color=["blue", "red"]
)

for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)

plt.title("Condition distribution by PCR Plate")
plt.xlabel("PCR Plate")
plt.ylabel("Number of samples")
plt.xticks(rotation=0)
plt.legend(title="Condition")
plt.ylim(0, plate_condition[["Normo", "Hypo"]].values.max() + 5)
plt.show()

The graph shows the distribution of the two experimental conditions, **Normo** and **Hypo**, within the different **PCR Plates**.

**Plate 1** is perfectly balanced, with the same number of Normo and Hypo samples.   

**Plates 2** and **Plate 3**, however, show a greater presence of Hypo samples.   

Conversely, **Plate 4** contains more Normo samples than Hypo samples.   

Although the overall distribution of the target variable is nearly balanced, this graph suggests that the **PCR Plate** variable must be considered during EDA. Indeed, any differences between the plates could introduce technical variability or possible batch effects, which could influence subsequent analyses and machine learning models.

##### Genes

In [ ]:
def gene_family(gene):
    if gene.startswith("MT-"):
        return "MT"
    elif gene.startswith(("RPS","RPL")):
        return "Ribosomal"
    elif re.search(r"P\d*$", gene):
        return "Pseudogene"
    else:
        return "Other"

In [ ]:
gene_cols = [col for col in EDA_df.columns if col not in metadata_cols]

In [ ]:
# np.random.seed(42) # Uncomment to make the result unique
random_genes = np.random.choice(gene_cols, size=12, replace=False)

plt.figure(figsize=(16, 12))

for i, gene in enumerate(random_genes, 1):
    plt.subplot(3, 4, i)

    values = EDA_df[gene]

    counts, bins, patches = plt.hist(values, bins=30)

    zero_bin = np.where((bins[:-1] <= 0) & (bins[1:] > 0))[0]

    if len(zero_bin) > 0:
        patches[zero_bin[0]].set_facecolor("red")

    plt.title(gene)
    plt.xlabel("Expression")
    plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

The distribution of a random selection of genes shows a strong presence of **zero values**.

This pattern suggests that many genes are either:

- **not expressed** in most samples, or
- **not detected** due to low expression levels or technical limitations.

When gene expression is present, the distributions are often **right-skewed**. This means that most cells show low expression values, while a few cells display **very high expression levels**.

This behavior is typical of high-dimensional single-cell RNA-seq data: thousands of genes are measured in many cells, but each single cell expresses only a limited fraction of all genes.

In this dataset, therefore, **sparsity** refers to the presence of a large number of zero values compared to non-zero values. Consequently, only a small portion of the matrix contains positive and potentially informative expression values.

##### Sparsity

In [ ]:
total_values = EDA_df[gene_cols].size
zero_values = (EDA_df[gene_cols] == 0).sum().sum()

sparsity = zero_values / total_values

print(f"Number of zero values: {zero_values}")
print(f"Total number of values: {total_values}")
print(f"Overall sparsity: {sparsity:.2%}")

In [ ]:
gene_sparsity = (EDA_df[gene_cols] == 0).mean().sort_values(ascending=False)

gene_sparsity.head(10)

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(gene_sparsity, bins=50, color="green")

plt.title("Sparsity distribution across genes")
plt.xlabel("Fraction of zero values per gene")
plt.ylabel("Number of genes")

plt.show()

The overall sparsity of the gene expression matrix is **55.85%**, indicating that more than half of the gene-cell expression values are equal to **zero**.

This confirms that the dataset is **sparse**, which is expected for **single-cell RNA-seq data**.


The sparsity distribution by gene shows that many genes have a high fraction of zero values across cells.


This result highlights the importance of applying **quality control** and **filtering procedures** before subsequent analyses.

##### Quality Control (QC)

Because the unfiltered expression matrix is ​​highly sparse and contains many zero values, a **quality control (QC)** step is required to perform a truly meaningful EDA.

Without quality control, summary statistics such as mean expression or total count can be misleading because they are influenced by low-quality cells, rarely detected genes, technical noise, and the high number of null values.

Quality control therefore helps reduce these problems by filtering out unreliable cells and genes with little informative content. This makes subsequent exploratory analyses more robust, interpretable, and biologically meaningful.

In [ ]:
gene_families = pd.Series(
    {gene: gene_family(gene) for gene in gene_cols}
)

mt_genes = gene_families[gene_families == 'MT'].index.tolist()
ribosomal_genes = gene_families[gene_families == 'Ribosomal'].index.tolist()
pseudogene_genes = gene_families[gene_families == 'Pseudogene'].index.tolist()
other_genes = gene_families[gene_families == 'Other'].index.tolist()

In [ ]:
EDAQC_df = EDA_df.copy()

EDAQC_df["total_counts"] = EDAQC_df[gene_cols].sum(axis=1)

In [ ]:
EDAQC_df['n_detected_genes'] = (EDAQC_df[gene_cols] > 0).sum(axis=1)

In [ ]:
EDAQC_df['pct_mito'] = np.where(
    EDAQC_df['total_counts'] > 0,
    EDAQC_df[mt_genes].sum(axis=1) / EDAQC_df['total_counts'] * 100,
    0
)

EDAQC_df['pct_ribosomal'] = np.where(
    EDAQC_df['total_counts'] > 0,
    EDAQC_df[ribosomal_genes].sum(axis=1) / EDAQC_df['total_counts'] * 100,
    0
)

EDAQC_df['pct_pseudogene'] = np.where(
    EDAQC_df['total_counts'] > 0,
    EDAQC_df[pseudogene_genes].sum(axis=1) / EDAQC_df['total_counts'] * 100,
    0
)

Several metrics were developed to describe the quality and complexity of each individual cell. 

1. `total_counts`

    Sum of all gene expression counts detected in a cell. 
    This metric was developed because it allows to understand how much transcriptomic information has been captured for each cell: values ​​that are too low may indicate low-quality cells or cells with little RNA, while values ​​that are too high may suggest possible technical outliers or anomalous cells.

2. `n_detected_genes`

    Number of genes with expression greater than zero in each cell. Unlike total_counts, it does not measure how much a gene is expressed, but rather how many genes are actually detected. 
    It was developed because it indicates the complexity of a cell's transcriptomic profile: a cell with few detected genes contains little biological information and may be damaged or poorly sequenced.

3. `pct_mito`

    Percentage of counts associated with mitochondrial genes compared to the total counts of the cell. 
    It was developed because in single-cell data, a high mitochondrial percentage can be a sign of cellular stress, damage, or poor quality. 

4. `pct_ribosomal`

    Percentage of counts derived from ribosomal genes, such as those belonging to the RPS and RPL families. 
    It was developed because ribosomal genes can be highly expressed and, if they dominate the signal too much, can influence the variability observed in the dataset.

5. `pct_pseudogene`

    Percentage of counts associated with pseudogenes. 
    It was developed because a high presence of pseudogenes may represent a less directly interpretable component of the expression signal or indicate noise related to gene annotation. 

In [ ]:
qc_metrics = ['total_counts', 'n_detected_genes', 'pct_mito', 'pct_ribosomal', 'pct_pseudogene']

metric_labels = {
    'total_counts': 'Total Count',
    'n_detected_genes': 'Detected Genes',
    'pct_mito': 'Mitochondrial Percentage',
    'pct_ribosomal': 'Ribosomal Percentage',
    'pct_pseudogene': 'Pseudogene Percentage'
}

fig, axes = plt.subplots(1, 5, figsize=(25, 4))

for ax, metric in zip(axes, qc_metrics):
    sns.histplot(
        data=EDAQC_df,
        x=metric,
        hue='Condition',
        palette= {'Normo': 'blue', 'Hypo': 'red'},
        bins=30,
        element='step',
        fill=False,
        linewidth=2,
        common_bins=True,
        stat='count',
        ax=ax
    )
    
    ax.set_title(metric_labels[metric])
    ax.set_xlabel(metric_labels[metric])
    ax.set_ylabel('Number of Cells')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for ax, metric in zip(axes, qc_metrics):
    sns.boxplot(
        data=EDAQC_df,
        x='Condition',
        y=metric,
        hue='Condition',
        palette={'Normo': 'blue', 'Hypo': 'red'},
        order=['Normo', 'Hypo'],
        ax=ax,
        legend=False
    )
    
    ax.set_title(metric_labels[metric])
    ax.set_xlabel('')
    ax.set_ylabel(metric_labels[metric])

plt.tight_layout()
plt.show()

Although several quality control metrics were calculated, only the first three were used to define the filtering thresholds: `total_counts`, `n_detected_genes`, and `pct_mito`.

This choice was made because these metrics are the most directly informative about the overall quality of the cell.

In [ ]:
min_counts = EDAQC_df['total_counts'].quantile(0.05)
min_genes = EDAQC_df['n_detected_genes'].quantile(0.05)
max_mito = EDAQC_df['pct_mito'].quantile(0.95)

print("QC thresholds")
print(f"Minimum Total Count: {min_counts:.2f}")
print(f"Minimum Detected Genes: {min_genes:.2f}")
print(f"Maximum Mito%: {max_mito:.2f}")

In [ ]:
col_condition = "Condition"
col_counts = "total_counts"
col_genes = "n_detected_genes"
col_mito = "pct_mito"

# MAD multipliers
MAD_COUNTS = 5
MAD_GENES = 5
MAD_MITO = 3

# Initialize outlier columns
EDAQC_df["outlier_MAD"] = False
EDAQC_df["outlier_low_counts"] = False
EDAQC_df["outlier_low_genes"] = False
EDAQC_df["outlier_high_mito"] = False

mad_thresholds = []

for condition in EDAQC_df[col_condition].unique():
    mask = EDAQC_df[col_condition] == condition
    sub_df = EDAQC_df.loc[mask].copy()

    # log1p transform because RNA-seq counts are very skewed: many cells with medium/low values ​​and few cells with very high values
    log_counts = np.log1p(sub_df[col_counts])
    log_genes = np.log1p(sub_df[col_genes])

    mito = sub_df[col_mito]

    # Total counts lower bound
    med_counts = np.median(log_counts)
    mad_counts = median_abs_deviation(log_counts, scale="normal")
    min_counts_log = med_counts - MAD_COUNTS * mad_counts
    min_counts = max(np.expm1(min_counts_log), 0)

    outlier_counts = log_counts < min_counts_log

    # Detected genes lower bound
    med_genes = np.median(log_genes)
    mad_genes = median_abs_deviation(log_genes, scale="normal")
    min_genes_log = med_genes - MAD_GENES * mad_genes
    min_genes = max(np.expm1(min_genes_log), 0)

    outlier_genes = log_genes < min_genes_log

    # Mitochondrial percentage upper bound
    med_mito = np.median(mito)
    mad_mito = median_abs_deviation(mito, scale="normal")
    max_mito = med_mito + MAD_MITO * mad_mito

    # Optional safety floor: do not set mitochondrial threshold below 5%
    max_mito = max(max_mito, 5.0)

    outlier_mito = mito > max_mito

    # Treshold Combination
    outlier_total = outlier_counts | outlier_genes | outlier_mito

    EDAQC_df.loc[mask, "outlier_low_counts"] = outlier_counts.values
    EDAQC_df.loc[mask, "outlier_low_genes"] = outlier_genes.values
    EDAQC_df.loc[mask, "outlier_high_mito"] = outlier_mito.values
    EDAQC_df.loc[mask, "outlier_MAD"] = outlier_total.values

    mad_thresholds.append({
        "Condition": condition,
        "Minimum Total Count": min_counts,
        "Minimum Detected Genes": min_genes,
        "Maximum Mito%": max_mito,
        "Removed low counts": int(outlier_counts.sum()),
        "Removed low genes": int(outlier_genes.sum()),
        "Removed high Mito%": int(outlier_mito.sum()),
        "Removed total": int(outlier_total.sum()),
        "Cells before QC": int(mask.sum()),
        "Cells after QC": int(mask.sum() - outlier_total.sum())
    })

In [ ]:
mad_thresholds_df = pd.DataFrame(mad_thresholds)

display(mad_thresholds_df)

EDAQC_MAD_df = EDAQC_df[~EDAQC_df["outlier_MAD"]].copy()

print(f"Cells before QC: {len(EDAQC_df)}")
print(f"Cells after QC: {len(EDAQC_MAD_df)}")
print(f"Removed cells: {EDAQC_df['outlier_MAD'].sum()}")

> Removing outliers is a strong and sometimes irreversible filtering decision!

This choice should ideally be reviewed in collaboration with domain experts

In [ ]:
# Treshold Visualization on Unfiltered Data
condition_colors = {
    "Normo": "tab:blue",
    "Hypo": "tab:red"
}

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

# Total Count
ax = axes[0]

for condition in ["Normo", "Hypo"]:
    mask = EDAQC_df["Condition"] == condition
    values = np.log1p(EDAQC_df.loc[mask, "total_counts"])

    threshold = mad_thresholds_df.loc[
        mad_thresholds_df["Condition"] == condition,
        "Minimum Total Count"
    ].iloc[0]

    ax.hist(
        values,
        bins=30,
        histtype="step",
        linewidth=2,
        label=condition,
        color=condition_colors[condition]
    )

    ax.axvline(
        np.log1p(threshold),
        linestyle="--",
        linewidth=2,
        color=condition_colors[condition]
    )

ax.set_title("Total Count")
ax.set_xlabel("log1p(Total Count)")
ax.set_ylabel("Cell count")
ax.legend()


# Detected Genes
ax = axes[1]

for condition in ["Normo", "Hypo"]:
    mask = EDAQC_df["Condition"] == condition
    values = np.log1p(EDAQC_df.loc[mask, "n_detected_genes"])

    threshold = mad_thresholds_df.loc[
        mad_thresholds_df["Condition"] == condition,
        "Minimum Detected Genes"
    ].iloc[0]

    ax.hist(
        values,
        bins=30,
        histtype="step",
        linewidth=2,
        label=condition,
        color=condition_colors[condition]
    )

    ax.axvline(
        np.log1p(threshold),
        linestyle="--",
        linewidth=2,
        color=condition_colors[condition]
    )

ax.set_title("Detected Genes")
ax.set_xlabel("log1p(Detected Genes)")
ax.set_ylabel("Cell count")
ax.legend()

# Mito%
ax = axes[2]

mito_bins = np.linspace(0, 10, 31)

for condition in ["Normo", "Hypo"]:
    mask = EDAQC_df["Condition"] == condition
    values = EDAQC_df.loc[mask, "pct_mito"]

    threshold = mad_thresholds_df.loc[
        mad_thresholds_df["Condition"] == condition,
        "Maximum Mito%"
    ].iloc[0]

    ax.hist(
        values,
        bins=mito_bins,
        histtype="step",
        linewidth=2,
        label=condition,
        color=condition_colors[condition]
    )

    ax.axvline(
        threshold,
        linestyle="--",
        linewidth=2,
        color=condition_colors[condition]
    )

ax.set_title("Mito%")
ax.set_xlabel("Mito%")
ax.set_ylabel("Cell count")
ax.set_xlim(0, 10)
ax.legend()

plt.tight_layout()
plt.show()

Three different approaches for defining QC thresholds were tested: **quantiles**, **IQR** and **MAD**.

1. **Quantiles:** value that divides a distribution of ordered data into intervals with the same percentage of observations.

    They are simple to apply, but they always cut a fixed percentage of cells. This can be a limitation because they don't necessarily distinguish between true technical outliers and biologically different but valid cells.

2. **IQR:** distance between the third quartile and the first quartile.

    IQR is more robust than quantiles because it uses the range between the first and third quartiles, but it may still be unsuitable when the distributions are highly skewed or sparse. It can produce negative tresholds.


3. **Mean Absolute Deviation:** how far values ​​deviate from the median, using a robust measure of variability.

$$
\text{MAD} = \mathrm{median}\left(\left|x_i - \mathrm{median}(x)\right|\right)
$$


MAD was the best choice because it is more robust than simple quantiles and IQR, especially in a single-cell RNA-seq dataset where the distributions of QC metrics are not symmetric, contain many extreme values, and may differ between normoxia and hypoxia.

MAD-based thresholds were calculated separately for **Normoxia** and **Hypoxia**.

This choice was made because the two experimental conditions can have different distributions in quality control metrics, due to both technical effects and actual biological differences induced by hypoxia.

Applying a single global threshold could have unbalanced removal of cells belonging to one of the two conditions.

The condition-specific calculation, however, allows each cell to be compared with cells belonging to the same condition, preserving biological variability and more appropriately identifying technical outliers.

##### Normalization & Log

Normalization and log-transformation were applied to make the gene expression data more comparable and interpretable. 


**Normalization** is a procedure that modifies the original values ​​to make them comparable between different samples or cells.    

   In our case, it corrects for differences in library size, i.e., the different total number of counts present in each cell.

   After normalization, the differences observed between cells are less dependent on technical factors and are more biologically interpretable.


**Log-transformation** is a mathematical transformation that applies the logarithm to expression values.      

   In our case, we use "log1p", i.e., "log(1 + x)", because many values ​​are zero.

   This transformation compresses very high values, reduces the effect of outliers, and makes distributions less skewed and easier to visualize and compare.

In [ ]:
EDA_norm = EDAQC_MAD_df.copy()

metadata_cols = ['PCR Plate', 'Pos', 'Condition', 'Cell name', 'outlier_MAD', 'outlier_low_counts', 'outlier_low_genes', 'outlier_high_mito', 'pct_mito', 'pct_ribosomal', 'pct_pseudogene', 'total_counts', 'n_detected_genes']
gene_cols = [col for col in EDA_norm.columns if col not in metadata_cols]

library_size = EDA_norm[gene_cols].sum(axis=1)

EDA_norm[gene_cols] = np.log1p(
    EDA_norm[gene_cols].div(library_size, axis=0) * 10000
)
EDA_norm[gene_cols].describe().T

To better understand the distribution of the data we use violin plots: a data visualization tool that combines a standard box plot with a rotated kernel density plot to display the distribution of numerical data

In [ ]:
df_plot = EDA_norm.copy()

# 30 random cells for visualization
# np.random.seed(42) # Uncomment to make the result unique
selected_cells = np.random.choice(df_plot["Cell name"], size=30, replace=False)

small_df = df_plot[df_plot["Cell name"].isin(selected_cells)]

long_cell_df = small_df[["Cell name", "Condition"] + gene_cols].melt(
    id_vars=["Cell name", "Condition"],
    value_vars=gene_cols,
    var_name="Gene",
    value_name="Expression"
)

plt.figure(figsize=(16, 6))

sns.violinplot(
    data=long_cell_df,
    x="Cell name",
    y="Expression",
    hue="Condition",
    palette={"Normo": "blue", "Hypo": "red"},
    inner="quartile",
    cut=0
)

plt.title("Random cells gene expression Distribution")
plt.xlabel("Cell")
plt.ylabel("Expression (normalized and log-transformed)")
plt.xticks(rotation=0)
plt.legend(title="Condition")
plt.tight_layout()
plt.show()

In [ ]:
long_gene_df = df_plot[["Condition"] + gene_cols].melt(
    id_vars="Condition",
    value_vars=gene_cols,
    var_name="Gene",
    value_name="Expression"
)

plt.figure(figsize=(8, 5))

sns.histplot(
    data=long_gene_df,
    x="Expression",
    hue="Condition",
    palette={"Normo": "blue", "Hypo": "red"},
    bins=50,
    element="step",
    stat="density",
    common_norm=False
)

plt.title("Gene expression: Normoxia vs Hypoxia")
plt.xlabel("Gene expression (normalized and log-transformed)")
plt.ylabel("Density")
plt.tight_layout()
plt.show()

In the cell-by-cell violin plot, each violin represents a randomly selected cell and shows the expression distribution of all genes in that cell.

The shape of the violins is very similar across cells: most values ​​are concentrated near zero, while only a small portion of genes reach higher values.

This indicates that, even after normalization and log-transformation, many gene expressions remain low or zero.

The long upward tails show that a few genes are expressed at higher levels.

There is no obvious difference between **Normo** and **Hypo** cells in terms of overall expression distribution.

---

In the Normo vs. Hypo histogram, the two distributions almost completely overlap.

A strong peak near zero and a long tail to the right are visible, too.

This means that most gene expression values ​​are low, while a few values ​​are higher.

The fact that **Normo** and **Hypo** are very similar suggests that the difference between the two conditions does not involve a global shift in gene expression as a whole, but likely concerns specific genes or groups of genes.

##### Diagnostic

Detailed evaluation of sparsity, skewness and kurtosis on normalised and log-transformed dataset

---

Let $X = (x_{ij})$ be an $N \times P$ gene expression matrix, with $i$ indexing cells and $j$ indexing genes. Then the *overall sparsity* $S(X)$, the *sparsity per cell* $s_i^{(c)}$ and the *sparsity per gene* $s_j^{(g)}$ are defined as follows:

$$
S(X) =
\frac{1}{N \times P}
\sum_{i=1}^{N}
\sum_{j=1}^{P}
\mathbf{1}\{x_{ij}=0\}
$$

$$
s_i^{(c)} =
\frac{1}{P}
\sum_{j=1}^{P}
\mathbf{1}\{x_{ij}=0\}
$$

$$
s_j^{(g)} =
\frac{1}{N}
\sum_{i=1}^{N}
\mathbf{1}\{x_{ij}=0\}
$$

---

Skewness is the standardized third central moment of the distribution, while kurtosis is the standardized fourth central moment:

$$
\text{Skewness}
=
\frac{1}{n}
\sum_{i=1}^{n}
\left(
\frac{x_i - \mu}{\sigma}
\right)^3
$$

$$
\text{Kurtosis}
=
\frac{1}{n}
\sum_{i=1}^{n}
\left(
\frac{x_i - \mu}{\sigma}
\right)^4
$$

In [ ]:
EDA_shape = EDA_norm.copy()
X = EDA_norm[gene_cols]

In [ ]:
# Overall sparsity
overall_sparsity = (X == 0).sum().sum() / X.size * 100

# Sparsity per cell
EDA_shape['sparsity_per_cell'] = (X == 0).sum(axis=1) / X.shape[1] * 100

# Sparsity per gene
sparsity_per_gene = (X == 0).sum(axis=0) / X.shape[0] * 100

sparsity_gene_df = pd.DataFrame({
    'Gene': sparsity_per_gene.index,
    'sparsity_per_gene': sparsity_per_gene.values
})

print(f"Overall sparsity: {overall_sparsity:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 5))

sns.boxplot(
    data=EDA_shape,
    x='Condition',
    y='sparsity_per_cell',
    hue='Condition',
    palette={'Normo': 'blue', 'Hypo': 'red'},
    legend=False,
    ax=axes[0]
)

axes[0].set_title('Sparsity per Cell by Condition')
axes[0].set_xlabel('Condition')
axes[0].set_ylabel('Sparsity per Cell (%)')

sns.histplot(
    data=EDA_shape,
    x='sparsity_per_cell',
    hue='Condition',
    bins=25,
    palette={'Normo': 'blue', 'Hypo': 'red'},
    alpha=0.5,
    ax=axes[1]
)

axes[1].set_title('Distribution of Sparsity per Cell')
axes[1].set_xlabel('Sparsity per Cell (%)')
axes[1].set_ylabel('Number of Cells')

sns.histplot(
    data=sparsity_gene_df,
    x='sparsity_per_gene',
    bins=30,
    alpha=0.5,
    ax=axes[2]
)

axes[2].set_title('Distribution of Sparsity per Gene')
axes[2].set_xlabel('Sparsity per Gene (%)')
axes[2].set_ylabel('Number of Genes')

fig.suptitle('Sparsity Analysis on Data (normalized and log-transformed)', fontsize=16, y=1.05)

plt.tight_layout()
plt.show()

Overall, these graphs show that normalization and logarithmic transformation make the data more comparable and interpretable, but do not eliminate sparsity.

At the cellular level, both **Normo** and **Hypo** show that approximately half of the genes have zero expression, with **Hypo** cells slightly more sparse and some more extreme outliers.

At the gene level, however, the distribution is strongly polarized: some genes are expressed in almost all cells, while many others are absent in most cells.

This confirms that sparsity remains a central feature of the dataset and must be considered in subsequent analyses.

In [ ]:
overall_skewness = skew(X.values.flatten(), nan_policy='omit')
overall_kurtosis = kurtosis(X.values.flatten(), fisher=True, nan_policy='omit')

print(f"Overall skewness: {overall_skewness:.3f}")
print(f"Overall kurtosis: {overall_kurtosis:.3f}")

In [ ]:
# Per cell
EDA_shape['skewness_per_cell'] = X.apply(
    lambda row: skew(row, nan_policy='omit'), axis=1
)

EDA_shape['kurtosis_per_cell'] = X.apply(
    lambda row: kurtosis(row, fisher=True, nan_policy='omit'), axis=1
)

# Per gene
skewness_per_gene = X.apply(
    lambda col: skew(col, nan_policy='omit'), axis=0
)

kurtosis_per_gene = X.apply(
    lambda col: kurtosis(col, fisher=True, nan_policy='omit'), axis=0
)

gene_shape_df = pd.DataFrame({
    'Gene': gene_cols,
    'sparsity_per_gene': sparsity_per_gene.values,
    'skewness_per_gene': skewness_per_gene.values,
    'kurtosis_per_gene': kurtosis_per_gene.values
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(
    data=EDA_shape,
    x='skewness_per_cell',
    hue='Condition',
    bins=25,
    palette={'Normo': 'blue', 'Hypo': 'red'},
    alpha=0.5,
    ax=axes[0]
)

axes[0].set_title('Distribution of Skewness per Cell')
axes[0].set_xlabel('Skewness per Cell')
axes[0].set_ylabel('Number of Cells')

sns.histplot(
    data=EDA_shape,
    x='kurtosis_per_cell',
    hue='Condition',
    bins=25,
    palette={'Normo': 'blue', 'Hypo': 'red'},
    alpha=0.5,
    ax=axes[1]
)

axes[1].set_title('Distribution of Kurtosis per Cell')
axes[1].set_xlabel('Kurtosis per Cell')
axes[1].set_ylabel('Number of Cells')

plt.tight_layout()
plt.show()

The overall **skewness = 3.326** indicates that, considering all the values ​​in the expression matrix together, the distribution is still strongly skewed to the right.

This means that most expression values ​​are low or zero, while a minority of values ​​are higher.

Even after log-transforming, therefore, the distribution does not become symmetric.



In the first histogram, **Distribution of Skewness per Cell**, each value represents the skewness of the gene distribution within a single cell.

All cells have positive skewness, so in each cell, most genes have low or zero expression, while only a smaller portion of genes have higher expression.

**Hypo** cells appear slightly shifted toward higher skewness values ​​than **Normo** cells, suggesting that under hypoxia, some cells have expression profiles that are slightly more skewed toward a few more highly expressed genes.

---

The overall **kurtosis = 15.533** is high.

Such a high value indicates a heavy-tailed distribution, meaning that some extreme values ​​are still present.

In the second histogram, **Distribution of Kurtosis per Cell**, we see that kurtosis per cell is also positive and elevated.

This indicates that the expression profiles of the cells have heavy tails: most genes have low values, but some genes have relatively high values.

**Hypo** cells appear slightly shifted toward higher values ​​than **Normo** cells, indicating slightly more extreme distributions in hypoxic cells.

#### Correlation Analysis

During RNA sequencing, different gene identifiers may sometimes show identical expression profiles across cells.

This can happen for two reasons:

1. Redundant annotations

    Gene annotation databases may contain multiple identifiers that actually refer to the same transcript

2. Overlapping transcripts

    Highly similar or overlapping transcripts can be assigned the same reads, producing identical expression profiles across the cell-by-gene count matrix.


Keeping these duplicated profiles could introduce redundancy and bias into the analysis.

In [ ]:
metadata_cols = ["PCR Plate", "Pos", "Condition", "Cell Name", "outlier_MAD", "outlier_low_counts", "outlier_low_genes", "outlier_high_mito", "pct_mito", "pct_ribosomal", "pct_pseudogene", "total_counts", "n_detected_genes"]
gene_cols = [col for col in EDA_norm.columns if col not in metadata_cols]

gene_expr = EDA_norm[gene_cols]

duplicated_genes_mask = gene_expr.T.duplicated(keep=False)
duplicated_genes = gene_expr.T[duplicated_genes_mask]

print("Number of duplicated genes:", duplicated_genes.shape[0])

In [ ]:
duplicated_genes_to_remove = gene_expr.T.duplicated(keep="first")

genes_to_remove = gene_expr.T.index[duplicated_genes_to_remove].tolist()

print("Number of genes that can be removed:", len(genes_to_remove))
print("Examples:")
print(genes_to_remove[:20])

EDA_norm_no_identical_genes = EDA_norm.drop(columns=genes_to_remove)

print("Shape before filtering:", EDA_norm.shape)
print("Shape after filtering:", EDA_norm_no_identical_genes.shape)

> Removing duplicates is a strong and sometimes irreversible filtering decision!

This choice should ideally be reviewed in collaboration with domain experts

In [ ]:
metadata_cols = ['PCR Plate', 'Pos', 'Condition', 'Cell name', 'PCRPlate_Pos', 'outlier_MAD', 'outlier_low_counts', 'outlier_low_genes', 'outlier_high_mito', 'pct_mito', 'pct_ribosomal', 'pct_pseudogene', 'total_counts', 'n_detected_genes']

gene_cols = [
    col for col in EDA_norm_no_identical_genes.columns
    if col not in metadata_cols
]

hypo_df = EDA_norm_no_identical_genes[EDA_norm_no_identical_genes['Condition'] == 'Hypo']
normo_df = EDA_norm_no_identical_genes[EDA_norm_no_identical_genes['Condition'] == 'Normo']

print("Numero cellule Hypo:", hypo_df.shape[0])
print("Numero cellule Normo:", normo_df.shape[0])

In [ ]:
results = []

for gene in gene_cols:
    
    hypo_values = hypo_df[gene]
    normo_values = normo_df[gene]
    
    mean_hypo = hypo_values.mean()
    mean_normo = normo_values.mean()
    
    diff = mean_hypo - mean_normo
    
    # Cells percentage with expression > 0
    pct_hypo = (hypo_values > 0).mean()
    pct_normo = (normo_values > 0).mean()
    
    results.append({
        'Gene': gene,
        'Mean_Hypo': mean_hypo,
        'Mean_Normo': mean_normo,
        'Difference_Hypo_minus_Normo': diff,
        'Abs_Difference': abs(diff),
        'Pct_Expressed_Hypo': pct_hypo,
        'Pct_Expressed_Normo': pct_normo
    })

gene_condition_results = pd.DataFrame(results)

gene_condition_results.head()

In [ ]:
# Gene filtering based on mean difference and expression percentage
filtered_gene_results = gene_condition_results[
    (gene_condition_results['Abs_Difference'] > 0.25) &
    (
        (gene_condition_results['Pct_Expressed_Hypo'] > 0.10) |
        (gene_condition_results['Pct_Expressed_Normo'] > 0.10)
    )
].copy()

In [ ]:
top_hypo_genes = (
    filtered_gene_results
    .sort_values('Difference_Hypo_minus_Normo', ascending=False)
    .head(20)
)

top_hypo_genes

In [ ]:
top_normo_genes = (
    filtered_gene_results
    .sort_values('Difference_Hypo_minus_Normo', ascending=True)
    .head(20)
)

top_normo_genes

In [ ]:
top_hypo_genes_display = top_hypo_genes.copy()
top_hypo_genes_display['Associated_condition'] = 'Hypo'

top_normo_genes_display = top_normo_genes.copy()
top_normo_genes_display['Associated_condition'] = 'Normo'

top_condition_genes = pd.concat(
    [top_hypo_genes_display, top_normo_genes_display],
    axis=0
)

top_condition_genes = top_condition_genes[
    [
        'Gene',
        'Associated_condition',
        'Mean_Hypo',
        'Mean_Normo',
        'Difference_Hypo_minus_Normo',
        'Abs_Difference',
        'Pct_Expressed_Hypo',
        'Pct_Expressed_Normo',
    ]
]

top_condition_genes

In [ ]:
top_hypo_10 = top_hypo_genes.head(10).copy()
top_hypo_10['Associated_condition'] = 'Hypo'

top_normo_10 = top_normo_genes.head(10).copy()
top_normo_10['Associated_condition'] = 'Normo'

top_condition_genes_10 = pd.concat(
    [top_hypo_10, top_normo_10],
    axis=0
)

selected_genes = top_condition_genes_10['Gene'].tolist()

# Dot plot

plot_data = []

for gene in selected_genes:
    
    associated_condition = (
        top_condition_genes_10
        .set_index('Gene')
        .loc[gene, 'Associated_condition']
    )
    
    for condition in ['Normo', 'Hypo']:
        
        sub = EDA_norm_no_identical_genes[
            EDA_norm_no_identical_genes['Condition'] == condition
        ]
        
        plot_data.append({
            'Gene': gene,
            'Condition': condition,
            'Associated_condition': associated_condition,
            'Mean_expression': sub[gene].mean(),
            'Pct_expressed': (sub[gene] > 0).mean() * 100
        })

dotplot_df = pd.DataFrame(plot_data)

gene_order = top_condition_genes_10['Gene'].tolist()

dotplot_df['Gene'] = pd.Categorical(
    dotplot_df['Gene'],
    categories=gene_order,
    ordered=True
)

normo_plot = dotplot_df[dotplot_df['Condition'] == 'Normo']
hypo_plot = dotplot_df[dotplot_df['Condition'] == 'Hypo']


plt.figure(figsize=(7, 9))

sns.scatterplot(
    data=normo_plot,
    x='Condition',
    y='Gene',
    size='Pct_expressed',
    hue='Mean_expression',
    palette='Blues',
    sizes=(40, 320),
    legend=False
)

sns.scatterplot(
    data=hypo_plot,
    x='Condition',
    y='Gene',
    size='Pct_expressed',
    hue='Mean_expression',
    palette='Reds',
    sizes=(40, 320),
    legend=False
)

plt.axhline(
    y=9.5,
    color='black',
    linewidth=1,
    linestyle='--'
)

for pct in [25, 50, 75, 100]:
    plt.scatter(
        [],
        [],
        s=40 + (pct / 100) * 280,
        color='gray',
        alpha=0.6,
        label=f'{pct}% expressed'
    )

plt.legend(
    title='Dot size',
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

plt.title("Dot plot of genes associated with Hypoxia and Normoxia")
plt.xlabel("Condition")
plt.ylabel("Genes")

plt.tight_layout()
plt.show()

After filtering **429 informative genes**, i.e. genes with a fairly high mean expression difference between **Hypo** and **Normo** and present in at least a small percentage of cells, were identified.

The main finding is that the separation between the two conditions is quite clear, especially for hypoxia-associated genes.

In the dot plot, genes in the upper half, such as `DDIT4`, `ANGPTL4`, `BNIP3`, `NDRG1`, `PLIN2`, `SLC2A1`, `PGK1`, `BHLHE40`, `ALDOC`, and `FOS`, show higher mean values ​​in **Hypo** cells than in **Normo** cells.

Furthermore, many of these genes have large dots, meaning they are expressed in a high percentage of hypoxic cells.

This suggests that hypoxia induces a fairly consistent transcriptional change in many cells.


It's also interesting to note that some hypoxic genes are not exclusive to the **Hypo** condition.

For example, `SLC2A1` is expressed in virtually both conditions, but more intensely in **Hypoxic** cells.

This means it isn't simply "present or absent," but is upregulated in hypoxia.

### MCF7 SmartS

Compared to **HCC1806**, the **MCF7** dataset shows a more balanced distribution of the target variable, with a more balanced representation of cells in normoxia and hypoxia.

The overall distribution of gene expression is similar to that observed in **HCC1806**, confirming the typical structure of single-cell RNA-seq data, characterized by many low or zero values.

However, **MCF7** exhibits slightly higher sparsity, indicating that a larger fraction of the expression matrix is ​​composed of zeros.

After log-normalization, the per-cell sparsity distributions between normoxia and hypoxia do not overlap as much as in **HCC1806**, suggesting a more evident difference between the two conditions in the number of detected genes.

Skewness is similar between the two cell lines, while kurtosis is higher in **MCF7**, indicating a more concentrated distribution with more marked extreme values.

Finally, the filtering applied to **MCF7** was more severe and, in our opinion, more precise than that applied to **HCC1806**, since it removed a greater number of cells and appears to match the filtered dataset in the folder

In [ ]:
EDAdraft_df_MCF7 = pd.read_csv("MCF7_final_unfiltered.csv")

metadata_cols = ["Filename","Cell Line","Lane","Pos","Condition", "Hours","Cell name", "PreprocessingTag","ProcessingComments"]

gene_cols = [col for col in EDAdraft_df_MCF7.columns if col not in metadata_cols]

eda_metadata_cols = ["Lane","Pos","Condition", "Cell name"]

EDA_df_MCF7 = EDAdraft_df_MCF7[eda_metadata_cols + list(gene_cols)]
print(EDA_df_MCF7.shape)
EDA_df_MCF7.head()

In [ ]:
EDA_df_MCF7.info()

In [ ]:
EDA_df_MCF7.dtypes

In [ ]:
EDA_df_MCF7.describe(include="all")

In [ ]:
EDA_df_MCF7.isna().sum()

In [ ]:
EDA_df_MCF7.duplicated().sum()

In [ ]:
print(EDA_df_MCF7["Condition"].value_counts())
EDA_df_MCF7["Condition"].value_counts(normalize=True) * 100

In [ ]:
condition_colors = {
    "Norm": "blue",
    "Hypo": "red"
}

In [ ]:
plt.figure(figsize=(5,4))

ax = sns.countplot(
    data=EDA_df_MCF7,
    x="Condition",
    hue="Condition",
    palette=condition_colors,
    legend=False
)

for container in ax.containers:
    ax.bar_label(container, padding=3)

plt.title("Distribution of Target Variable")
plt.xlabel("Condition")
plt.ylabel("Count")
plt.ylim(0, EDA_df_MCF7["Condition"].value_counts().max() + 15)

plt.show()

In [ ]:
pd.crosstab(EDA_df_MCF7["Lane"], EDA_df_MCF7["Condition"])

In [ ]:
plate_condition = pd.crosstab(EDA_df_MCF7["Lane"], EDA_df_MCF7["Condition"])

ax = plate_condition[["Norm", "Hypo"]].plot(
    kind="bar",
    figsize=(10, 5),
    color=["blue", "red"]
)

for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)

plt.title("Condition distribution by Lane")
plt.xlabel("Lane")
plt.ylabel("Number of samples")
plt.xticks(rotation=0)
plt.legend(title="Condition")
plt.ylim(0, plate_condition[["Norm", "Hypo"]].values.max() + 5)
plt.show()

In [ ]:
def gene_family(gene):
    if gene.startswith("MT-"):
        return "MT"
    elif gene.startswith(("RPS","RPL")):
        return "Ribosomal"
    elif re.search(r"P\d*$", gene):
        return "Pseudogene"
    else:
        return "Other"

In [ ]:
gene_cols = [col for col in EDA_df_MCF7.columns if col not in metadata_cols]

In [ ]:
# np.random.seed(42) # Uncomment to make the result unique
random_genes = np.random.choice(gene_cols, size=12, replace=False)

plt.figure(figsize=(16, 12))

for i, gene in enumerate(random_genes, 1):
    plt.subplot(3, 4, i)

    values = EDA_df_MCF7[gene]

    counts, bins, patches = plt.hist(values, bins=30)

    zero_bin = np.where((bins[:-1] <= 0) & (bins[1:] > 0))[0]

    if len(zero_bin) > 0:
        patches[zero_bin[0]].set_facecolor("red")

    plt.title(gene)
    plt.xlabel("Expression")
    plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
total_values = EDA_df_MCF7[gene_cols].size
zero_values = (EDA_df_MCF7[gene_cols] == 0).sum().sum()

sparsity = zero_values / total_values

print(f"Number of zero values: {zero_values}")
print(f"Total number of values: {total_values}")
print(f"Overall sparsity: {sparsity:.2%}")

In [ ]:
gene_sparsity = (EDA_df_MCF7[gene_cols] == 0).mean().sort_values(ascending=False)

gene_sparsity.head(10)

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(gene_sparsity, bins=50, color="green")

plt.title("Sparsity distribution across genes")
plt.xlabel("Fraction of zero values per gene")
plt.ylabel("Number of genes")

plt.show()

In [ ]:
gene_families = pd.Series(
    {gene: gene_family(gene) for gene in gene_cols}
)

mt_genes = gene_families[gene_families == 'MT'].index.tolist()
ribosomal_genes = gene_families[gene_families == 'Ribosomal'].index.tolist()
pseudogene_genes = gene_families[gene_families == 'Pseudogene'].index.tolist()
other_genes = gene_families[gene_families == 'Other'].index.tolist()

In [ ]:
EDAQC_df_MCF7 = EDA_df_MCF7.copy()

EDAQC_df_MCF7["total_counts"] = EDAQC_df_MCF7[gene_cols].sum(axis=1)

In [ ]:
EDAQC_df_MCF7['n_detected_genes'] = (EDAQC_df_MCF7[gene_cols] > 0).sum(axis=1)

In [ ]:
EDAQC_df_MCF7['pct_mito'] = np.where(
    EDAQC_df_MCF7['total_counts'] > 0,
    EDAQC_df_MCF7[mt_genes].sum(axis=1) / EDAQC_df_MCF7['total_counts'] * 100,
    0
)

EDAQC_df_MCF7['pct_ribosomal'] = np.where(
    EDAQC_df_MCF7['total_counts'] > 0,
    EDAQC_df_MCF7[ribosomal_genes].sum(axis=1) / EDAQC_df_MCF7['total_counts'] * 100,
    0
)

EDAQC_df_MCF7['pct_pseudogene'] = np.where(
    EDAQC_df_MCF7['total_counts'] > 0,
    EDAQC_df_MCF7[pseudogene_genes].sum(axis=1) / EDAQC_df_MCF7['total_counts'] * 100,
    0
)

In [ ]:
qc_metrics = ['total_counts', 'n_detected_genes', 'pct_mito', 'pct_ribosomal', 'pct_pseudogene']

metric_labels = {
    'total_counts': 'Total Count',
    'n_detected_genes': 'Detected Genes',
    'pct_mito': 'Mitochondrial Percentage',
    'pct_ribosomal': 'Ribosomal Percentage',
    'pct_pseudogene': 'Pseudogene Percentage'
}

fig, axes = plt.subplots(1, 5, figsize=(25, 4))

for ax, metric in zip(axes, qc_metrics):
    sns.histplot(
        data=EDAQC_df_MCF7,
        x=metric,
        hue='Condition',
        palette= {'Norm': 'blue', 'Hypo': 'red'},
        bins=30,
        element='step',
        fill=False,
        linewidth=2,
        common_bins=True,
        stat='count',
        ax=ax
    )
    
    ax.set_title(metric_labels[metric])
    ax.set_xlabel(metric_labels[metric])
    ax.set_ylabel('Number of Cells')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for ax, metric in zip(axes, qc_metrics):
    sns.boxplot(
        data=EDAQC_df_MCF7,
        x='Condition',
        y=metric,
        hue='Condition',
        palette={'Norm': 'blue', 'Hypo': 'red'},
        order=['Norm', 'Hypo'],
        ax=ax,
        legend=False
    )
    
    ax.set_title(metric_labels[metric])
    ax.set_xlabel('')
    ax.set_ylabel(metric_labels[metric])

plt.tight_layout()
plt.show()

In [ ]:
min_counts = EDAQC_df_MCF7['total_counts'].quantile(0.05)
min_genes = EDAQC_df_MCF7['n_detected_genes'].quantile(0.05)
max_mito = EDAQC_df_MCF7['pct_mito'].quantile(0.95)

print("QC thresholds")
print(f"Minimum Total Count: {min_counts:.2f}")
print(f"Minimum Detected Genes: {min_genes:.2f}")
print(f"Maximum Mito%: {max_mito:.2f}")

In [ ]:
col_condition = "Condition"
col_counts = "total_counts"
col_genes = "n_detected_genes"
col_mito = "pct_mito"

# MAD multipliers
MAD_COUNTS = 5
MAD_GENES = 5
MAD_MITO = 3

# Initialize outlier columns
EDAQC_df_MCF7["outlier_MAD"] = False
EDAQC_df_MCF7["outlier_low_counts"] = False
EDAQC_df_MCF7["outlier_low_genes"] = False
EDAQC_df_MCF7["outlier_high_mito"] = False

mad_thresholds = []

for condition in EDAQC_df_MCF7[col_condition].unique():
    mask = EDAQC_df_MCF7[col_condition] == condition
    sub_df = EDAQC_df_MCF7.loc[mask].copy()

    # log1p transform because RNA-seq counts are very skewed: many cells with medium/low values ​​and few cells with very high values
    log_counts = np.log1p(sub_df[col_counts])
    log_genes = np.log1p(sub_df[col_genes])

    mito = sub_df[col_mito]

    # Total counts lower bound
    med_counts = np.median(log_counts)
    mad_counts = median_abs_deviation(log_counts, scale="normal")
    min_counts_log = med_counts - MAD_COUNTS * mad_counts
    min_counts = max(np.expm1(min_counts_log), 0)

    outlier_counts = log_counts < min_counts_log

    # Detected genes lower bound
    med_genes = np.median(log_genes)
    mad_genes = median_abs_deviation(log_genes, scale="normal")
    min_genes_log = med_genes - MAD_GENES * mad_genes
    min_genes = max(np.expm1(min_genes_log), 0)

    outlier_genes = log_genes < min_genes_log

    # Mitochondrial percentage upper bound
    med_mito = np.median(mito)
    mad_mito = median_abs_deviation(mito, scale="normal")
    max_mito = med_mito + MAD_MITO * mad_mito

    # Optional safety floor: do not set mitochondrial threshold below 5%
    max_mito = max(max_mito, 5.0)

    outlier_mito = mito > max_mito

    # Treshold Combination
    outlier_total = outlier_counts | outlier_genes | outlier_mito

    EDAQC_df_MCF7.loc[mask, "outlier_low_counts"] = outlier_counts.values
    EDAQC_df_MCF7.loc[mask, "outlier_low_genes"] = outlier_genes.values
    EDAQC_df_MCF7.loc[mask, "outlier_high_mito"] = outlier_mito.values
    EDAQC_df_MCF7.loc[mask, "outlier_MAD"] = outlier_total.values

    mad_thresholds.append({
        "Condition": condition,
        "Minimum Total Count": min_counts,
        "Minimum Detected Genes": min_genes,
        "Maximum Mito%": max_mito,
        "Removed low counts": int(outlier_counts.sum()),
        "Removed low genes": int(outlier_genes.sum()),
        "Removed high Mito%": int(outlier_mito.sum()),
        "Removed total": int(outlier_total.sum()),
        "Cells before QC": int(mask.sum()),
        "Cells after QC": int(mask.sum() - outlier_total.sum())
    })

In [ ]:
mad_thresholds_df = pd.DataFrame(mad_thresholds)

display(mad_thresholds_df)

EDAQC_MAD_df_MCF7 = EDAQC_df_MCF7[~EDAQC_df_MCF7["outlier_MAD"]].copy()

print(f"Cells before QC: {len(EDAQC_df_MCF7)}")
print(f"Cells after QC: {len(EDAQC_MAD_df_MCF7)}")
print(f"Removed cells: {EDAQC_df_MCF7['outlier_MAD'].sum()}")

In [ ]:
# Treshold Visualization on Unfiltered Data
condition_colors = {
    "Norm": "tab:blue",
    "Hypo": "tab:red"
}

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

# Total Count
ax = axes[0]

for condition in ["Norm", "Hypo"]:
    mask = EDAQC_df_MCF7["Condition"] == condition
    values = np.log1p(EDAQC_df_MCF7.loc[mask, "total_counts"])

    threshold = mad_thresholds_df.loc[
        mad_thresholds_df["Condition"] == condition,
        "Minimum Total Count"
    ].iloc[0]

    ax.hist(
        values,
        bins=30,
        histtype="step",
        linewidth=2,
        label=condition,
        color=condition_colors[condition]
    )

    ax.axvline(
        np.log1p(threshold),
        linestyle="--",
        linewidth=2,
        color=condition_colors[condition]
    )

ax.set_title("Total Count")
ax.set_xlabel("log1p(Total Count)")
ax.set_ylabel("Cell count")
ax.legend()


# Detected Genes
ax = axes[1]

for condition in ["Norm", "Hypo"]:
    mask = EDAQC_df_MCF7["Condition"] == condition
    values = np.log1p(EDAQC_df_MCF7.loc[mask, "n_detected_genes"])

    threshold = mad_thresholds_df.loc[
        mad_thresholds_df["Condition"] == condition,
        "Minimum Detected Genes"
    ].iloc[0]

    ax.hist(
        values,
        bins=30,
        histtype="step",
        linewidth=2,
        label=condition,
        color=condition_colors[condition]
    )

    ax.axvline(
        np.log1p(threshold),
        linestyle="--",
        linewidth=2,
        color=condition_colors[condition]
    )

ax.set_title("Detected Genes")
ax.set_xlabel("log1p(Detected Genes)")
ax.set_ylabel("Cell count")
ax.legend()

# Mito%
ax = axes[2]

mito_bins = np.linspace(0, 10, 31)

for condition in ["Norm", "Hypo"]:
    mask = EDAQC_df_MCF7["Condition"] == condition
    values = EDAQC_df_MCF7.loc[mask, "pct_mito"]

    threshold = mad_thresholds_df.loc[
        mad_thresholds_df["Condition"] == condition,
        "Maximum Mito%"
    ].iloc[0]

    ax.hist(
        values,
        bins=mito_bins,
        histtype="step",
        linewidth=2,
        label=condition,
        color=condition_colors[condition]
    )

    ax.axvline(
        threshold,
        linestyle="--",
        linewidth=2,
        color=condition_colors[condition]
    )

ax.set_title("Mito%")
ax.set_xlabel("Mito%")
ax.set_ylabel("Cell count")
ax.set_xlim(0, 10)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
EDA_norm_MCF7 = EDAQC_MAD_df_MCF7.copy()

metadata_cols = ['Lane', 'Pos', 'Condition', 'Cell name', 'outlier_MAD', 'outlier_low_counts', 'outlier_low_genes', 'outlier_high_mito', 'pct_mito', 'pct_ribosomal', 'pct_pseudogene', 'total_counts', 'n_detected_genes']
gene_cols = [col for col in EDA_norm_MCF7.columns if col not in metadata_cols]

library_size = EDA_norm_MCF7[gene_cols].sum(axis=1)

EDA_norm_MCF7[gene_cols] = np.log1p(
    EDA_norm_MCF7[gene_cols].div(library_size, axis=0) * 10000
)
EDA_norm_MCF7[gene_cols].describe().T

In [ ]:
df_plot_MCF7 = EDA_norm_MCF7.copy()

# 30 random cells for visualization
# np.random.seed(42) # Uncomment to make the result unique
selected_cells = np.random.choice(df_plot_MCF7["Cell name"], size=30, replace=False)

small_df = df_plot_MCF7[df_plot_MCF7["Cell name"].isin(selected_cells)]

long_cell_df = small_df[["Cell name", "Condition"] + gene_cols].melt(
    id_vars=["Cell name", "Condition"],
    value_vars=gene_cols,
    var_name="Gene",
    value_name="Expression"
)

plt.figure(figsize=(16, 6))

sns.violinplot(
    data=long_cell_df,
    x="Cell name",
    y="Expression",
    hue="Condition",
    palette={"Norm": "blue", "Hypo": "red"},
    inner="quartile",
    cut=0
)

plt.title("Random cells gene expression Distribution")
plt.xlabel("Cell")
plt.ylabel("Expression (normalized and log-transformed)")
plt.xticks(rotation=0)
plt.legend(title="Condition")
plt.tight_layout()
plt.show()

In [ ]:
long_gene_df = df_plot_MCF7[["Condition"] + gene_cols].melt(
    id_vars="Condition",
    value_vars=gene_cols,
    var_name="Gene",
    value_name="Expression"
)

plt.figure(figsize=(8, 5))

sns.histplot(
    data=long_gene_df,
    x="Expression",
    hue="Condition",
    palette={"Norm": "blue", "Hypo": "red"},
    bins=50,
    element="step",
    stat="density",
    common_norm=False
)

plt.title("Gene expression: Normoxia vs Hypoxia")
plt.xlabel("Gene expression (normalized and log-transformed)")
plt.ylabel("Density")
plt.tight_layout()
plt.show()

In [ ]:
EDA_shape_MCF7 = EDA_norm_MCF7.copy()
X_MCF7 = EDA_norm_MCF7[gene_cols]

In [ ]:
# Overall sparsity
overall_sparsity = (X_MCF7 == 0).sum().sum() / X_MCF7.size * 100

# Sparsity per cell
EDA_shape_MCF7['sparsity_per_cell'] = (X_MCF7 == 0).sum(axis=1) / X_MCF7.shape[1] * 100

# Sparsity per gene
sparsity_per_gene = (X_MCF7 == 0).sum(axis=0) / X_MCF7.shape[0] * 100

sparsity_gene_df = pd.DataFrame({
    'Gene': sparsity_per_gene.index,
    'sparsity_per_gene': sparsity_per_gene.values
})

print(f"Overall sparsity: {overall_sparsity:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 5))

sns.boxplot(
    data=EDA_shape_MCF7,
    x='Condition',
    y='sparsity_per_cell',
    hue='Condition',
    palette={'Norm': 'blue', 'Hypo': 'red'},
    legend=False,
    ax=axes[0]
)

axes[0].set_title('Sparsity per Cell by Condition')
axes[0].set_xlabel('Condition')
axes[0].set_ylabel('Sparsity per Cell (%)')

sns.histplot(
    data=EDA_shape_MCF7,
    x='sparsity_per_cell',
    hue='Condition',
    bins=25,
    palette={'Norm': 'blue', 'Hypo': 'red'},
    alpha=0.5,
    ax=axes[1]
)

axes[1].set_title('Distribution of Sparsity per Cell')
axes[1].set_xlabel('Sparsity per Cell (%)')
axes[1].set_ylabel('Number of Cells')

sns.histplot(
    data=sparsity_gene_df,
    x='sparsity_per_gene',
    bins=30,
    alpha=0.5,
    ax=axes[2]
)

axes[2].set_title('Distribution of Sparsity per Gene')
axes[2].set_xlabel('Sparsity per Gene (%)')
axes[2].set_ylabel('Number of Genes')

fig.suptitle('Sparsity Analysis on Data (normalized and log-transformed)', fontsize=16, y=1.05)

plt.tight_layout()
plt.show()

In [ ]:
overall_skewness = skew(X_MCF7.values.flatten(), nan_policy='omit')
overall_kurtosis = kurtosis(X_MCF7.values.flatten(), fisher=True, nan_policy='omit')

print(f"Overall skewness: {overall_skewness:.3f}")
print(f"Overall kurtosis: {overall_kurtosis:.3f}")

In [ ]:
# Per cell
EDA_shape_MCF7['skewness_per_cell'] = X_MCF7.apply(
    lambda row: skew(row, nan_policy='omit'), axis=1
)

EDA_shape_MCF7['kurtosis_per_cell'] = X_MCF7.apply(
    lambda row: kurtosis(row, fisher=True, nan_policy='omit'), axis=1
)

# Per gene
skewness_per_gene = X_MCF7.apply(
    lambda col: skew(col, nan_policy='omit'), axis=0
)

kurtosis_per_gene = X_MCF7.apply(
    lambda col: kurtosis(col, fisher=True, nan_policy='omit'), axis=0
)

gene_shape_df = pd.DataFrame({
    'Gene': gene_cols,
    'sparsity_per_gene': sparsity_per_gene.values,
    'skewness_per_gene': skewness_per_gene.values,
    'kurtosis_per_gene': kurtosis_per_gene.values
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(
    data=EDA_shape_MCF7,
    x='skewness_per_cell',
    hue='Condition',
    bins=25,
    palette={'Norm': 'blue', 'Hypo': 'red'},
    alpha=0.5,
    ax=axes[0]
)

axes[0].set_title('Distribution of Skewness per Cell')
axes[0].set_xlabel('Skewness per Cell')
axes[0].set_ylabel('Number of Cells')

sns.histplot(
    data=EDA_shape_MCF7,
    x='kurtosis_per_cell',
    hue='Condition',
    bins=25,
    palette={'Norm': 'blue', 'Hypo': 'red'},
    alpha=0.5,
    ax=axes[1]
)

axes[1].set_title('Distribution of Kurtosis per Cell')
axes[1].set_xlabel('Kurtosis per Cell')
axes[1].set_ylabel('Number of Cells')

plt.tight_layout()
plt.show()

In [ ]:
metadata_cols = ["Lane", "Pos", "Condition", "Cell name", "outlier_MAD", "outlier_low_counts", "outlier_low_genes", "outlier_high_mito", "pct_mito", "pct_ribosomal", "pct_pseudogene", "total_counts", "n_detected_genes"]
gene_cols = [col for col in EDA_norm_MCF7.columns if col not in metadata_cols]

gene_expr_MCF7 = EDA_norm_MCF7[gene_cols]

duplicated_genes_mask = gene_expr_MCF7.T.duplicated(keep=False)
duplicated_genes = gene_expr_MCF7.T[duplicated_genes_mask]

print("Number of duplicated genes:", duplicated_genes.shape[0])

In [ ]:
duplicated_genes_to_remove_MCF7 = gene_expr_MCF7.T.duplicated(keep="first")

genes_to_remove = gene_expr_MCF7.T.index[duplicated_genes_to_remove_MCF7].tolist()

print("Number of genes that can be removed:", len(genes_to_remove))
print("Examples:")
print(genes_to_remove[:20])

EDA_norm_no_identical_genes_MCF7 = EDA_norm_MCF7.drop(columns=genes_to_remove)

print("Shape before filtering:", EDA_norm_MCF7.shape)
print("Shape after filtering:", EDA_norm_no_identical_genes_MCF7.shape)

In [ ]:
metadata_cols = ['Lane', 'Pos', 'Condition', 'Cell name', 'PCRPlate_Pos', 'outlier_MAD', 'outlier_low_counts', 'outlier_low_genes', 'outlier_high_mito', 'pct_mito', 'pct_ribosomal', 'pct_pseudogene', 'total_counts', 'n_detected_genes']

gene_cols = [
    col for col in EDA_norm_no_identical_genes_MCF7.columns
    if col not in metadata_cols
]

hypo_df_MCF7 = EDA_norm_no_identical_genes_MCF7[EDA_norm_no_identical_genes_MCF7['Condition'] == 'Hypo']
normo_df_MCF7 = EDA_norm_no_identical_genes_MCF7[EDA_norm_no_identical_genes_MCF7['Condition'] == 'Norm']

print("Numero cellule Hypo:", hypo_df_MCF7.shape[0])
print("Numero cellule Normo:", normo_df_MCF7.shape[0])

In [ ]:
results = []

for gene in gene_cols:
    
    hypo_values_MCF7 = hypo_df_MCF7[gene]
    normo_values_MCF7 = normo_df_MCF7[gene]
    
    mean_hypo_MCF7 = hypo_values_MCF7.mean()
    mean_normo_MCF7 = normo_values_MCF7.mean()
    
    diff_MCF7 = mean_hypo_MCF7 - mean_normo_MCF7
    
    # Cells percentage with expression > 0
    pct_hypo_MCF7 = (hypo_values_MCF7 > 0).mean()
    pct_normo_MCF7 = (normo_values_MCF7 > 0).mean()
    
    results.append({
        'Gene': gene,
        'Mean_Hypo': mean_hypo_MCF7,
        'Mean_Normo': mean_normo_MCF7,
        'Difference_Hypo_minus_Normo': diff_MCF7,
        'Abs_Difference': abs(diff_MCF7),
        'Pct_Expressed_Hypo': pct_hypo_MCF7,
        'Pct_Expressed_Normo': pct_normo_MCF7
    })

gene_condition_results_MCF7 = pd.DataFrame(results)

gene_condition_results_MCF7.head()

In [ ]:
# Gene filtering based on mean difference and expression percentage
filtered_gene_results_MCF7 = gene_condition_results_MCF7[
    (gene_condition_results_MCF7['Abs_Difference'] > 0.25) &
    (
        (gene_condition_results_MCF7['Pct_Expressed_Hypo'] > 0.10) |
        (gene_condition_results_MCF7['Pct_Expressed_Normo'] > 0.10)
    )
].copy()

In [ ]:
top_hypo_genes_MCF7 = (
    filtered_gene_results_MCF7
    .sort_values('Difference_Hypo_minus_Normo', ascending=False)
    .head(20)
)

top_hypo_genes_MCF7

In [ ]:
top_normo_genes_MCF7 = (
    filtered_gene_results_MCF7
    .sort_values('Difference_Hypo_minus_Normo', ascending=True)
    .head(20)
)

top_normo_genes_MCF7

In [ ]:
top_hypo_10_MCF7 = top_hypo_genes_MCF7.head(10).copy()
top_hypo_10_MCF7['Associated_condition'] = 'Hypo'

top_normo_10_MCF7 = top_normo_genes_MCF7.head(10).copy()
top_normo_10_MCF7['Associated_condition'] = 'Normo'

top_condition_genes_10_MCF7 = pd.concat(
    [top_hypo_10_MCF7, top_normo_10_MCF7],
    axis=0
)

selected_genes_MCF7 = top_condition_genes_10_MCF7['Gene'].tolist()

# Dot plot

plot_data = []

for gene in selected_genes_MCF7:
    
    associated_condition = (
        top_condition_genes_10_MCF7
        .set_index('Gene')
        .loc[gene, 'Associated_condition']
    )
    
    for condition in ['Norm', 'Hypo']:
        
        sub = EDA_norm_no_identical_genes_MCF7[
            EDA_norm_no_identical_genes_MCF7['Condition'] == condition
        ]
        
        plot_data.append({
            'Gene': gene,
            'Condition': condition,
            'Associated_condition': associated_condition,
            'Mean_expression': sub[gene].mean(),
            'Pct_expressed': (sub[gene] > 0).mean() * 100
        })

dotplot_df_MCF7 = pd.DataFrame(plot_data)

gene_order_MCF7 = top_condition_genes_10_MCF7['Gene'].tolist()

dotplot_df_MCF7['Gene'] = pd.Categorical(
    dotplot_df_MCF7['Gene'],
    categories=gene_order_MCF7,
    ordered=True
)

normo_plot = dotplot_df_MCF7[dotplot_df_MCF7['Condition'] == 'Norm']
hypo_plot = dotplot_df_MCF7[dotplot_df_MCF7['Condition'] == 'Hypo']


plt.figure(figsize=(7, 9))

sns.scatterplot(
    data=normo_plot,
    x='Condition',
    y='Gene',
    size='Pct_expressed',
    hue='Mean_expression',
    palette='Blues',
    sizes=(40, 320),
    legend=False
)

sns.scatterplot(
    data=hypo_plot,
    x='Condition',
    y='Gene',
    size='Pct_expressed',
    hue='Mean_expression',
    palette='Reds',
    sizes=(40, 320),
    legend=False
)

plt.axhline(
    y=9.5,
    color='black',
    linewidth=1,
    linestyle='--'
)

for pct in [25, 50, 75, 100]:
    plt.scatter(
        [],
        [],
        s=40 + (pct / 100) * 280,
        color='gray',
        alpha=0.6,
        label=f'{pct}% expressed'
    )

plt.legend(
    title='Dot size',
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

plt.title("Dot plot of genes associated with Hypoxia and Normoxia")
plt.xlabel("Condition")
plt.ylabel("Genes")

plt.tight_layout()
plt.show()

### Filtered and Normalized Dataset

Eventually, we check the performance of our filtering process by comparing our filtered datasets with those provided in the folder

#### HCC1806

In [ ]:
EDA_filtered_compare = EDA_norm_no_identical_genes.copy()

print("Group filtered dataset shape:", EDA_filtered_compare.shape)
print("Professor filtered transposed shape:", filtered_data_Buffa_T.shape)

In [ ]:
metadata_cols = ["PCR Plate", "Pos", "Condition", "Cell Name", "outlier_MAD", "outlier_low_counts", "outlier_low_genes", "outlier_high_mito", "pct_mito", "pct_ribosomal", "pct_pseudogene", "total_counts", "n_detected_genes"]

my_gene_cols = [col for col in EDA_filtered_compare.columns if col not in metadata_cols]
prof_gene_cols = filtered_data_Buffa_T.columns.tolist()

my_genes = set(my_gene_cols)
prof_genes = set(prof_gene_cols)

common_genes = my_genes.intersection(prof_genes)
only_my_genes = my_genes - prof_genes
only_prof_genes = prof_genes - my_genes

gene_comparison = pd.DataFrame({
    "Category": [
        "Genes in both datasets",
        "Genes only in our filtered dataset",
        "Genes only in professor filtered dataset"
    ],
    "Count": [
        len(common_genes),
        len(only_my_genes),
        len(only_prof_genes)
    ]
})

gene_comparison

The comparison with the professor-provided filtered dataset is useful because it tells us whether our QC/filtering strategy preserved the same biological information while applying our own criteria.

For HCC1806, the overlap at gene level is very high: almost all genes selected in the reference filtered file are also present after our filtering. This means that our procedure did not remove the main expression structure used in the official preprocessing. At the same time, our dataset retains additional genes, so our filtering is slightly less restrictive. Biologically, this is acceptable as long as the extra genes do not introduce noise that dominates the downstream analyses; this is why the comparison is followed by supervised and unsupervised checks rather than treated as a final validation by itself.

#### MCF7

In [ ]:
EDA_filtered_compare_MCF7 = EDA_norm_no_identical_genes_MCF7.copy()

print("Group filtered dataset shape:", EDA_filtered_compare_MCF7.shape)
print("Professor filtered transposed shape:", filtered_data_Buffa_T_MCF7.shape)

In [ ]:
metadata_cols = ["Lane", "Pos", "Condition", "Cell Name", "outlier_MAD", "outlier_low_counts", "outlier_low_genes", "outlier_high_mito", "pct_mito", "pct_ribosomal", "pct_pseudogene", "total_counts", "n_detected_genes"]

my_gene_cols_MCF7 = [col for col in EDA_filtered_compare_MCF7.columns if col not in metadata_cols]
prof_gene_cols_MCF7 = filtered_data_Buffa_T_MCF7.columns.tolist()

my_genes_MCF7 = set(my_gene_cols_MCF7)
prof_genes_MCF7 = set(prof_gene_cols_MCF7)

common_genes = my_genes_MCF7.intersection(prof_genes_MCF7)
only_my_genes = my_genes_MCF7 - prof_genes_MCF7
only_prof_genes = prof_genes_MCF7 - my_genes_MCF7

gene_comparison_MCF7 = pd.DataFrame({
    "Category": [
        "Genes in both datasets",
        "Genes only in our filtered dataset",
        "Genes only in professor filtered dataset"
    ],
    "Count": [
        len(common_genes),
        len(only_my_genes),
        len(only_prof_genes)
    ]
})

gene_comparison_MCF7

### Filtering comparison: MCF7 interpretation

For MCF7, the same comparison should be interpreted together with the stronger filtering effect observed during the QC section. If the overlap with the professor-provided filtered matrix remains high, it supports the idea that our filtering removed low-quality cells while preserving the main gene-expression signal. If the number of genes retained only by our procedure is large, it should be treated cautiously: these genes may contain useful line-specific information, but they may also add noise in downstream models.

**To be completed after running the notebook:** report the exact MCF7 overlap values from `gene_comparison_MCF7` and state whether the difference with HCC1806 is mainly due to cell filtering, gene filtering, or both.

---

## Unsupervised Learning Analysis

This section asks whether the Hypoxia/Normoxia structure appears in the data without directly using the labels to build the representation. The labels are used only after PCA, clustering, NMF or UMAP have been computed, so that we can interpret the biological meaning of the structures that emerge.

The analysis focuses on three biological questions: whether hypoxia is a dominant source of variation, whether the signal is shared by HCC1806 and MCF7, and whether the recovered programs resemble known hypoxia biology rather than only technical variation or cell-cycle effects.

### Setup and data summary

The raw SmartSeq objects used here were prepared in the global loading section from the unfiltered matrices merged with metadata. They keep raw counts and log-normalised values inside the same `AnnData` objects. This allows the section to compare normalisation choices, clustering results and biological scores without reloading the same files.

In [ ]:
summary

Identical cells and 100% of the official variable genes are contained in our gene set (we add only the few Buffa genes not already among the top-3000). So our loader is the official pipeline plus the extra unscaled layers the scaled output discards — alignment demonstrated, not just asserted.

> **Takeaway —** both lines are balanced (~50/50) and well-sequenced; QC removes ~10/20 cells; **all 50 Buffa genes are retained**; and our preprocessing provably matches the group's official one while keeping the layers needed for Pearson/NMF.

Two facts that shape what follows:

- **Both datasets are full-length SmartSeq**, so they are only moderately sparse (~54–59% zeros) and deep (thousands of genes detected per cell). There is no droplet-style >97% dropout to fight here — which, as we will see, means **sequencing depth is a minor confounder** and the analysis can focus on biological structure.
- **MCF7 is larger** (363 vs 233 QC-passing cells) and the two lines are biologically distinct (luminal MCF7 vs basal-like HCC1806). The interesting comparison is therefore *between cell lines*: same hypoxic program, or different ones?

## First look: structure, or a technical artifact?

Before any machinery, a *diagnostic* first look rather than the flattering answer. We project the harder of the two lines — **HCC1806**, which (we will confirm) separates less sharply than MCF7 — with a naive `log1p` → PCA, colored two ways: by **condition** and by **sequencing depth**.

The question to keep in mind: *does the apparent structure track biology, or a technical variable?* On these deep full-length cells the answer is reassuring — depth does **not** explain the layout — but the conditions only partly separate in two raw PCs, which is exactly why the rest of the notebook is needed: to confirm the structure is hypoxia and not, say, cell cycle. We show the *question* here, not the answer.

In [ ]:
# Naive log1p -> PCA on the harder line, colored by condition vs by sequencing depth
a0 = raw["HCC1806"]
ln0 = ad.AnnData(np.asarray(a0.layers["counts"]).copy())
sc.pp.normalize_total(ln0, target_sum=1e4); sc.pp.log1p(ln0)
Z0 = np.asarray(ln0.X); Z0 = np.clip((Z0 - Z0.mean(0)) / (Z0.std(0) + 1e-8), -10, 10)
P0 = PCA(n_components=2, random_state=0).fit_transform(Z0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
for cond, col in [("Normoxia", "#3b78c2"), ("Hypoxia", "#c23b3b")]:
    m = (a0.obs.condition == cond).values
    axes[0].scatter(P0[m, 0], P0[m, 1], s=10, alpha=0.6, c=col, label=cond)
axes[0].legend(markerscale=2); axes[0].set_title("colored by CONDITION")
s = axes[1].scatter(P0[:, 0], P0[:, 1], s=10, alpha=0.7,
                    c=np.log1p(a0.obs["total_counts"].values), cmap="viridis")
axes[1].set_title("colored by SEQUENCING DEPTH (log total_counts)")
plt.colorbar(s, ax=axes[1], shrink=0.75)
for ax in axes: ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
fig.suptitle("First look — HCC1806, naive log1p -> PCA: is the structure biology or a technical artifact?", y=1.02)
plt.tight_layout(); plt.show()

> **Takeaway —** unlike droplet data, here a naive projection is *not* obviously driven by depth (the right panel has no clean depth gradient). The conditions partly separate, but "seeing the data first" still cannot tell us whether that is hypoxia or a confounder — which is why visualization is deferred to Sec. 10 and the next sections de-confound first.

## Normalization: `log1p(CP10K)` or Pearson residuals?

Before reducing dimensionality we must decide which matrix to work on. It is the most consequential choice of the whole unsupervised analysis, because PCA, clustering and UMAP are all sensitive to scale and to the structure of the noise.

**The problem.** scRNA-seq counts are *heteroskedastic*: a gene's variance grows with its mean (Poisson / negative-binomial noise). If left uncorrected, PCA is dominated by the few highly expressed genes (simply because they have larger absolute variance), not by the biologically informative ones.

**The two options compared.**

| | `log1p(CP10K)` | Pearson residuals (analytic) |
|---|---|---|
| Idea | normalize by depth (counts-per-10k) and compress with the log | model each gene as negative-binomial and use the standardized residual $(x-\hat\mu)/\sqrt{\hat\mu + \hat\mu^2/\theta}$ |
| Pro | simple, standard, interpretable | stabilizes variance *by construction*, dampens the weight of depth and dropout |
| Con | variance stays tied to the mean; the most-expressed genes dominate the projection | less interpretable as "expression" |

The reason we consider Pearson residuals — which are not in the course syllabus — is concrete: the syllabus covers PCA, and PCA assumes features have comparable and informative variance. On raw counts this assumption is *systematically violated*. Pearson residuals (sctransform / Lause et al. 2021) are the principled way to restore it for count data. **We do not replace `log1p`**: we put it alongside and let a quantitative diagnostic decide (Section 4).

We check the argument empirically with the per-gene **mean–variance** relationship, before and after each transformation.

In [ ]:
def add_normalizations(a):
    "Add the 'lognorm' (already present) and 'pearson' layers, computed from raw counts."
    pr = ad.AnnData(np.asarray(a.layers["counts"]).copy())
    sc.experimental.pp.normalize_pearson_residuals(pr)   # theta=100 by default
    a.layers["pearson"] = np.asarray(pr.X).copy()
    return a

for a in raw.values():
    add_normalizations(a)

# Mean-variance diagnostic on HCC1806
a = raw["HCC1806"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, lyr, title in zip(axes, ["counts", "lognorm", "pearson"],
                          ["Raw counts", "log1p(CP10K)", "Pearson residuals"]):
    M = np.asarray(a.layers[lyr])
    ax.scatter(M.mean(0), M.var(0), s=6, alpha=0.3)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("per-gene mean"); ax.set_ylabel("per-gene variance"); ax.set_title(title)
fig.suptitle("HCC1806 — per-gene mean-variance relationship", y=1.02)
plt.tight_layout(); plt.show()

**Reading.** On raw counts and on `log1p` the variance grows with the mean: the most expressed genes will dominate any PCA. On Pearson residuals the cloud flattens — variance is no longer a simple function of the mean — which is exactly the property we want before projecting. This is the *a priori* argument; the next section confirms it *a posteriori* on confounders.

> **Takeaway —** Pearson residuals decouple variance from mean, so PCA can be driven by biology rather than by which genes are simply most expressed. We adopt them as the primary representation and keep `log1p` for interpretable scores.

## Dimensionality reduction and confounder diagnostics

We compute PCA on both representations: on `log1p` we standardize genes then PCA (standard practice); on **Pearson residuals** we run PCA *directly* (they are already variance-stabilized; re-scaling would re-introduce the weight of noisy genes).

Then the check a naive analysis omits: **correlating the leading components with confounders**. In scRNA-seq, PC1 often captures *sequencing depth* (`total_counts`) or the *cell cycle* rather than the phenotype. If the condition is "buried" under these axes we need to know, because it changes the interpretation of every clustering downstream.

In [ ]:
def pca_embeddings(a, n_comps=30):
    Xl = np.asarray(a.layers["lognorm"]).copy()
    Xl = np.clip((Xl - Xl.mean(0)) / (Xl.std(0) + 1e-8), -10, 10)
    pl = PCA(n_components=n_comps, random_state=0).fit(Xl)
    a.obsm["X_pca_log"] = pl.transform(Xl)
    Xp = np.asarray(a.layers["pearson"]).copy()
    pp = PCA(n_components=n_comps, random_state=0).fit(Xp)
    a.obsm["X_pca_pr"] = pp.transform(Xp)
    return pl.explained_variance_ratio_, pp.explained_variance_ratio_

def score_cell_cycle(a):
    sg = [g for g in S_GENES if g in a.var_names]
    gg = [g for g in G2M_GENES if g in a.var_names]
    tmp = ad.AnnData(np.asarray(a.layers["lognorm"]).copy())
    tmp.var_names = a.var_names; tmp.obs_names = a.obs_names
    sc.tl.score_genes_cell_cycle(tmp, s_genes=sg, g2m_genes=gg)
    for c in ["S_score", "G2M_score", "phase"]:
        a.obs[c] = tmp.obs[c].values
    return len(sg), len(gg)

evr, cc_genes = {}, {}
for name, a in raw.items():
    evr[name] = pca_embeddings(a)
    cc_genes[name] = score_cell_cycle(a)

def confounder_table(a):
    y = (a.obs.condition == "Hypoxia").astype(int).values
    drivers = {"depth (total_counts)": a.obs["total_counts"].values,
               "cycle (G2M_score)": a.obs["G2M_score"].values,
               "CONDITION (hypoxia)": y.astype(float)}
    rows = {}
    for rep, key in [("log1p", "X_pca_log"), ("pearson", "X_pca_pr")]:
        P = a.obsm[key][:, :10]
        for dname, d in drivers.items():
            r = np.array([abs(np.corrcoef(P[:, i], d)[0, 1]) for i in range(P.shape[1])])
            rows[(rep, dname)] = np.nanmax(r)   # max |corr| among the first 10 PCs
    return rows

conf = pd.DataFrame({name: confounder_table(a) for name, a in raw.items()}).T
conf.columns = pd.MultiIndex.from_tuples(conf.columns)
conf.round(2)

Each entry is the **maximum absolute correlation** between one of the first 10 PCs and a factor. Two readings: (i) on these full-length data **depth correlates only weakly** with the leading PCs — confirming the first-look impression that sequencing depth is not the driver here; (ii) the **condition correlates strongly** with some PC, more so in MCF7 than in HCC1806 — an early hint that MCF7's hypoxic signal is the more dominant axis. We adopt **Pearson residuals as the primary representation** for clustering and visualization, keeping `log1p` for the interpretable scores (Section 5).

In [ ]:
# Comparative scree plot (cumulative explained variance) for the two normalization schemes
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, name in zip(axes, raw):
    el, ep = evr[name]
    ax.plot(np.cumsum(el)*100, marker="o", ms=3, label="log1p")
    ax.plot(np.cumsum(ep)*100, marker="s", ms=3, label="pearson")
    ax.set_title(f"{name}\n(S/G2M cycle genes present: {cc_genes[name][0]}/{cc_genes[name][1]})", fontsize=9)
    ax.set_xlabel("# PC"); ax.grid(alpha=0.3)
axes[0].set_ylabel("cumulative variance %"); axes[0].legend()
plt.tight_layout(); plt.show()

*Note on the cell cycle.* Starting from the full (unfiltered) gene set, many more cell-cycle genes survive into the variable-gene set than in pre-filtered data (counts in the titles), so the cycle score is more reliable here — we use it as a genuine confounder check, not just a token one.

> **Takeaway —** on full-length SmartSeq, depth is a *minor* confounder; the condition is a strong PC in both lines and especially dominant in MCF7. Pearson residuals are the cleaner base for the unsupervised work.

## Buffa metagene score: three estimators compared

The Buffa signature (Buffa et al., 2010) is a 50-gene meta-analytic marker of tumor hypoxia. Starting from the raw counts, **all 50 genes are present** in both datasets (not ~half, as in pre-filtered matrices), so the score is now as complete as the signature allows.

Compressing it into a single per-cell "hypoxia score" is useful, but *how* matters. We compare three estimators on `log1p` restricted to the Buffa genes:

1. **PC1** of the z-scored signature — captures maximal covariation, but the **sign is arbitrary** and on noisy data PC1 can latch onto technical covariation.
2. **Mean of the z-scored genes** — simple, well-defined sign, no parameters.
3. **`score_genes`** (Scanpy) — signature mean minus a background control set matched by expression level; the field standard.

Criterion: **(absolute) correlation with the condition** and **consistency across the two lines**.

In [ ]:
def buffa_scores(a):
    present = [g for g in BUFFA if g in a.var_names]
    ln = ad.AnnData(np.asarray(a.layers["lognorm"]).copy())
    ln.var_names = a.var_names; ln.obs_names = a.obs_names
    sub = np.asarray(ln[:, present].X)
    z = (sub - sub.mean(0)) / (sub.std(0) + 1e-8)
    mean_z = z.mean(1)
    pc1 = PCA(n_components=2, random_state=0).fit_transform(z)[:, 0]
    if np.corrcoef(pc1, mean_z)[0, 1] < 0:
        pc1 = -pc1
    sc.tl.score_genes(ln, present, score_name="sg")
    sg = ln.obs["sg"].values
    a.obs["buffa_meanz"] = mean_z; a.obs["buffa_pc1"] = pc1; a.obs["buffa_sg"] = sg
    y = (a.obs.condition == "Hypoxia").astype(int).values
    return {"n_present": len(present),
            "|r| mean_z": abs(np.corrcoef(mean_z, y)[0, 1]),
            "|r| PC1": abs(np.corrcoef(pc1, y)[0, 1]),
            "|r| score_genes": abs(np.corrcoef(sg, y)[0, 1])}

score_cmp = pd.DataFrame({name: buffa_scores(a) for name, a in raw.items()}).T
score_cmp.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, name in zip(axes, raw):
    a = raw[name]
    for cond, col in [("Normoxia", "#3b78c2"), ("Hypoxia", "#c23b3b")]:
        sns.kdeplot(a.obs.loc[a.obs.condition == cond, "buffa_sg"], ax=ax,
                    fill=True, alpha=0.4, color=col, label=cond)
    ax.set_title(name, fontsize=10); ax.set_xlabel("Buffa score_genes")
axes[0].legend()
plt.tight_layout(); plt.show()

**Choice.** `score_genes` gives the strongest, most consistent correlation with the condition in both lines, with a defined sign; PC1 can match it but carries sign ambiguity. We adopt **`score_genes`**, keeping `mean_z` as a sanity check. The score is a *label-free* feature (the signature is prior literature knowledge, not estimated from our labels), so it is legitimate in an unsupervised analysis. Notably the score separates the conditions well in **both** lines, and slightly better in MCF7.

> **Takeaway —** with all 50 genes available, the background-corrected `score_genes` is a strong, sign-stable hypoxia readout in both lines (correlation with condition ≈ 0.86 HCC1806, ≈ 0.93 MCF7).

## Unsupervised clustering with methodological checks

This is the central part. Three methodological decisions, all justified.

**Method hierarchy — by design, not a collection of unrelated methods.** The *primary* unsupervised result is the partition produced by **KMeans on the Pearson-residual PCA**, evaluated against the condition with ARI/AMI. We run Agglomerative, GMM and graph-based **Leiden alongside it not to cherry-pick the best score, but as mutual controls**: if independent algorithms with different assumptions agree, the conclusion is robust; if they disagree, that disagreement is itself informative. Reporting several algorithms is a deliberate robustness design, not "I tried everything and kept the nicest number".

### Do not assume *a priori* k = 2

Two experimental conditions does **not** mean the data form two natural clusters: cell cycle, sub-states and noise create other structure. Fixing `k=2` forces the answer. We estimate the number of groups *from the data* with two independent criteria, both on the Pearson-PCA, **without using the labels**: **silhouette** (KMeans) and **BIC** (GMM).

In [ ]:
def choose_k(a, ks=range(2, 8), n_pcs=20):
    X = a.obsm["X_pca_pr"][:, :n_pcs]
    idx = RNG.choice(X.shape[0], min(2000, X.shape[0]), replace=False)
    sil, bic = [], []
    for k in ks:
        km = KMeans(k, n_init=10, random_state=0).fit(X)
        sil.append(silhouette_score(X[idx], km.labels_[idx]))
        bic.append(GaussianMixture(k, covariance_type="full", random_state=0).fit(X).bic(X))
    return list(ks), sil, bic

fig, axes = plt.subplots(2, 2, figsize=(10, 6))
k_star = {}
for j, name in enumerate(raw):
    ks, sil, bic = choose_k(raw[name])
    axes[0, j].plot(ks, sil, marker="o"); axes[0, j].set_title(name, fontsize=10)
    axes[0, j].set_xlabel("k"); axes[0, j].set_ylabel("silhouette" if j == 0 else "")
    axes[1, j].plot(ks, bic, marker="s", color="darkorange")
    axes[1, j].set_xlabel("k"); axes[1, j].set_ylabel("BIC (GMM)" if j == 0 else "")
    k_star[name] = ks[int(np.argmax(sil))]
    axes[0, j].axvline(k_star[name], ls="--", c="grey")
plt.suptitle("Selecting k without labels  —  top: silhouette (KMeans), bottom: BIC (GMM)", y=1.02)
plt.tight_layout(); plt.show()
print("k with maximal silhouette per dataset:", k_star)

### Which algorithm? Generic vs graph-based

The syllabus covers KMeans, hierarchical clustering and Gaussian mixtures — all assuming **convex / Gaussian** clusters, an assumption the scRNA manifold violates. We add **neighborhood-graph clustering (Leiden)**, the field standard, which makes no shape assumption, precisely to *contrast* it with the course methods.

### The right metric: ARI/AMI, not accuracy

Majority-mapping **accuracy** is misleading with imbalanced clusters: it can be high while nothing was discovered. We use **chance-corrected, permutation-invariant** metrics — **ARI** and **AMI** — and report accuracy only to **show how much it overestimates**.

In [ ]:
def majority_accuracy(y_true, labels):
    mapped = np.zeros_like(y_true)
    for c in np.unique(labels):
        m = labels == c
        mapped[m] = np.bincount(y_true[m]).argmax()
    cm = confusion_matrix(y_true, mapped, labels=[0, 1])
    return cm.trace() / cm.sum()

def run_clustering(a, k):
    X = a.obsm["X_pca_pr"][:, :20]
    y = (a.obs.condition == "Hypoxia").astype(int).values
    algos = {
        f"KMeans (k={k})":        KMeans(k, n_init=10, random_state=0).fit_predict(X),
        f"Agglomerative (k={k})": AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X),
        f"GMM (k={k})":           GaussianMixture(k, covariance_type="full", random_state=0).fit_predict(X),
        "KMeans (k=2)":           KMeans(2, n_init=10, random_state=0).fit_predict(X),
    }
    tmp = ad.AnnData(X.copy()); tmp.obsm["X_pca"] = X
    sc.pp.neighbors(tmp, n_neighbors=15, use_rep="X_pca", random_state=0)
    sc.tl.leiden(tmp, resolution=1.0, random_state=0, flavor="igraph", n_iterations=2, directed=False)
    leid = tmp.obs["leiden"].astype(int).values
    a.obs["leiden"] = pd.Categorical(leid.astype(str))
    algos[f"Leiden ({leid.max()+1} communities)"] = leid

    idx = RNG.choice(X.shape[0], min(2000, X.shape[0]), replace=False)
    out = {}
    for nm, lab in algos.items():
        out[nm] = {"n_clusters": len(np.unique(lab)),
                   "silhouette": silhouette_score(X[idx], lab[idx]) if len(np.unique(lab)) > 1 else np.nan,
                   "ARI": adjusted_rand_score(y, lab),
                   "AMI": adjusted_mutual_info_score(y, lab),
                   "accuracy (maj.)": majority_accuracy(y, lab)}
    return out

records = []
for name in raw:
    for algo, m in run_clustering(raw[name], k_star[name]).items():
        records.append({"dataset": name, "algorithm": algo, **m})
clust = pd.DataFrame(records)
clust.round(3)

In [ ]:
piv = clust[clust.algorithm == "KMeans (k=2)"].set_index("dataset")[["accuracy (maj.)", "ARI"]]
ax = piv.plot.bar(figsize=(7, 4), color=["#bbbbbb", "#c23b3b"])
ax.set_title("KMeans k=2: majority-mapping accuracy vs the honest ARI")
ax.set_ylabel("metric value"); ax.axhline(0, c="k", lw=0.6)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

**Main interpretation.** In **both** lines the condition is recovered with no label guidance, but to a different degree: **MCF7** separates very cleanly (KMeans ARI ≈ 0.96 — the condition is essentially MCF7's dominant axis of variation), while **HCC1806** is recovered but more modestly (ARI ≈ 0.63). KMeans, Agglomerative and GMM agree closely within each line, so the result is a property of the data, not of one algorithm. Leiden, relaxing the convexity assumption, splits each line into 7–8 communities (lower ARI against the binary condition): there are **more axes of variation than the oxygen dichotomy** — and an unsupervised method reveals that instead of hiding it behind an imposed k=2.

> **Takeaway —** the hypoxic signature emerges unsupervised in both breast-cancer lines, cleanly in MCF7 (ARI ≈ 0.96) and more modestly in HCC1806 (ARI ≈ 0.63); the three convex methods agree.

### A note on "guided" feature selection — why we do not use it as discovery

A common alternative selects genes by a **t-test** between conditions (with Benjamini–Hochberg correction), then clusters on those genes. But this **uses the labels**: clustering on genes chosen *because* they separate the classes, then "discovering" that they separate the classes, is **circular**. We compute it and compare with the pure clustering of Sec. 6 — the ARI difference *is* the circularity — and present it only as a **semi-supervised upper bound**.

In [ ]:
def guided_upperbound(a, n_top=100, n_pcs=10):
    X = np.asarray(a.layers["lognorm"]); Xz = (X - X.mean(0)) / (X.std(0) + 1e-8)
    y = (a.obs.condition == "Hypoxia").astype(int).values
    _, p = ttest_ind(Xz[y == 1], Xz[y == 0], axis=0, equal_var=False)
    p = np.nan_to_num(p, nan=1.0)
    n_sig = int(multipletests(p, alpha=0.05, method="fdr_bh")[0].sum())
    top = np.argsort(p)[:n_top]
    P = PCA(min(n_pcs, n_top), random_state=0).fit_transform(Xz[:, top])
    ari = adjusted_rand_score(y, KMeans(2, n_init=10, random_state=0).fit_predict(P))
    return n_sig, ari

pure_ari = clust[clust.algorithm == "KMeans (k=2)"].set_index("dataset")["ARI"]
rows = []
for name, a in raw.items():
    n_sig, ari_g = guided_upperbound(a)
    rows.append({"dataset": name, "sig. genes (BH q<0.05)": n_sig,
                 "guided ARI (KMeans k=2)": round(ari_g, 3),
                 "pure ARI (Sec. 6)": round(float(pure_ari[name]), 3)})
m1 = pd.DataFrame(rows).set_index("dataset")
m1["gap = circularity"] = (m1["guided ARI (KMeans k=2)"] - m1["pure ARI (Sec. 6)"]).round(3)
m1

The **gap** differs by line and is itself informative. On **MCF7** it is negligible (pure clustering already recovers the condition, so label guidance adds nothing). On **HCC1806** it is moderate (≈ 0.25): the harder line is where label-guided selection most inflates the apparent agreement — exactly where one must be most suspicious of a "guided" result. The notebook's reference analysis therefore remains the **pure** one of Sec. 6.

> **Takeaway —** label-guided feature selection adds essentially nothing on MCF7 but inflates HCC1806 by ≈ 0.25 ARI. The harder the line, the more a guided pipeline flatters itself — a reason to report the unguided result.

## Clustering stability (consensus)

A partition is useful only if it is **reproducible** under resampling. We bootstrap 80% of the cells, re-cluster, and measure agreement via mean pairwise ARI — the stability *of the assignments*, not the variability of a mean expression (which looks "more stable" exactly where there is less information). We compare `k=2` with the data-chosen `k*`. Crucial distinction: stability is an **internal** metric (reproducibility); the ARI-against-condition of Sec. 6 is **external** (biological relevance). The two need not coincide.

In [ ]:
def stability(a, k, B=25, frac=0.8, n_pcs=20):
    X = a.obsm["X_pca_pr"][:, :n_pcs]; n = X.shape[0]
    labelings = []
    for _ in range(B):
        idx = np.sort(RNG.choice(n, int(frac*n), replace=False))
        labelings.append((idx, KMeans(k, n_init=5, random_state=None).fit_predict(X[idx])))
    aris = []
    for i in range(B):
        for j in range(i+1, B):
            (ia, la), (ib, lb) = labelings[i], labelings[j]
            common, ca, cb = np.intersect1d(ia, ib, return_indices=True)
            if len(common) > 10:
                aris.append(adjusted_rand_score(la[ca], lb[cb]))
    return float(np.mean(aris)), float(np.std(aris))

stab = []
for name in raw:
    m2, _ = stability(raw[name], 2)
    mk, _ = stability(raw[name], k_star[name])
    stab.append({"dataset": name, "mean ARI (k=2)": m2, "mean ARI (k*)": mk, "k*": k_star[name]})
pd.DataFrame(stab).set_index("dataset").round(3)

The `k=2` partitions are highly reproducible in both lines. Read together with Sec. 6 this is reassuring rather than (as on harder data) a warning: here the stable partition also *corresponds* to the condition, especially in MCF7. Still, the methodological point stands — stability alone is necessary, not sufficient; it must be paired with an external metric.

> **Takeaway —** both lines give stable k=2 partitions, and (unlike noisier data) those stable partitions do track the condition — internal and external evidence agree here.

## Interpretability: SparsePCA + enrichment test, and NMF (cross-line programs)

Standard PCA gives dense, hard-to-read components. **SparsePCA** adds an L1 penalty so each component involves few genes and reads as a transcriptional "program". We then make it a **quantitative claim** with a **hypergeometric test** of each component's top loadings against the Buffa signature: *which* unsupervised axis is the hypoxia program, and at what rank.

In [ ]:
def sparsepca_enrichment(a, n_comp=10, alpha=2.0, top=50, n_sub=2500):
    present = [g for g in BUFFA if g in a.var_names]
    X = np.asarray(a.layers["lognorm"]).copy()
    if X.shape[0] > n_sub:
        X = X[RNG.choice(X.shape[0], n_sub, replace=False)]
    X = (X - X.mean(0)) / (X.std(0) + 1e-8)
    spca = SparsePCA(n_components=n_comp, alpha=alpha, ridge_alpha=0.01,
                     max_iter=200, random_state=0).fit(X)
    L = pd.DataFrame(spca.components_.T, index=a.var_names,
                     columns=[f"C{i+1}" for i in range(n_comp)])
    N, K = a.n_vars, len(present)
    rows = []
    for c in L.columns:
        topset = set(L[c].abs().nlargest(top).index)
        hits = len(topset & set(present))
        rows.append({"component": c, "loadings != 0": int((np.abs(L[c]) > 1e-9).sum()),
                     f"Buffa in top-{top}": hits, "p (hypergeom.)": hypergeom.sf(hits - 1, N, K, top)})
    return pd.DataFrame(rows)

enr = {}
for name in raw:
    d = sparsepca_enrichment(raw[name]); d["dataset"] = name; enr[name] = d
pd.concat(enr.values()).set_index(["dataset", "component"]).round(4)

In both lines a SparsePCA component is significantly enriched in Buffa genes (small p), so a hypoxia axis emerges unsupervised — but it is generally **not the first** component: hypoxia is a *real but not top-ranked* source of variance, sharing the space with proliferation/cycle programs. This is consistent with the clustering picture.

### NMF: additive programs, and a cross-line comparison

PCA/SparsePCA loadings can be positive or negative — biologically ambiguous. **NMF** factorizes $X \approx W H$ with $W, H \ge 0$: each component is an **additive program** of co-expressed genes (a row of $H$), and $W$ says how strongly each cell switches it on. This fits biology (transcripts add up; a hypoxic cell *activates* programs rather than being the "negative" of a normoxic one).

**Technical caveat:** NMF needs non-negative input, so we apply it to `log1p(CP10K)` **without** mean-centering. We pick k by the reconstruction-error elbow (soft) and weigh interpretability.

This is where the two-cell-line design pays off. The *a posteriori* questions: does an NMF component separate the conditions in **each** line (the condition never enters the fit)? And — biologically — **is it the same program in HCC1806 and MCF7, or line-specific?**

In [ ]:
def nmf_recon_curve(a, ks=(2,4,6,8,10,15)):
    L = np.asarray(a.layers["lognorm"])
    return list(ks), [NMF(n_components=k, init="nndsvda", random_state=0,
                          max_iter=300).fit(L).reconstruction_err_ for k in ks]

def run_nmf(a, k=8, topn=15):
    L = np.asarray(a.layers["lognorm"])            # non-negative, NOT centered
    m = NMF(n_components=k, init="nndsvda", random_state=0, max_iter=400)
    W = m.fit_transform(L); H = m.components_
    y = (a.obs.condition == "Hypoxia").astype(int).values
    cors = np.array([abs(np.corrcoef(W[:, j], y)[0, 1]) for j in range(k)])
    bj = int(np.argmax(cors))
    a.obs["nmf_hyp"] = W[:, bj]
    top = list(np.array(a.var_names)[np.argsort(H[bj])[::-1][:topn]])
    return {"best_comp": int(bj), "best_|corr|": float(cors[bj]),
            "n Buffa in top-15": int(sum(g in BUFFA_SET for g in top)), "top_genes": top}

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, name in zip(axes, raw):
    ks, err = nmf_recon_curve(raw[name])
    ax.plot(ks, err, marker="o"); ax.set_title(name, fontsize=10)
    ax.set_xlabel("k (n. NMF components)"); ax.set_ylabel("reconstruction error")
plt.suptitle("Choosing k for NMF — soft elbow: we pick k=8", y=1.03)
plt.tight_layout(); plt.show()

nmf_res = {name: run_nmf(a, k=8) for name, a in raw.items()}
nmf_tab = pd.DataFrame(nmf_res).T[["best_comp", "best_|corr|", "n Buffa in top-15"]]
nmf_tab["top genes of the hypoxia program"] = [", ".join(nmf_res[n]["top_genes"]) for n in nmf_res]
shared = sorted(set(nmf_res["HCC1806"]["top_genes"]) & set(nmf_res["MCF7"]["top_genes"]))
print("Top-15 hypoxia-program genes shared by BOTH lines:", shared)
nmf_tab

### NMF pathway analysis

The previous NMF analysis identifies the NMF component that best corresponds to the hypoxic program in each cell line. Here we take the highest-weight genes from those NMF components and test whether they are enriched for known pathways. This is different from supervised pathway analysis because the gene programs are learned without using the class labels; the labels are used only afterwards to identify which component is most hypoxia-like.

In [ ]:
# Pathway analysis on NMF hypoxia-like components.
# The gene programs are derived from NMF loadings, not from a supervised classifier.

NMF_PATHWAY_TOP_N = 100
NMF_PATHWAY_GENE_SETS = ["MSigDB_Hallmark_2020", "GO_Biological_Process_2023"]


def nmf_program_genes_for_enrichment(a, k=8, topn=NMF_PATHWAY_TOP_N):
    L = np.asarray(a.layers["lognorm"])
    model = NMF(n_components=k, init="nndsvda", random_state=UNSUPERVISED_RANDOM_STATE, max_iter=500)
    W = model.fit_transform(L)
    H = model.components_
    y = (a.obs.condition == "Hypoxia").astype(int).values
    correlations = np.array([abs(np.corrcoef(W[:, j], y)[0, 1]) for j in range(k)])
    hypoxia_component = int(np.nanargmax(correlations))
    gene_weights = pd.DataFrame({
        "Gene": list(a.var_names),
        "NMF_weight": H[hypoxia_component],
    }).sort_values("NMF_weight", ascending=False)
    return hypoxia_component, float(correlations[hypoxia_component]), gene_weights.head(topn)

nmf_pathway_inputs = {}
nmf_pathway_tables = {}

for dataset_name, adata in raw.items():
    component, corr, top_gene_table = nmf_program_genes_for_enrichment(adata)
    nmf_pathway_inputs[dataset_name] = top_gene_table["Gene"].tolist()
    nmf_pathway_tables[dataset_name] = top_gene_table.assign(
        dataset=dataset_name,
        hypoxia_component=component,
        component_condition_correlation=corr,
    )

nmf_top_gene_table = pd.concat(nmf_pathway_tables.values(), axis=0)
display(nmf_top_gene_table.groupby("dataset").head(15))

nmf_enrichment_results = {}
for dataset_name, genes in nmf_pathway_inputs.items():
    enr = gp.enrichr(
        gene_list=genes,
        gene_sets=NMF_PATHWAY_GENE_SETS,
        organism="Human",
        outdir=None,
    )
    nmf_enrichment_results[dataset_name] = enr.results
    print(f"\n{dataset_name} NMF pathway enrichment")
    display(enr.results[["Gene_set", "Term", "Adjusted P-value", "Overlap", "Combined Score"]].head(10))


### Interpretation of the NMF pathway analysis

This enrichment step should clarify whether the NMF hypoxia-like components correspond to expected hypoxia biology, such as glycolysis, HIF-related transcriptional response, angiogenesis, metabolic stress, extracellular-matrix remodeling or cell-line-specific epithelial programs.

**To be completed after running the notebook:** summarize the top enriched terms for HCC1806 and MCF7, and explicitly state which pathways are shared and which appear cell-line specific. The key biological question is whether NMF recovers a common hypoxic core or two different hypoxia-related programs.

**Result — and this is the biologically interesting part.** In both lines, NMF recovers a component strongly tied to the condition (|corr| ≈ 0.86 HCC1806, ≈ 0.76 MCF7) **without being told the labels**. Reading the genes:

- **HCC1806's** program is led by *canonical HIF targets*: DDIT4, ANGPTL4, SLC2A1, BNIP3 together with the glycolytic core LDHA, PGK1, ALDOA, ENO1, TPI1 — a textbook hypoxia/glycolysis program.
- **MCF7's** program is dominated by *keratins and structural genes* (KRT8, KRT18, KRT19, ACTB) plus MIF, with the glycolytic enzymes (ALDOA, GAPDH, PKM) present but lower-ranked.
- The two top-15 lists **share a glycolytic/structural core** (printed above — e.g. ALDOA, GAPDH, KRT19, ACTB), but the dominant genes diverge.

So the answer to "same response or different?" is **both**: a **shared glycolytic hypoxic core** is activated in both breast-cancer lines, but the *emphasis* is line-specific — canonical HIF/angiogenic markers in basal-like HCC1806, keratin/cytoskeletal remodelling in luminal MCF7. That is exactly the kind of conserved-core / line-specific-periphery picture one expects biologically, and NMF makes it readable gene by gene as PCA cannot.

> **Takeaway —** unsupervised NMF recovers a hypoxia program in both lines; they share a glycolytic core but differ at the top (HCC1806: DDIT4/ANGPTL4/SLC2A1/BNIP3; MCF7: KRT8/18/19 + MIF). Conserved response, line-specific emphasis.

## DBSCAN: outliers and transition states

**Premise.** DBSCAN is *not* used to find the two groups — it is density-based, for an arbitrary number of arbitrarily-shaped clusters, and expression in PCA space has no sharp density islands. Its value here is its native ability to label points as **noise** (−1): a detector of **outliers / transition states** — cells not clearly in any dense population (intermediate hypoxia responses, since hypoxia is a *continuum*, or technical residuals). We set `eps` from the **k-distance plot** and `min_samples ≈ 2·dim`.

In [ ]:
def run_dbscan(a, n_pcs=15, pct=90):
    P = a.obsm["X_pca_pr"][:, :n_pcs]; ms = 2 * n_pcs
    d, _ = NearestNeighbors(n_neighbors=ms).fit(P).kneighbors(P)
    kdist = np.sort(d[:, -1]); eps = np.percentile(kdist, pct)
    lab = DBSCAN(eps=eps, min_samples=ms).fit_predict(P)
    a.obs["dbscan_noise"] = lab == -1
    noise = lab == -1; y = a.obs.condition.values
    return {"eps": eps, "min_samples": ms, "n_dense_clusters": len(set(lab)) - (1 if -1 in lab else 0),
            "% noise": 100*noise.mean(),
            "% Hypoxia among noise": 100*float((y[noise] == "Hypoxia").mean()) if noise.sum() else np.nan,
            "% Hypoxia overall": 100*(y == "Hypoxia").mean(), "n_noise": int(noise.sum()), "kdist": kdist}

dbscan_res = {name: run_dbscan(a) for name, a in raw.items()}

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, name in zip(axes, raw):
    r = dbscan_res[name]
    ax.plot(r["kdist"]); ax.axhline(r["eps"], c="r", ls="--", label=f"eps={r['eps']:.1f}")
    ax.set_title(name, fontsize=10); ax.set_xlabel("cells (sorted)"); ax.legend(fontsize=8)
axes[0].set_ylabel("distance to k-th neighbor")
plt.suptitle("k-distance plot: eps at the 90th percentile", y=1.03)
plt.tight_layout(); plt.show()

pd.DataFrame(dbscan_res).T.drop(columns="kdist").round(2)

As expected, DBSCAN finds a single dense region plus a **small** fraction of noise cells — and here that fraction is genuinely small (≈ 1% and below), because full-length QC'd data are clean. With so few outlier cells, their condition composition is not interpretable (too few to be meaningful); the honest statement is simply that these high-quality datasets contain very few ambiguous/transition cells. We overlay them on the embedding next.

> **Takeaway —** on clean full-length data DBSCAN flags only ~1% outliers — too few to over-read, and a sign of high data quality rather than a hidden third group.

## Visualization: UMAP and t-SNE

Two complementary non-linear embeddings, both **in 2D** (a flattened 3D projection makes perceived distances misleading; 2D is the reading/publication standard). **UMAP** preserves global structure better; **t-SNE** emphasizes local neighborhoods and is sensitive to `perplexity` (we show the effect) — its inter-cluster distances are not quantitatively interpretable.

The most informative choice is the **color**. We color the *same* UMAP by condition, depth, cell cycle, Leiden clusters, the NMF hypoxia program and DBSCAN outliers — a cross-cutting diagnostic canvas tying the methods together.

*Effect of perplexity* — t-SNE's most important parameter — on HCC1806:

In [ ]:
def tsne_embed(a, n_pcs=20, perplexity=30):
    return TSNE(n_components=2, perplexity=perplexity, init="pca",
                random_state=0).fit_transform(a.obsm["X_pca_pr"][:, :n_pcs])

a = raw["HCC1806"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, perp in zip(axes, [5, 30, 50]):
    emb = tsne_embed(a, perplexity=perp)
    for cond, col in [("Normoxia", "#3b78c2"), ("Hypoxia", "#c23b3b")]:
        m = (a.obs.condition == cond).values
        ax.scatter(emb[m, 0], emb[m, 1], s=10, alpha=0.6, c=col, label=cond)
    ax.set_title(f"perplexity = {perp}"); ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(markerscale=2)
plt.suptitle("t-SNE — the shape changes with perplexity: do not over-interpret distances", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# t-SNE 2D by condition, both lines (perplexity=30)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
for ax, name in zip(axes, raw):
    a = raw[name]; emb = tsne_embed(a, perplexity=30)
    for c, col in [("Normoxia", "#3b78c2"), ("Hypoxia", "#c23b3b")]:
        m = (a.obs.condition == c).values
        ax.scatter(emb[m, 0], emb[m, 1], s=8, alpha=0.6, c=col, label=c)
    ax.set_title(name, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(markerscale=2)
plt.suptitle("t-SNE 2D by condition (blue = Normoxia, red = Hypoxia)", y=1.03)
plt.tight_layout(); plt.show()

In [ ]:
# UMAP as a cross-cutting diagnostic canvas: 6 panels per line
def compute_umap(a, n_pcs=20):
    tmp = ad.AnnData(a.obsm["X_pca_pr"][:, :n_pcs].copy())
    tmp.obsm["X_pca"] = a.obsm["X_pca_pr"][:, :n_pcs]
    sc.pp.neighbors(tmp, n_neighbors=15, use_rep="X_pca", random_state=0)
    sc.tl.umap(tmp, random_state=0)
    a.obsm["X_umap"] = tmp.obsm["X_umap"]

for a in raw.values():
    compute_umap(a)

for name in raw:
    a = raw[name]; U = a.obsm["X_umap"]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8)); ax = axes.ravel()
    for cond, col in [("Normoxia", "#3b78c2"), ("Hypoxia", "#c23b3b")]:
        m = (a.obs.condition == cond).values
        ax[0].scatter(U[m, 0], U[m, 1], s=6, alpha=0.6, c=col, label=cond)
    ax[0].legend(markerscale=2); ax[0].set_title("condition (truth, a posteriori)")
    s = ax[1].scatter(U[:, 0], U[:, 1], s=6, alpha=0.7,
                      c=np.log1p(a.obs["total_counts"].values), cmap="viridis")
    ax[1].set_title("log(total_counts) [technical]"); plt.colorbar(s, ax=ax[1], shrink=0.7)
    s = ax[2].scatter(U[:, 0], U[:, 1], s=6, alpha=0.7, c=a.obs["G2M_score"].values, cmap="cividis")
    ax[2].set_title("G2M score [biological nuisance]"); plt.colorbar(s, ax=ax[2], shrink=0.7)
    leid = a.obs["leiden"].astype(int).values
    ax[3].scatter(U[:, 0], U[:, 1], s=6, alpha=0.7, c=leid, cmap="tab10")
    ax[3].set_title(f"Leiden clusters ({leid.max()+1})")
    s = ax[4].scatter(U[:, 0], U[:, 1], s=6, alpha=0.7, c=a.obs["nmf_hyp"].values, cmap="magma")
    ax[4].set_title("NMF hypoxia program"); plt.colorbar(s, ax=ax[4], shrink=0.7)
    noise = a.obs["dbscan_noise"].values
    ax[5].scatter(U[~noise, 0], U[~noise, 1], s=6, alpha=0.4, c="#bbbbbb")
    ax[5].scatter(U[noise, 0], U[noise, 1], s=22, alpha=0.9, c="#d62728", label="outlier")
    ax[5].legend(markerscale=1.2); ax[5].set_title("DBSCAN outliers (-1)")
    for a_ in ax: a_.set_xticks([]); a_.set_yticks([])
    fig.suptitle(f"{name} — UMAP as a cross-cutting diagnostic canvas", y=1.02, fontsize=12)
    plt.tight_layout(); plt.show()

### Cross-line projection

The next analysis asks whether the low-dimensional structure learned in one cell line can be meaningfully projected onto the other. This is not a supervised transfer experiment: no classifier is trained here. Instead, PCA is fitted on one cell line using the common genes and then the other cell line is projected into the same PCA space. If hypoxia remains organized along a similar direction, this supports the presence of a shared transcriptional response across lines.

In [ ]:
# Cross-line PCA projection using only common genes between HCC1806 and MCF7.


def get_lognorm_dataframe(a):
    return pd.DataFrame(np.asarray(a.layers["lognorm"]), index=a.obs_names, columns=a.var_names)


def crossline_projection(source_name, target_name, n_components=2):
    source = raw[source_name]
    target = raw[target_name]
    source_df = get_lognorm_dataframe(source)
    target_df = get_lognorm_dataframe(target)
    common = source_df.columns.intersection(target_df.columns)

    scaler = StandardScaler().fit(source_df[common])
    pca = PCA(n_components=n_components, random_state=UNSUPERVISED_RANDOM_STATE).fit(
        scaler.transform(source_df[common])
    )

    source_proj = pca.transform(scaler.transform(source_df[common]))
    target_proj = pca.transform(scaler.transform(target_df[common]))

    projected = pd.concat([
        pd.DataFrame(source_proj, columns=["PC1", "PC2"], index=source.obs_names).assign(
            fitted_on=source_name,
            projected_dataset=source_name,
            condition=source.obs["condition"].astype(str).values,
        ),
        pd.DataFrame(target_proj, columns=["PC1", "PC2"], index=target.obs_names).assign(
            fitted_on=source_name,
            projected_dataset=target_name,
            condition=target.obs["condition"].astype(str).values,
        ),
    ])
    return projected, pca.explained_variance_ratio_, len(common)

crossline_projection_results = {}
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (source_name, target_name) in zip(axes, [("HCC1806", "MCF7"), ("MCF7", "HCC1806")]):
    projected, evr, n_common = crossline_projection(source_name, target_name)
    crossline_projection_results[(source_name, target_name)] = projected
    sns.scatterplot(
        data=projected,
        x="PC1",
        y="PC2",
        hue="condition",
        style="projected_dataset",
        alpha=0.65,
        s=35,
        ax=ax,
    )
    ax.set_title(f"PCA fitted on {source_name}, projected on {target_name}\ncommon genes = {n_common}; PC1+PC2 = {evr.sum():.2%}")
    ax.axhline(0, linewidth=0.5, color="black", alpha=0.3)
    ax.axvline(0, linewidth=0.5, color="black", alpha=0.3)

plt.tight_layout()
plt.show()


### Interpretation of the cross-line projection

This plot should be read cautiously because cell-line identity is a strong biological and technical source of variation. A good result is not necessarily perfect overlap between HCC1806 and MCF7. The important question is whether hypoxic and normoxic cells remain separated in a comparable direction after projection.

**To be completed after running the notebook:** describe whether the projected cell line preserves the hypoxia/normoxia direction learned from the source cell line, and whether the transfer looks more convincing from HCC1806 to MCF7 or from MCF7 to HCC1806.

In both lines the **NMF program** panel mirrors the **condition** panel — visual confirmation that the unsupervised hypoxia axis matches the labels — and the **depth** panel shows no competing gradient. The Leiden sub-structure is real but finer than the binary split, and the handful of DBSCAN outliers sit at the margins.

> **Takeaway —** on the same embedding, the NMF/condition panels coincide in both lines while depth shows no gradient — the recovered structure is hypoxia, not a technical artifact.

## From unsupervised to predictive: the payoff

Does the label-free work help to **predict** the condition, and are the engineered biological features worth their cost? We test rigorously with the syllabus classifiers (logistic regression, linear SVM, Random Forest).

**Anti-leakage protocol.** Stratified train/test split (70/30). *All* preprocessing — normalization, scaling, PCA, KMeans, the Buffa z-scoring parameters — is fit **on the train only** and applied to the test. We use `log1p` (per-cell, trivially inductive) and KMeans (`.predict`), chosen because they apply cleanly to unseen cells. (Test sets are small — ~70 and ~109 cells — so we read AUC qualitatively, not to the third decimal.)

**Ablation.** Nested label-free feature sets: A = PCA only (15 comp.) · B = PCA + Buffa score · C = PCA + KMeans cluster · D = all. Metric: **AUC** (threshold-independent, robust to imbalance).

In [ ]:
CLFS = {"LogReg":  LogisticRegression(max_iter=1000, class_weight="balanced"),
        "SVM-lin": SVC(kernel="linear", probability=True, class_weight="balanced", random_state=0),
        "RF":      RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=0)}

def lognorm_block(counts):
    t = ad.AnnData(np.asarray(counts).astype(np.float32))
    sc.pp.normalize_total(t, target_sum=1e4); sc.pp.log1p(t)
    return np.asarray(t.X)

def fit_eval(Ftr, Fte, ytr, yte):
    out = {}
    for cn, clf in CLFS.items():
        s = StandardScaler().fit(Ftr); clf.fit(s.transform(Ftr), ytr)
        out[cn] = roc_auc_score(yte, clf.predict_proba(s.transform(Fte))[:, 1])
    return out

bridge_rows, ablation_rows = [], []
for name, a in raw.items():
    counts = np.asarray(a.layers["counts"]); y = (a.obs.condition == "Hypoxia").astype(int).values
    bidx = [i for i, g in enumerate(a.var_names) if g in BUFFA_SET]
    Xtr, Xte, ytr, yte = train_test_split(counts, y, test_size=0.3, stratify=y, random_state=0)
    Ltr, Lte = lognorm_block(Xtr), lognorm_block(Xte)
    scl = StandardScaler().fit(Ltr)
    pca = PCA(15, random_state=0).fit(scl.transform(Ltr))
    Ptr, Pte = pca.transform(scl.transform(Ltr)), pca.transform(scl.transform(Lte))
    mu, sd = Ltr[:, bidx].mean(0), Ltr[:, bidx].std(0) + 1e-8
    btr = ((Ltr[:, bidx]-mu)/sd).mean(1); bte = ((Lte[:, bidx]-mu)/sd).mean(1)
    km = KMeans(2, n_init=10, random_state=0).fit(Ptr)
    ktr, kte = km.predict(Ptr).reshape(-1, 1), km.predict(Pte).reshape(-1, 1)
    feats = {"A: PCA": (Ptr, Pte), "B: PCA+Buffa": (np.c_[Ptr, btr], np.c_[Pte, bte]),
             "C: PCA+KMeans": (np.c_[Ptr, ktr], np.c_[Pte, kte]),
             "D: PCA+Buffa+KM": (np.c_[Ptr, btr, ktr], np.c_[Pte, bte, kte])}
    for fn, (Ftr, Fte) in feats.items():
        for cn, auc in fit_eval(Ftr, Fte, ytr, yte).items():
            bridge_rows.append({"dataset": name, "features": fn, "clf": cn, "AUC": auc})
    for fn, F in {"Buffa score (1 feat.)": (btr.reshape(-1,1), bte.reshape(-1,1)),
                  "PCA-15 (unsup.)": (Ptr, Pte)}.items():
        ablation_rows.append({"dataset": name, "features": fn,
                              "AUC": fit_eval(F[0], F[1], ytr, yte)["LogReg"]})

bridge = pd.DataFrame(bridge_rows)
mean_auc = bridge.groupby(["dataset", "features"]).AUC.mean().unstack().round(3)
mean_auc["Buffa lift (B-A)"] = (mean_auc["B: PCA+Buffa"] - mean_auc["A: PCA"]).round(3)
mean_auc

In [ ]:
abl = pd.DataFrame(ablation_rows).pivot_table(index="dataset", columns="features", values="AUC").loc[list(raw)]
ax = abl[["Buffa score (1 feat.)", "PCA-15 (unsup.)"]].plot.bar(figsize=(7, 4), color=["#c23b3b", "#3b78c2"])
ax.set_ylabel("test AUC"); ax.set_ylim(0.5, 1.02)
ax.set_title("A single biological feature vs 15 unsupervised PCs")
ax.axhline(0.5, c="k", lw=0.6, ls=":"); plt.xticks(rotation=0)
plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

**Interpretation.**

1. **Both lines are essentially perfectly classifiable** from 15 unsupervised PCs (AUC ≈ 1.0). On full-length data the hypoxic signal is strong and linearly separable — no surprise after the clustering, where the condition was already a dominant axis.

2. **Engineered features add ≈ 0 to AUC** (the *lift* column). Once a clean PCA is in place, the Buffa score and the cluster id do not move the predictive metric — their value is **interpretability** (Sec. 8), not squeezing an already-saturated AUC. Chasing the third decimal with heavier ensembles would risk overfitting more than it gains.

3. **The single Buffa feature alone is a strong baseline** (AUC ≈ 0.93 HCC1806, ≈ 0.80 MCF7): one biologically-motivated number rivals a 15-dimensional unsupervised representation — a compact, interpretable predictor.

> **Takeaway —** clean unsupervised PCA already classifies hypoxia near-perfectly in both lines; engineered features add nothing to AUC (their worth is interpretability), and a single Buffa score is a strong standalone predictor.

## Robustness: cross-line comparative table

The analysis was run on two cell lines with several methods to test **robustness**: if the Hypoxia/Normoxia separation emerges consistently across independent approaches, the conclusion is solid. We summarize one metric per approach (clustering, factorization, biological score, prediction) plus a noise metric.

In [ ]:
robust = pd.DataFrame({
    "ARI clustering (best)":   clust.groupby("dataset").ARI.max(),
    "NMF |corr| (best comp.)": pd.Series({k: v["best_|corr|"] for k, v in nmf_res.items()}),
    "Buffa score |r|":         score_cmp["|r| score_genes"],
    "supervised AUC (PCA)":    mean_auc["A: PCA"],
    "DBSCAN % outliers":       pd.Series({k: v["% noise"] for k, v in dbscan_res.items()}),
}).loc[list(raw)]
robust.round(3)

In [ ]:
sig = robust.drop(columns="DBSCAN % outliers")
fig, ax = plt.subplots(figsize=(8, 2.8))
sns.heatmap(sig, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1,
            cbar_kws={"label": "signal strength"}, ax=ax)
ax.set_title("How recoverable hypoxia is, by cell line and by approach")
plt.tight_layout(); plt.show()

Both rows are green across every approach — the hypoxic signature is robustly recoverable in **both** breast-cancer lines, by independent methods. The consistent gradient is that **MCF7 is the stronger/cleaner case** (clustering and Buffa-score columns) while **HCC1806 carries the more canonical HIF-target program** (Sec. 8 NMF) — the same biology, read slightly differently by the two lines. Convergence across four method families is what makes the conclusion robust rather than a single-pipeline artifact.

> **Takeaway —** every method recovers hypoxia in both lines (uniformly green); MCF7 separates more sharply, HCC1806 shows the more canonical hypoxia program — agreement across methods is the evidence.

## Unsupervised learning conclusions

The unsupervised analysis shows that hypoxia is not only predictable after training a classifier: it is already visible in the structure of the single-cell expression profiles. PCA, clustering, the Buffa score and NMF all recover a hypoxia-related signal, although the strength and biological flavour of the signal differ between the two cell lines.

MCF7 generally shows a cleaner separation between normoxic and hypoxic cells, suggesting that oxygen condition is a particularly strong axis of variation in this line. HCC1806 is less clean from a clustering point of view, but its NMF programs tend to recover more canonical hypoxia genes, including genes connected to HIF response, glycolysis and stress adaptation. This is biologically important: the two lines do not simply repeat the same transcriptional response, but they appear to share a hypoxic core while emphasizing different line-specific programs.

The main limitation is that these analyses are still performed separately by cell line. A fully integrated cross-line analysis would require explicit batch or cell-line correction, otherwise the strongest difference would likely be HCC1806 versus MCF7 rather than hypoxia versus normoxia.

## References

- **Buffa FM, Harris AL, West CM, Miller CJ.** *Large meta-analysis of multiple cancers reveals a common, compact and highly prognostic hypoxia metagene.* **British Journal of Cancer**, 102(2):428–435, 2010. doi: 10.1038/sj.bjc.6605450. (Source of the 50-gene hypoxia signature used throughout this notebook.)
- **Tirosh I, et al.** *Dissecting the multicellular ecosystem of metastatic melanoma by single-cell RNA-seq.* **Science**, 352(6282):189–196, 2016. (Source of the cell-cycle gene sets used for the confounder diagnostic in Section 4.)
- **Lause J, Berens P, Kobak D.** *Analytic Pearson residuals for normalization of single-cell RNA-seq UMI data.* **Genome Biology**, 22:258, 2021. (Methodological basis for the Pearson residuals normalization compared in Section 3.)
- **Traag VA, Waltman L, van Eck NJ.** *From Louvain to Leiden: guaranteeing well-connected communities.* **Scientific Reports**, 9:5233, 2019. (Leiden community detection, used as a graph-based contrast to KMeans/GMM in Section 6.)
- **McInnes L, Healy J, Melville J.** *UMAP: Uniform Manifold Approximation and Projection for dimension reduction.* arXiv:1802.03426, 2018. (Non-linear visualization in Section 10.)

---

## Supervised Learning Analysis

The supervised section uses the datasets loaded at the beginning of the notebook. The first supervised cells therefore start from train/test split construction and preprocessing, instead of loading the same SmartSeq, QCd and DropSeq files again.

## Supervised learning goal

The supervised-learning part tests whether single-cell gene-expression profiles can be used to classify cells as normoxic or hypoxic. The analysis compares three dataset versions: the professor-provided SmartSeq normalised data, our QCd SmartSeq data, and DropSeq normalised data. This makes the evaluation both predictive and methodological, because model performance is interpreted together with preprocessing and sequencing technology.

## SmartSeq_normalised data

The first dataset family is the filtered and normalised SmartSeq data provided in the course material. These matrices are already preprocessed: low-quality information has been filtered out, expression values have been normalised, and the matrices contain a reduced set of genes. In the original files, genes are stored in rows and cells are stored in columns, while `scikit-learn` expects samples in rows and features in columns. For this reason, each matrix is transposed after loading.

The labelled training files contain the oxygen condition in the cell/sample names. The function `extract_condition_from_cell_name` converts these names into the binary target used throughout the notebook: `Normoxia = 0` and `Hypoxia = 1`. The anonymous test files are loaded separately because they do not contain labels and must not be used for model selection.

### Internal split for SmartSeq_normalised

Although the SmartSeq files already distinguish between labelled training data and anonymous test data, the labelled training data are split again into an **internal training set** and an **internal test set**. The internal training set is used for cross-validation and hyperparameter tuning. The internal test set is kept aside and used only to evaluate the model after it has been selected.

This split is important because it gives a realistic estimate of generalisation before using the final anonymous test set. The split is stratified by condition, so that the proportion of normoxic and hypoxic cells remains similar in the internal train and test partitions. A combined dataset is also created by keeping only genes shared by HCC1806 and MCF7, allowing us to test whether a single model can learn a common hypoxia signal across both cell lines.

In [ ]:
# Create SmartSeq_normalised internal splits and combined dataset.
TEST_SIZE = 0.20
N_SPLITS = 5

stratified_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)


def make_train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE):
    """Brief helper used in the supervised learning workflow."""
    return train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=random_state,
    )


splits = {}

splits["HCC1806"] = {}
(
    splits["HCC1806"]["X_train"],
    splits["HCC1806"]["X_test"],
    splits["HCC1806"]["y_train"],
    splits["HCC1806"]["y_test"],
) = make_train_test_split(X_hcc, y_hcc)

splits["MCF7"] = {}
(
    splits["MCF7"]["X_train"],
    splits["MCF7"]["X_test"],
    splits["MCF7"]["y_train"],
    splits["MCF7"]["y_test"],
) = make_train_test_split(X_mcf, y_mcf)

# Combined dataset: it uses only common genes across cell lines
common_genes = X_hcc.columns.intersection(X_mcf.columns)
X_combined = pd.concat([X_hcc[common_genes], X_mcf[common_genes]], axis=0)
y_combined = pd.concat([y_hcc, y_mcf], axis=0)
cell_line_combined = pd.Series(
    ["HCC1806"] * len(X_hcc) + ["MCF7"] * len(X_mcf),
    index=X_combined.index,
    name="cell_line",
)

splits["Combined"] = {}
(
    splits["Combined"]["X_train"],
    splits["Combined"]["X_test"],
    splits["Combined"]["y_train"],
    splits["Combined"]["y_test"],
) = make_train_test_split(X_combined, y_combined)

for dataset_name, data in splits.items():
    print(f"\n{dataset_name}")
    print("Train:", data["X_train"].shape, data["y_train"].value_counts().to_dict())
    print("Test: ", data["X_test"].shape, data["y_test"].value_counts().to_dict())

print(f"\nCommon genes between HCC1806 and MCF7: {len(common_genes)}")

## SmartSeq_QCd data

The second dataset family is built from the QCd SmartSeq count matrices. These files are organised directly as **cells by genes**, so they do not need to be transposed. Unlike the SmartSeq_normalised matrices, these data are intended to pass through our own preprocessing pipeline during model training. This makes it possible to compare the professor-provided preprocessing with a pipeline designed and controlled by us.

At this stage, the notebook only loads the QCd matrices, extracts the labels from the cell identifiers and creates metadata tables. The actual preprocessing steps are not applied here. They are deliberately inserted later inside the model pipeline, so that gene filtering, normalisation, feature engineering, scaling and feature selection are fitted only on training folds during cross-validation.

### Internal split for SmartSeq_QCd

The same internal splitting strategy is used for the QCd data. This is necessary to make the comparison with SmartSeq_normalised fair: both dataset versions are evaluated through an internal train-test split, cross-validation on the training part, and final hold-out evaluation on the internal test part.

The combined QCd dataset is created using only genes shared between the two QCd cell-line matrices. This is especially important for cross-cell-line evaluation, because a model trained on one cell line can only be tested on another if both matrices have the same feature space.

In [ ]:
# Create SmartSeq_QCd internal splits and combined dataset.

splits["HCC1806_QCd"] = {}
(
    splits["HCC1806_QCd"]["X_train"],
    splits["HCC1806_QCd"]["X_test"],
    splits["HCC1806_QCd"]["y_train"],
    splits["HCC1806_QCd"]["y_test"],
) = make_train_test_split(X_hcc_qcd, y_hcc_qcd)

splits["MCF7_QCd"] = {}
(
    splits["MCF7_QCd"]["X_train"],
    splits["MCF7_QCd"]["X_test"],
    splits["MCF7_QCd"]["y_train"],
    splits["MCF7_QCd"]["y_test"],
) = make_train_test_split(X_mcf_qcd, y_mcf_qcd)

# Combined QCd dataset: it uses only common genes across cell lines
common_genes_qcd = X_hcc_qcd.columns.intersection(X_mcf_qcd.columns)

X_combined_qcd = pd.concat(
    [X_hcc_qcd[common_genes_qcd], X_mcf_qcd[common_genes_qcd]],
    axis=0,
)

y_combined_qcd = pd.concat([y_hcc_qcd, y_mcf_qcd], axis=0)

cell_line_combined_qcd = pd.Series(
    ["HCC1806"] * len(X_hcc_qcd) + ["MCF7"] * len(X_mcf_qcd),
    index=X_combined_qcd.index,
    name="cell_line",
)

splits["Combined_QCd"] = {}
(
    splits["Combined_QCd"]["X_train"],
    splits["Combined_QCd"]["X_test"],
    splits["Combined_QCd"]["y_train"],
    splits["Combined_QCd"]["y_test"],
) = make_train_test_split(X_combined_qcd, y_combined_qcd)

for dataset_name in ["HCC1806_QCd", "MCF7_QCd", "Combined_QCd"]:
    data = splits[dataset_name]
    print(f"\n{dataset_name}")
    print("Train:", data["X_train"].shape, data["y_train"].value_counts().to_dict())
    print("Test: ", data["X_test"].shape, data["y_test"].value_counts().to_dict())

print(f"\nCommon genes between HCC1806_QCd and MCF7_QCd: {len(common_genes_qcd)}")

## DropSeq_normalised data

The third dataset family is the filtered and normalised DropSeq data. These files follow the same general structure as the SmartSeq_normalised matrices: genes are stored in rows, cells are stored in columns, and labelled train files are provided separately from anonymous test files. DropSeq contains many more cells than the SmartSeq data, so it is the most relevant dataset for final model training and prediction.

DropSeq is kept separate from SmartSeq because the two technologies measure single-cell expression with different experimental protocols and different technical biases. Applying the same supervised learning framework to DropSeq allows us to test whether the conclusions obtained on SmartSeq remain valid when the number of cells increases and the sequencing technology changes.

### Internal split for DropSeq_normalised

The labelled DropSeq train files are internally split for the same reason as the SmartSeq data: the anonymous test files have no labels, so they cannot be used to evaluate model quality during development. The internal test split gives a labelled hold-out set for performance estimation, while the anonymous DropSeq test files are reserved for the final predictions.

The combined DropSeq dataset is created from common genes between HCC1806 and MCF7. This allows the same five-scenario evaluation used for SmartSeq: within-cell-line prediction, combined prediction, and cross-cell-line transfer.

In [ ]:
# Internal train-test splits for DropSeq labelled data

splits["DropSeq_HCC1806"] = {}
(
    splits["DropSeq_HCC1806"]["X_train"],
    splits["DropSeq_HCC1806"]["X_test"],
    splits["DropSeq_HCC1806"]["y_train"],
    splits["DropSeq_HCC1806"]["y_test"],
) = make_train_test_split(X_hcc_dropseq, y_hcc_dropseq)

splits["DropSeq_MCF7"] = {}
(
    splits["DropSeq_MCF7"]["X_train"],
    splits["DropSeq_MCF7"]["X_test"],
    splits["DropSeq_MCF7"]["y_train"],
    splits["DropSeq_MCF7"]["y_test"],
) = make_train_test_split(X_mcf_dropseq, y_mcf_dropseq)

common_genes_dropseq = X_hcc_dropseq.columns.intersection(X_mcf_dropseq.columns)

X_dropseq_combined = pd.concat(
    [
        X_hcc_dropseq[common_genes_dropseq],
        X_mcf_dropseq[common_genes_dropseq],
    ],
    axis=0,
)

y_dropseq_combined = pd.concat([y_hcc_dropseq, y_mcf_dropseq], axis=0)

cell_line_dropseq_combined = pd.Series(
    ["HCC1806"] * len(X_hcc_dropseq) + ["MCF7"] * len(X_mcf_dropseq),
    index=X_dropseq_combined.index,
    name="cell_line",
)

splits["DropSeq_Combined"] = {}
(
    splits["DropSeq_Combined"]["X_train"],
    splits["DropSeq_Combined"]["X_test"],
    splits["DropSeq_Combined"]["y_train"],
    splits["DropSeq_Combined"]["y_test"],
) = make_train_test_split(X_dropseq_combined, y_dropseq_combined)

for dataset_name in ["DropSeq_HCC1806", "DropSeq_MCF7", "DropSeq_Combined"]:
    data = splits[dataset_name]
    print(f"\n{dataset_name}")
    print("Train:", data["X_train"].shape, data["y_train"].value_counts().to_dict())
    print("Test: ", data["X_test"].shape, data["y_test"].value_counts().to_dict())

print(f"\nCommon genes between DropSeq HCC1806 and DropSeq MCF7: {len(common_genes_dropseq)}")

## Evaluation metrics and helper functions

The supervised task is binary classification, but a single metric is not enough to describe model performance. The notebook therefore computes several complementary metrics:

- **Accuracy**: the fraction of correctly classified cells. It is intuitive, but it can be misleading if the two classes are imbalanced.
- **Balanced accuracy**: the average recall of the two classes. This is the main model-selection metric because it gives equal importance to Normoxia and Hypoxia.
- **Macro F1-score**: the average of the F1-score computed separately for the two classes. It balances precision and recall and is useful when errors in both classes matter.
- **ROC-AUC**: measures how well the model separates the two classes across all possible probability thresholds. It is useful when models provide probabilities or decision scores.
- **Precision for Hypoxia**: among the cells predicted as hypoxic, the fraction that is truly hypoxic.
- **Recall for Hypoxia**: among truly hypoxic cells, the fraction correctly identified by the model. This is biologically relevant because missing hypoxic cells may hide an important stress-response population.

The helper functions below make the rest of the notebook shorter and more consistent. They evaluate either one model on an internal test set or one model through cross-validation, always returning results in a tabular format.

In [ ]:
# Define evaluation metrics and helper functions.
SCORING = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1_macro": "f1_macro",
    "roc_auc": "roc_auc",
}


def get_prediction_scores(fitted_model, X):
    """Return a probability score for the positive class Hypoxia = 1"""
    if hasattr(fitted_model, "predict_proba"):
        return fitted_model.predict_proba(X)[:, 1]
    if hasattr(fitted_model, "decision_function"):
        return fitted_model.decision_function(X)
    return None


def evaluation_model(model, X_train, y_train, X_test, y_test, model_name="model", dataset_name="dataset"):
    """Fit one model on the train set and evaluate it on one hold-out test set."""
    fitted_model = clone(model)
    fitted_model.fit(X_train, y_train)

    y_pred = fitted_model.predict(X_test)
    y_score = get_prediction_scores(fitted_model, X_test)

    result = {
        "dataset": dataset_name,
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "f1_macro": f1_score(y_test, y_pred, average="macro"),
        "precision_hypoxia": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "recall_hypoxia": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "roc_auc": np.nan,
    }

    if y_score is not None and len(np.unique(y_test)) == 2:
        result["roc_auc"] = roc_auc_score(y_test, y_score)

    return result, fitted_model, y_pred, y_score


def evaluation_model_cv(model, X, y, cv=stratified_cv, dataset_name="dataset", model_name="model", scoring=SCORING):
    """Evaluate one model by cross-validation and return."""
    cv_scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=True,
    )

    result = {
        "dataset": dataset_name,
        "model": model_name,
        "cv_accuracy_mean": np.mean(cv_scores["test_accuracy"]),
        "cv_accuracy_std": np.std(cv_scores["test_accuracy"]),
        "cv_balanced_accuracy_mean": np.mean(cv_scores["test_balanced_accuracy"]),
        "cv_balanced_accuracy_std": np.std(cv_scores["test_balanced_accuracy"]),
        "cv_f1_macro_mean": np.mean(cv_scores["test_f1_macro"]),
        "cv_f1_macro_std": np.std(cv_scores["test_f1_macro"]),
        "cv_roc_auc_mean": np.mean(cv_scores["test_roc_auc"]),
        "cv_roc_auc_std": np.std(cv_scores["test_roc_auc"]),
        "train_balanced_accuracy_mean": np.mean(cv_scores["train_balanced_accuracy"]),
    }

    return result, cv_scores

## Models considered in the analisys

The notebook compares a broad set of supervised models. This is useful because gene-expression classification is a high-dimensional problem: there are many more genes than cells, and different algorithms may react differently to this setting.

- **DummyClassifier** is a baseline that ignores gene expression. It verifies that real models improve over a trivial prediction strategy.
- **Logistic Regression** is a linear model. With regularisation, it is often strong in high-dimensional biological data and remains relatively interpretable.
- **k-Nearest Neighbours** predicts a cell by looking at nearby cells in expression space. It is simple, but can suffer in high-dimensional data because distances become less informative.
- **Random Forest** combines many decision trees trained on bootstrap samples and random subsets of features. It can model non-linear effects and is robust, although it may struggle when thousands of genes are mostly noisy.
- **Gradient Boosting** and **AdaBoost** build trees sequentially, where each new tree tries to correct previous mistakes.
- **Linear SVM** searches for a separating hyperplane with maximum margin. It is well suited for high-dimensional data.

Three models deserve a slightly more detailed explanation:

**SVM with RBF kernel.** A standard SVM searches for a decision boundary with large margin. With the RBF kernel, the similarity between two cells $x_i$ and $x_j$ is

$$K(x_i, x_j) = \exp(-\gamma \lVert x_i - x_j 
Vert^2).$$

This allows a non-linear boundary in the original gene-expression space. The main hyperparameters tuned are $C$, controlling the penalty for misclassified cells, $\gamma$, controlling the locality of the RBF kernel, and the number of selected genes or PCA components used before the classifier.

**XGBoost.** XGBoost is a gradient-boosted tree ensemble. It builds an additive model

$$\hat{y}_i = \sum_{m=1}^{M} f_m(x_i),$$

where each $f_m$ is a regression/classification tree. At each iteration, a new tree is fitted to reduce the objective function, which combines prediction loss and regularisation. The main tuned hyperparameters are the number of trees, maximum tree depth, learning rate, subsampling fractions, column sampling, and regularisation terms such as `reg_alpha` and `reg_lambda`.

**MLPClassifier.** A multi-layer perceptron is a feed-forward neural network. For one hidden layer, the transformation can be written as

$$h = \sigma(W_1 x + b_1), \qquad \hat{y} = \sigma(W_2 h + b_2),$$

where $\sigma$ is a non-linear activation function. In this project, the MLP is treated cautiously because the number of genes is large compared with the number of cells. The main tuned hyperparameters are hidden-layer size, activation function, L2 regularisation strength (`alpha`) and learning rate. PCA is sometimes used before the MLP to reduce the number of input dimensions.

### Model dictionary

The first model dictionary is used for a fast initial benchmark on SmartSeq_normalised data. This benchmark is not the final model-selection step; its purpose is to check whether the classification signal is strong, whether the code runs correctly, and which model families appear promising before the more expensive tuning phase.

In [ ]:
# Build the first model dictionary for quick benchmarking.
# If you want to evaluate models on all genes, set QUICK_K = "all".
QUICK_K = 1000
QUICK_PCA_COMPONENTS = 30

models = {
    "DummyClassifier": Pipeline([
        ("model", DummyClassifier(strategy="most_frequent")),
    ]),

    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=safe_f_classif, k=QUICK_K)),
        ("model", LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]),

    "kNN": Pipeline([
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=safe_f_classif, k=QUICK_K)),
        ("model", KNeighborsClassifier()),
    ]),

    "RandomForest": Pipeline([
        ("selector", SelectKBest(score_func=safe_f_classif, k=QUICK_K)),
        ("model", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),

    "GradientBoosting": Pipeline([
        ("selector", SelectKBest(score_func=safe_f_classif, k=QUICK_K)),
        ("model", GradientBoostingClassifier(random_state=RANDOM_STATE)),
    ]),

    "AdaBoost": Pipeline([
        ("selector", SelectKBest(score_func=safe_f_classif, k=QUICK_K)),
        ("model", AdaBoostClassifier(
            n_estimators=100,
            learning_rate=0.5,
            random_state=RANDOM_STATE,
        )),
    ]),

    "SVM_linear": Pipeline([
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=safe_f_classif, k=100)),
        ("model", SVC(
            kernel="linear",
            probability=True,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            C = 0.01,
        )),
    ]),

    "SVM_rbf": Pipeline([
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=safe_f_classif, k=QUICK_K)),
        ("pca", PCA(n_components=QUICK_PCA_COMPONENTS, random_state=RANDOM_STATE)),
        ("model", SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]),

    "MLP": Pipeline([
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=safe_f_classif, k=QUICK_K)),
        ("pca", PCA(n_components=QUICK_PCA_COMPONENTS, random_state=RANDOM_STATE)),
        ("model", MLPClassifier(
            hidden_layer_sizes=(64,),
            activation="relu",
            alpha=0.01,
            learning_rate_init=0.001,
            max_iter=1000,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=20,
            random_state=RANDOM_STATE,
        )),
    ]),

    "XGBoost": Pipeline([
        ("selector", SelectKBest(score_func=safe_f_classif, k=QUICK_K)),
        ("model", XGBClassifier(
            n_estimators=200,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            tree_method="hist",
        )),
    ])
    }

print("Models in quick benchmark:")
print(list(models.keys()))

## Quick benchmark on SmartSeq_normalised

The quick benchmark evaluates the initial model dictionary on the SmartSeq_normalised datasets. For each model and dataset, the notebook computes cross-validation scores on the internal training split and a hold-out score on the internal test split. This gives a first impression of separability before hyperparameter tuning.

The benchmark is intentionally lightweight. It uses a limited number of selected genes and PCA components for some models, so the results should not be interpreted as final. However, if several models already perform very well here, it suggests that the normoxia/hypoxia signal is strong in the professor-processed SmartSeq data.

In [ ]:
# Run the first quick benchmark on SmartSeq_normalised.
RUN_EVALUATION = True
DATASETS_EVAL = ["HCC1806", "MCF7", "Combined",]

quick_cv_results = []
quick_holdout_results = []
quick_fitted_models = {}
quick_predictions = {}
quick_scores = {}

if RUN_EVALUATION:
    for dataset_name in DATASETS_EVAL:
        X_train = splits[dataset_name]["X_train"]
        y_train = splits[dataset_name]["y_train"]
        X_test = splits[dataset_name]["X_test"]
        y_test = splits[dataset_name]["y_test"]

        quick_fitted_models[dataset_name] = {}
        quick_predictions[dataset_name] = {}
        quick_scores[dataset_name] = {}

        for model_name, model in models.items():
            print(f"\nQuick evaluation | {dataset_name} | {model_name}")
            cv_row, _ = evaluation_model_cv(
                model,
                X_train,
                y_train,
                dataset_name=dataset_name,
                model_name=model_name,
            )
            quick_cv_results.append(cv_row)

            holdout_row, fitted_model, y_pred, y_score = evaluation_model(
                model,
                X_train,
                y_train,
                X_test,
                y_test,
                model_name=model_name,
                dataset_name=dataset_name,
            )
            quick_holdout_results.append(holdout_row)
            quick_fitted_models[dataset_name][model_name] = fitted_model
            quick_predictions[dataset_name][model_name] = y_pred
            quick_scores[dataset_name][model_name] = y_score

    quick_cv_results = pd.DataFrame(quick_cv_results)
    quick_holdout_results = pd.DataFrame(quick_holdout_results)


    quick_cv_results = quick_cv_results.sort_values(
        ["dataset", "cv_balanced_accuracy_mean"],
        ascending=[True, False],
    ).reset_index(drop=True)

    
    quick_holdout_results = quick_holdout_results.sort_values(
        ["dataset", "balanced_accuracy"],
        ascending=[True, False],
    ).reset_index(drop=True)

    print("Cross-validation results:")
    display(quick_cv_results)

    print("Hold-out results:")
    display(quick_holdout_results)
    
else:
    quick_cv_results = pd.DataFrame()
    quick_holdout_results = pd.DataFrame()
    print("Quick evaluation was not run.")

The probability parameter will no longer exist in future versions, and a new way of storing the probabilities assigned to each prediction will be implemented. This is why so many warnings are being raised.

## Cross-cell-line evaluation

Cross-cell-line evaluation asks a harder biological question than ordinary train-test evaluation. Instead of training and testing within the same cell line, the model is trained on one biological background and tested on another:

- **HCC1806 → MCF7**: the model learns hypoxia from HCC1806 and predicts MCF7 cells;
- **MCF7 → HCC1806**: the model learns hypoxia from MCF7 and predicts HCC1806 cells.

If performance drops strongly in these settings, it suggests that the two cell lines respond to hypoxia with partly different transcriptional programs, or that the model has learned cell-line-specific patterns rather than a shared hypoxia signature. If performance remains high, it supports the presence of a robust hypoxia signal that generalises across cancer cell lines. In the preliminary SmartSeq run, tree-based models tend to transfer better than several distance- or margin-based models, suggesting that some non-linear or threshold-like expression patterns may be more stable across cell lines.

In [ ]:
# Run preliminary cross-cell-line evaluation on SmartSeq_normalised.
RUN_CROSS_CELL_LINE = True

cross_cell_line_results = []

if RUN_CROSS_CELL_LINE:
    common = X_hcc.columns.intersection(X_mcf.columns)
    Xh = X_hcc[common]
    Xm = X_mcf[common]

    cross_cell_line_splits = [
        ("HCC1806_to_MCF7", Xh, y_hcc, Xm, y_mcf),
        ("MCF7_to_HCC1806", Xm, y_mcf, Xh, y_hcc),
    ]

    for direction, X_train, y_train, X_test, y_test in cross_cell_line_splits:
        for model_name, model in models.items():
            print(f"Cross-cell-line | {direction} | {model_name}")
            result, _, _, _ = evaluation_model(
                model,
                X_train,
                y_train,
                X_test,
                y_test,
                model_name=model_name,
                dataset_name=direction,
            )
            cross_cell_line_results.append(result)
            

    cross_cell_line_results = pd.DataFrame(cross_cell_line_results)

    cross_cell_line_results = cross_cell_line_results.sort_values(
        ["dataset", "balanced_accuracy"],
        ascending=[True, False],
    ).reset_index(drop=True)
    display(cross_cell_line_results)
else:
    cross_cell_line_results = pd.DataFrame()
    print("Cross-cell-line benchmark was not run.")

### Preliminary conclusions from the benchmark

The first benchmark indicates that hypoxia classification is feasible in the SmartSeq_normalised data. Within-cell-line and combined evaluations are generally very strong, which means that the expression matrices contain a clear oxygen-condition signal. The cross-cell-line benchmark is more informative biologically: when performance remains high across cell lines, it suggests that part of the hypoxia response is shared; when performance decreases, it indicates that the model may be relying on cell-line-specific transcriptional differences.

## Hyperparameter tuning

The tuning section is designed to reuse the same functions across SmartSeq_normalised, SmartSeq_QCd and DropSeq_normalised. Most models are tuned with `RandomizedSearchCV`, which samples a fixed number of hyperparameter combinations from a predefined search space and evaluates each combination by cross-validation. This is more efficient than an exhaustive grid when many models and datasets are compared.

XGBoost is tuned with **Optuna**, a sequential optimisation library. Instead of sampling every trial independently, Optuna uses previous trial results to suggest more promising hyperparameter combinations. In this notebook, each Optuna trial proposes values such as tree depth, learning rate, number of trees and regularisation parameters. The candidate XGBoost pipeline is then evaluated by cross-validation using balanced accuracy as the main score. The best trial is refitted on the full internal training split.

The function `run_tuning_suite` is the main entry point: it receives a tuning dictionary, runs the appropriate search method for each model, and returns three objects: a summary table, the fitted search objects and the best estimators.

In [ ]:
# Define reusable hyperparameter tuning functions.
def tuning_randomized_search(
    model_name,
    estimator,
    param_grid,
    X_train,
    y_train,
    cv=stratified_cv,
    scoring=SCORING,
    refit="balanced_accuracy",
    n_iter=30,
    random_state=RANDOM_STATE,
    n_jobs=-1,
):
    """Tune non-XGBoost model with RandomizedSearchCV."""
    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_grid,
        n_iter=n_iter,
        scoring=scoring,
        refit=refit,
        cv=cv,
        n_jobs=n_jobs,
        random_state=random_state,
        return_train_score=True,
    )

    print(f"Tuning {model_name} with RandomizedSearchCV...")
    search.fit(X_train, y_train)
    
    best_row = pd.DataFrame(search.cv_results_).iloc[[search.best_index_]].copy()

    summary = {
        "model": model_name,
        "search_method": "RandomizedSearchCV",
        "best_params": search.best_params_,
        "best_score_refit": search.best_score_,
        "mean_test_accuracy": best_row["mean_test_accuracy"].iloc[0],
        "std_test_accuracy": best_row["std_test_accuracy"].iloc[0],
        "mean_train_accuracy": best_row["mean_train_accuracy"].iloc[0],
        "mean_test_balanced_accuracy": best_row["mean_test_balanced_accuracy"].iloc[0],
        "std_test_balanced_accuracy": best_row["std_test_balanced_accuracy"].iloc[0],
        "mean_train_balanced_accuracy": best_row["mean_train_balanced_accuracy"].iloc[0],
        "mean_test_f1_macro": best_row["mean_test_f1_macro"].iloc[0],
        "std_test_f1_macro": best_row["std_test_f1_macro"].iloc[0],
        "mean_train_f1_macro": best_row["mean_train_f1_macro"].iloc[0],
        "mean_test_roc_auc": best_row["mean_test_roc_auc"].iloc[0],
        "std_test_roc_auc": best_row["std_test_roc_auc"].iloc[0],
        "mean_train_roc_auc": best_row["mean_train_roc_auc"].iloc[0],
    }
    
    return search, summary, search.best_estimator_


def tuning_xgboost_optuna(
    model_name,
    estimator,
    X_train,
    y_train,
    cv=stratified_cv,
    scoring=SCORING,
    refit="balanced_accuracy",
    n_trials=30,
    random_state=RANDOM_STATE,
    selector_k_param="selector__k",
):
    """Tune XGBoost with Optuna"""

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    available_k = [k for k in [100, 500, 1000]]
    if len(available_k) == 0:
        available_k = ["all"]

    def objective(trial):
        """Brief helper used in the supervised learning workflow."""
        params = {
            selector_k_param: trial.suggest_categorical(selector_k_param, available_k),
            "model__n_estimators": trial.suggest_int("model__n_estimators", 80, 350, step=25),
            "model__max_depth": trial.suggest_int("model__max_depth", 2, 5),
            "model__learning_rate": trial.suggest_float("model__learning_rate", 0.01, 0.20, log=True),
            "model__subsample": trial.suggest_float("model__subsample", 0.60, 1.00),
            "model__colsample_bytree": trial.suggest_float("model__colsample_bytree", 0.60, 1.00),
            "model__min_child_weight": trial.suggest_int("model__min_child_weight", 1, 6),
            "model__gamma": trial.suggest_float("model__gamma", 0.00, 2.00),
            "model__reg_alpha": trial.suggest_float("model__reg_alpha", 0.00, 1.00),
            "model__reg_lambda": trial.suggest_float("model__reg_lambda", 0.50, 10.00, log=True),
        }

        candidate = clone(estimator)
        candidate.set_params(**params)

        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=UserWarning)
            warnings.filterwarnings("ignore", category=RuntimeWarning)
            warnings.filterwarnings("ignore", message="Features .* are constant.*")
            warnings.filterwarnings("ignore", message="invalid value encountered.*")
            with np.errstate(divide="ignore", invalid="ignore"):
                cv_scores = cross_validate(
                    candidate,
                    X_train,
                    y_train,
                    cv=cv,
                    scoring=scoring,
                    n_jobs=1,
                    return_train_score=True,
                )

        metrics = {
            "mean_test_accuracy": np.mean(cv_scores["test_accuracy"]),
            "std_test_accuracy": np.std(cv_scores["test_accuracy"]),
            "mean_train_accuracy": np.mean(cv_scores["train_accuracy"]),
            "mean_test_balanced_accuracy": np.mean(cv_scores["test_balanced_accuracy"]),
            "std_test_balanced_accuracy": np.std(cv_scores["test_balanced_accuracy"]),
            "mean_train_balanced_accuracy": np.mean(cv_scores["train_balanced_accuracy"]),
            "mean_test_f1_macro": np.mean(cv_scores["test_f1_macro"]),
            "std_test_f1_macro": np.std(cv_scores["test_f1_macro"]),
            "mean_train_f1_macro": np.mean(cv_scores["train_f1_macro"]),
            "mean_test_roc_auc": np.mean(cv_scores["test_roc_auc"]),
            "std_test_roc_auc": np.std(cv_scores["test_roc_auc"]),
            "mean_train_roc_auc": np.mean(cv_scores["train_roc_auc"]),
        }
        trial.set_user_attr("metrics", metrics)

        return metrics[f"mean_test_{refit}"]

    print(f"Tuning {model_name} with Optuna...")
    sampler = optuna.samplers.TPESampler(seed=random_state)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_params = study.best_trial.params
    best_estimator = clone(estimator)
    best_estimator.set_params(**best_params)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning)
        warnings.filterwarnings("ignore", category=RuntimeWarning)
        warnings.filterwarnings("ignore", message="Features .* are constant.*")
        warnings.filterwarnings("ignore", message="invalid value encountered.*")
        with np.errstate(divide="ignore", invalid="ignore"):
            best_estimator.fit(X_train, y_train)

    summary = {
        "model": model_name,
        "search_method": "Optuna",
        "best_params": best_params,
        "best_score_refit": study.best_value,
    }
    summary.update(study.best_trial.user_attrs.get("metrics", {}))

    return study, summary, best_estimator


def tuning(
    model_name,
    estimator,
    param_grid,
    X_train,
    y_train,
    cv=stratified_cv,
    scoring=SCORING,
    refit="balanced_accuracy",
    search_type="random",
    n_iter=30,
    n_trials=30,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    selector_k_param="selector__k",
):

    """Brief helper used in the supervised learning workflow."""
    if search_type == "optuna":
        return tuning_xgboost_optuna(
            model_name=model_name,
            estimator=estimator,
            X_train=X_train,
            y_train=y_train,
            cv=cv,
            scoring=scoring,
            refit=refit,
            n_trials=n_trials,
            random_state=random_state,
            selector_k_param=selector_k_param,
        )

    if search_type == "random":
        return tuning_randomized_search(
            model_name=model_name,
            estimator=estimator,
            param_grid=param_grid,
            X_train=X_train,
            y_train=y_train,
            cv=cv,
            scoring=scoring,
            refit=refit,
            n_iter=n_iter,
            random_state=random_state,
            n_jobs=n_jobs,
        )


def run_tuning_suite(
    model_tuning,
    X_train,
    y_train,
    dataset_name="dataset",
    default_search_type="random",
    n_iter=30,
    n_trials=30,
):
    """Final function that is called to tune all models"""
    summaries = []
    searches = {}
    best_estimators = {}

    for model_name, spec in model_tuning.items():
        search, summary, best_estimator = tuning(
            model_name=model_name,
            estimator=spec["estimator"],
            param_grid=spec.get("params"),
            X_train=X_train,
            y_train=y_train,
            search_type=spec.get("search_type", default_search_type),
            n_iter=spec.get("n_iter", n_iter),
            n_trials=spec.get("n_trials", n_trials),
            selector_k_param=spec.get("selector_k_param", "selector__k"),
        )
        summary["dataset"] = dataset_name
        summaries.append(summary)
        searches[model_name] = search
        best_estimators[model_name] = best_estimator

    summary_df = pd.DataFrame(summaries)
    summary_df = summary_df.sort_values("mean_test_balanced_accuracy", ascending=False).reset_index(drop=True)
    return summary_df, searches, best_estimators

### SmartSeq_normalised tuning

The first full tuning run is performed on the SmartSeq_normalised datasets. These data are already filtered and normalised, so the model pipelines focus on machine-learning preprocessing: scaling, supervised gene selection with `SelectKBest`, optional PCA and the classifier itself. Three datasets are tuned: HCC1806, MCF7 and the combined HCC1806+MCF7 dataset.

In [ ]:
# Define SmartSeq_normalised tuning dictionary and switches.
TUNE_DUMMY = False
TUNE_LOGREG = True
TUNE_KNN = True
TUNE_RANDOM_FOREST = True
TUNE_GRADIENT_BOOSTING = True
TUNE_ADABOOST = True
TUNE_XGBOOST = True
TUNE_SVM_LINEAR = True
TUNE_SVM_RBF = True
TUNE_MLP = True


K_VALUES = [100, 500, 1000, "all"]
PCA_COMPONENTS = [10, 30, 50]

model_tuning = {}

if TUNE_DUMMY:
    model_tuning.update({
        "DummyClassifier": {
            "estimator": Pipeline([
                ("model", DummyClassifier()),
            ]),
            "params": {
                "model__strategy": ["most_frequent", "stratified"],
            },
            "search_type": "random",
            "n_iter": 2,
        }
    })

if TUNE_LOGREG:
    model_tuning.update({
        "LogisticRegression": {
            "estimator": Pipeline([
                ("scaler", StandardScaler()),
                ("selector", SelectKBest(score_func=safe_f_classif)),
                ("model", LogisticRegression(
                    max_iter=5000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                )),
            ]),
            "params": {
                "selector__k": K_VALUES,
                "model__C": [0.001, 0.01, 0.1, 1, 10],
                "model__l1_ratio": [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
                "model__solver": ["saga"],
            },
            "search_type": "random",
        }
    })

if TUNE_KNN:
    model_tuning.update({
        "kNN": {
            "estimator": Pipeline([
                ("scaler", StandardScaler()),
                ("selector", SelectKBest(score_func=safe_f_classif)),
                ("model", KNeighborsClassifier()),
            ]),
            "params": {
                "selector__k": K_VALUES,
                "model__n_neighbors": [3, 5, 7, 11],
                "model__weights": ["uniform", "distance"],
                "model__metric": ["euclidean", "manhattan"],
            },
            "search_type": "random",
        }
    })

if TUNE_RANDOM_FOREST:
    model_tuning.update({
        "RandomForest": {
            "estimator": Pipeline([
                ("selector", SelectKBest(score_func=safe_f_classif)),
                ("model", RandomForestClassifier(
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                )),
            ]),
            "params": {
                "selector__k": K_VALUES,
                "model__n_estimators": [100, 200, 300, 500],
                "model__max_depth": [None, 5, 10],
                "model__min_samples_split": [2, 5],
                "model__min_samples_leaf": [1, 2, 4],
                "model__max_features": ["sqrt", "log2"],
            },
            "search_type": "random",
        }
    })

if TUNE_GRADIENT_BOOSTING:
    model_tuning.update({
        "GradientBoosting": {
            "estimator": Pipeline([
                ("selector", SelectKBest(score_func=safe_f_classif)),
                ("model", GradientBoostingClassifier(random_state=RANDOM_STATE)),
            ]),
            "params": {
                "selector__k": K_VALUES,
                "model__n_estimators": [50, 100, 200],
                "model__learning_rate": [0.01, 0.05, 0.1],
                "model__max_depth": [2, 3],
            },
            "search_type": "random",
        }
    })

if TUNE_ADABOOST:
    model_tuning.update({
        "AdaBoost": {
            "estimator": Pipeline([
                ("selector", SelectKBest(score_func=safe_f_classif)),
                ("model", AdaBoostClassifier(random_state=RANDOM_STATE)),
            ]),
            "params": {
                "selector__k": K_VALUES,
                "model__n_estimators": [50, 100, 200],
                "model__learning_rate": [0.01, 0.1, 0.5, 1.0],
            },
            "search_type": "random",
        }
    })

if TUNE_XGBOOST:
    model_tuning.update({
        "XGBoost": {
            "estimator": Pipeline([
                ("selector", SelectKBest(score_func=safe_f_classif)),
                ("model", XGBClassifier(
                    eval_metric="logloss",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                    tree_method="hist",
                )),
            ]),
            "params": None,
            "search_type": "optuna",
            "n_trials": 30,
        }
    })

if TUNE_SVM_LINEAR:
    model_tuning.update({
        "SVM_linear": {
            "estimator": Pipeline([
                ("scaler", StandardScaler()),
                ("selector", SelectKBest(score_func=safe_f_classif)),
                ("model", SVC(
                    kernel="linear",
                    probability=True,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                )),
            ]),
            "params": {
                "selector__k": K_VALUES,
                "model__C": [0.001, 0.01, 0.1, 1, 10],
            },
            "search_type": "random",
        }
    })

if TUNE_SVM_RBF:
    model_tuning.update({
        "SVM_rbf": {
            "estimator": Pipeline([
                ("scaler", StandardScaler()),
                ("selector", SelectKBest(score_func=safe_f_classif)),
                ("pca", PCA(random_state=RANDOM_STATE)),
                ("model", SVC(
                    kernel="rbf",
                    probability=True,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                )),
            ]),
            "params": {
                "selector__k": K_VALUES,
                "pca__n_components": PCA_COMPONENTS,
                "model__C": [0.1, 1, 10],
                "model__gamma": ["scale", 0.001, 0.01],
            },
            "search_type": "random",
        }
    })

if TUNE_MLP:
    model_tuning.update({
        "MLP": {
            "estimator": Pipeline([
                ("scaler", StandardScaler()),
                ("selector", SelectKBest(score_func=safe_f_classif)),
                ("pca", PCA(random_state=RANDOM_STATE)),
                ("model", MLPClassifier(
                    max_iter=1000,
                    early_stopping=True,
                    validation_fraction=0.15,
                    n_iter_no_change=20,
                    random_state=RANDOM_STATE,
                )),
            ]),
            "params": {
                "selector__k": K_VALUES,
                "pca__n_components": PCA_COMPONENTS,
                "model__hidden_layer_sizes": [(32,), (64,), (64, 32)],
                "model__activation": ["relu", "tanh"],
                "model__alpha": [0.001, 0.01, 0.1],
                "model__learning_rate_init": [0.0001, 0.001],
            },
            "search_type": "random",
        }
    })

print("Models selected for tuning:")
print(list(model_tuning.keys()))

In [ ]:
# Run hyperparameter tuning on SmartSeq_normalised datasets.
RUN_TUNING = True
DATASETS_TO_TUNE = ["HCC1806", "MCF7","Combined"]
RANDOM_SEARCH_ITER = 30
OPTUNA_TRIALS = 30

all_tuning_summaries = []
all_tuning_searches = {}
all_best_estimators = {}

if RUN_TUNING:
    for dataset_name in DATASETS_TO_TUNE:
        print(f"\n########## TUNING ON {dataset_name} ##########")
        X_train = splits[dataset_name]["X_train"]
        y_train = splits[dataset_name]["y_train"]

        tuning_summary, tuning_searches, best_estimators = run_tuning_suite(
            model_tuning,
            X_train,
            y_train,
            dataset_name=dataset_name,
            default_search_type="random",
            n_iter=RANDOM_SEARCH_ITER,
            n_trials=OPTUNA_TRIALS,
        )

        all_tuning_summaries.append(tuning_summary)
        all_tuning_searches[dataset_name] = tuning_searches
        all_best_estimators[dataset_name] = best_estimators

    all_tuning_summary = pd.concat(all_tuning_summaries, axis=0).reset_index(drop=True)
    display(all_tuning_summary)
else:
    all_tuning_summary = pd.DataFrame()
    print("Tuning was not run.")

### SmartSeq_QCd preprocessing pipeline

The QCd branch tests our own preprocessing strategy instead of relying only on the professor-provided filtered/normalised files. The preprocessing is placed inside the sklearn pipeline to avoid leakage. This means that every step that learns from data is fitted only on the training fold during cross-validation and then applied to validation or test folds.

The custom preprocessing includes: selecting numeric gene columns, library-size normalisation, log1p transformation, supervised selection of informative genes, imputation, scaling and optional engineered per-cell features. These steps are combined with the same model families used for SmartSeq_normalised, making the comparison between preprocessing strategies fair and controlled.

Add gaspa's explanation

In [ ]:
# Define the custom SmartSeq_QCd preprocessing pipeline.

TARGET_COLUMN = "Condition"

METADATA_COLUMNS = [
    "Cell",
    "Cell Line",
    "PCR Plate",
    "Pos",
    "Condition",
    "Hours",
    "Cell name",
    "PreprocessingTag",
    "ProcessingComments",
]

MITOCHONDRIAL_PREFIX = "MT-"
RIBOSOMAL_PREFIXES = ("RPL", "RPS")

HYPOXIA_SIGNATURE = [
    "SLC2A1", "SLC2A3", "HK1", "HK2", "PFKL", "ALDOA",
    "PGAM1", "ENO1", "LDHA", "PDK1",
    "VEGFA", "VEGFC", "ANGPT2", "FLT1", "NRP1",
    "CA9", "CA12", "SLC9A1",
    "BNIP3", "BNIP3L", "BCL2L1",
    "EPO", "TFRC",
    "LOX", "MMP2", "CXCR4",
    "CDKN1A", "CCND1",
]

class GeneColumnSelector(BaseEstimator, TransformerMixin):
    """
    Select only numeric gene-expression columns.
    Metadata columns are excluded because they are not molecular features.
    This also prevents leakage from identifiers such as Cell or Cell name.
    """
    def __init__(self, metadata_columns=METADATA_COLUMNS):
        self.metadata_columns = metadata_columns

    def fit(self, X, y=None):
        blocked = set(self.metadata_columns)

        self.gene_columns_ = [
            col for col in X.columns
            if col not in blocked and pd.api.types.is_numeric_dtype(X[col])
        ]

        if len(self.gene_columns_) == 0:
            raise ValueError("No gene-expression columns found.")

        return self

    def transform(self, X):
        return X.loc[:, self.gene_columns_].to_numpy(dtype=np.float32, copy=False)


class LibrarySizeNormalizer(BaseEstimator, TransformerMixin):
    """
    Normalize each cell by its total counts.
    This makes cells comparable even if they have different sequencing depth.
    """
    def __init__(self, scale_factor=1e4):
        self.scale_factor = scale_factor

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=np.float32)
        total_counts = X.sum(axis=1, keepdims=True)
        total_counts = np.where(total_counts <= 0, 1.0, total_counts)

        return (X / total_counts) * self.scale_factor


class Log1pTransformer(BaseEstimator, TransformerMixin):
    """
    Apply log(1 + x).
    This reduces the skewness of gene-expression counts.
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.log1p(np.asarray(X, dtype=np.float32))


class FeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Create per-cell engineered features.
    These features are computed row-wise and do not use the target variable.

    Features:
    - log total counts
    - number of detected genes
    - percentage of detected genes
    - percentage of zero genes
    - mitochondrial count fraction
    - ribosomal count fraction
    - top-gene fraction
    - optional hypoxia signature score
    """

    def __init__(
        self,
        metadata_columns=METADATA_COLUMNS,
        mitochondrial_prefix=MITOCHONDRIAL_PREFIX,
        ribosomal_prefixes=RIBOSOMAL_PREFIXES,
        hypoxia_signature=HYPOXIA_SIGNATURE,
        add_hypoxia_signature_score=True,
        scale_factor=1e4,
    ):
        self.metadata_columns = metadata_columns
        self.mitochondrial_prefix = mitochondrial_prefix
        self.ribosomal_prefixes = ribosomal_prefixes
        self.hypoxia_signature = hypoxia_signature
        self.add_hypoxia_signature_score = add_hypoxia_signature_score
        self.scale_factor = scale_factor

    def fit(self, X, y=None):
        blocked = set(self.metadata_columns)

        self.gene_columns_ = [
            col for col in X.columns
            if col not in blocked and pd.api.types.is_numeric_dtype(X[col])
        ]

        self.mitochondrial_columns_ = [
            col for col in self.gene_columns_
            if str(col).startswith(self.mitochondrial_prefix)
        ]

        self.ribosomal_columns_ = [
            col for col in self.gene_columns_
            if str(col).startswith(self.ribosomal_prefixes)
        ]

        self.hypoxia_signature_columns_ = [
            gene for gene in self.hypoxia_signature
            if gene in self.gene_columns_
        ]

        return self

    def transform(self, X):
        counts = X.loc[:, self.gene_columns_].to_numpy(dtype=np.float32, copy=False)

        total_counts = counts.sum(axis=1)
        safe_total_counts = np.where(total_counts <= 0, 1.0, total_counts)

        n_genes = counts.shape[1]

        n_detected_genes = (counts > 0).sum(axis=1).astype(np.float32)
        pct_detected_genes = n_detected_genes / n_genes
        pct_zero_genes = 1.0 - pct_detected_genes

        top_gene_fraction = counts.max(axis=1) / safe_total_counts

        mito_counts = self._sum_columns(X, self.mitochondrial_columns_)
        ribo_counts = self._sum_columns(X, self.ribosomal_columns_)

        pct_mito_counts = mito_counts / safe_total_counts
        pct_ribo_counts = ribo_counts / safe_total_counts

        features = [
            np.log1p(total_counts),
            n_detected_genes,
            pct_detected_genes,
            pct_zero_genes,
            pct_mito_counts,
            pct_ribo_counts,
            top_gene_fraction,
        ]

        if self.add_hypoxia_signature_score:
            features.append(
                self._compute_hypoxia_signature_score(X, safe_total_counts)
            )

        return np.column_stack(features).astype(np.float32)

    def _sum_columns(self, X, columns):
        if len(columns) == 0:
            return np.zeros(X.shape[0], dtype=np.float32)

        return (
            X.loc[:, columns]
            .to_numpy(dtype=np.float32, copy=False)
            .sum(axis=1)
        )

    def _compute_hypoxia_signature_score(self, X, safe_total_counts):
        if len(self.hypoxia_signature_columns_) == 0:
            return np.zeros(X.shape[0], dtype=np.float32)

        signature_counts = (
            X.loc[:, self.hypoxia_signature_columns_]
            .to_numpy(dtype=np.float32, copy=False)
        )

        normalized_signature = (
            signature_counts / safe_total_counts[:, None]
        ) * self.scale_factor

        return np.log1p(normalized_signature).mean(axis=1).astype(np.float32)

def make_gene_expression_branch():
    """
    Main gene-expression preprocessing branch.
    The 3000 genes are selected with SelectKBest using f_classif.

    This is supervised feature selection, but it is safe because it is
    inside the sklearn Pipeline and therefore fitted only on training folds
    during cross-validation.
    """

    return Pipeline(steps=[
        ("select_gene_columns",GeneColumnSelector()),
        ("normalize_library_size",LibrarySizeNormalizer(scale_factor=1e4)),
        ("log1p",Log1pTransformer()),
        ("select_k_best_genes",SelectKBest(score_func=safe_f_classif,k=3000)),
        ("imputer",SimpleImputer(strategy="median")),
        ("scaler",StandardScaler()),
    ])


def make_engineered_feature_branch(add_hypoxia_signature_score=True):
    """
    Branch for engineered per-cell features.
    These features are added to the selected gene-expression features.
    """

    return Pipeline(steps=[
        ("feature_engineer",FeatureEngineer(add_hypoxia_signature_score=add_hypoxia_signature_score)),
        ("imputer",SimpleImputer(strategy="median")),
        ("scaler",StandardScaler()),
    ])


def make_preprocessor(
    use_engineered_features=True,
    add_hypoxia_signature_score=True,
):
    """
    Complete preprocessing object.
    It combines:
    - gene-expression preprocessing with fixed SelectKBest(k=3000)
    - optional engineered per-cell features
    """

    gene_branch = make_gene_expression_branch()

    if use_engineered_features:
        feature_block = FeatureUnion(transformer_list=[
            ("genes",gene_branch),
            ("engineered_features",make_engineered_feature_branch(add_hypoxia_signature_score=add_hypoxia_signature_score)),
        ])
    else:
        feature_block = gene_branch

    return feature_block

### SmartSeq_QCd tuning

In [ ]:
# Define SmartSeq_QCd tuning dictionary and switches.
# These switches decide what model are trained
TUNE_DUMMY_2 = False
TUNE_LOGREG_2 = True
TUNE_KNN_2 = True
TUNE_RANDOM_FOREST_2 = True
TUNE_GRADIENT_BOOSTING_2 = True
TUNE_ADABOOST_2 = True
TUNE_XGBOOST_2 = True
TUNE_SVM_LINEAR_2 = True
TUNE_SVM_RBF_2 = True
TUNE_MLP_2 = True


K_VALUES_2 = [100, 500, 1000, "all"]
PCA_COMPONENTS_2 = [10, 30, 50]
SELECTOR_K_PARAM_2 = "Preprocess__genes__select_k_best_genes__k"

model_tuning_2 = {}

if TUNE_DUMMY_2:
    model_tuning_2.update({
        "DummyClassifier": {
            "estimator": Pipeline([
                ("model", DummyClassifier()),
            ]),
            "params": {
                "model__strategy": ["most_frequent", "stratified"],
            },
            "search_type": "random",
            "n_iter": 2,
        }
    })

if TUNE_LOGREG_2:
    model_tuning_2.update({
        "LogisticRegression": {
            "estimator": Pipeline([
                ("Preprocess", make_preprocessor()),
                ("model", LogisticRegression(
                    max_iter=5000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                )),
            ]),
            "params": {
                SELECTOR_K_PARAM_2: K_VALUES_2,
                "model__C": [0.001, 0.01, 0.1, 1, 10],
                "model__l1_ratio": [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
                "model__solver": ["saga"],
            },
            "search_type": "random",
        }
    })

if TUNE_KNN_2:
    model_tuning_2.update({
        "kNN": {
            "estimator": Pipeline([
                ("Preprocess", make_preprocessor()),
                ("model", KNeighborsClassifier()),
            ]),
            "params": {
                SELECTOR_K_PARAM_2: K_VALUES_2,
                "model__n_neighbors": [3, 5, 7, 11],
                "model__weights": ["uniform", "distance"],
                "model__metric": ["euclidean", "manhattan"],
            },
            "search_type": "random",
        }
    })

if TUNE_RANDOM_FOREST_2:
    model_tuning_2.update({
        "RandomForest": {
            "estimator": Pipeline([
                ("Preprocess", make_preprocessor()),
                ("model", RandomForestClassifier(
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                )),
            ]),
            "params": {
                SELECTOR_K_PARAM_2: K_VALUES_2,
                "model__n_estimators": [100, 200, 300, 500],
                "model__max_depth": [None, 5, 10],
                "model__min_samples_split": [2, 5],
                "model__min_samples_leaf": [1, 2, 4],
                "model__max_features": ["sqrt", "log2"],
            },
            "search_type": "random",
        }
    })

if TUNE_GRADIENT_BOOSTING_2:
    model_tuning_2.update({
        "GradientBoosting": {
            "estimator": Pipeline([
                ("Preprocess", make_preprocessor()),
                ("model", GradientBoostingClassifier(random_state=RANDOM_STATE)),
            ]),
            "params": {
                SELECTOR_K_PARAM_2: K_VALUES_2,
                "model__n_estimators": [50, 100, 200],
                "model__learning_rate": [0.01, 0.05, 0.1],
                "model__max_depth": [2, 3],
            },
            "search_type": "random",
        }
    })

if TUNE_ADABOOST_2:
    model_tuning_2.update({
        "AdaBoost": {
            "estimator": Pipeline([
                ("Preprocess", make_preprocessor()),
                ("model", AdaBoostClassifier(random_state=RANDOM_STATE)),
            ]),
            "params": {
                SELECTOR_K_PARAM_2: K_VALUES_2,
                "model__n_estimators": [50, 100, 200],
                "model__learning_rate": [0.01, 0.1, 0.5, 1.0],
            },
            "search_type": "random",
        }
    })

if TUNE_XGBOOST_2:
    model_tuning_2.update({
        "XGBoost": {
            "estimator": Pipeline([
                ("Preprocess", make_preprocessor()),
                ("model", XGBClassifier(
                    eval_metric="logloss",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                    tree_method="hist",
                )),
            ]),
            "params": None,
            "search_type": "optuna",
            "n_trials": 30,
            "selector_k_param": SELECTOR_K_PARAM_2,
        }
    })

if TUNE_SVM_LINEAR_2:
    model_tuning_2.update({
        "SVM_linear": {
            "estimator": Pipeline([
                ("Preprocess", make_preprocessor()),
                ("model", SVC(
                    kernel="linear",
                    probability=True,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                )),
            ]),
            "params": {
                SELECTOR_K_PARAM_2: K_VALUES_2,
                "model__C": [0.001, 0.01, 0.1, 1, 10],
            },
            "search_type": "random",
        }
    })

if TUNE_SVM_RBF_2:
    model_tuning_2.update({
        "SVM_rbf": {
            "estimator": Pipeline([
                ("Preprocess", make_preprocessor()),
                ("pca", PCA(random_state=RANDOM_STATE)),
                ("model", SVC(
                    kernel="rbf",
                    probability=True,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                )),
            ]),
            "params": {
                SELECTOR_K_PARAM_2: K_VALUES_2,
                "pca__n_components": PCA_COMPONENTS_2,
                "model__C": [0.1, 1, 10],
                "model__gamma": ["scale", 0.001, 0.01],
            },
            "search_type": "random",
        }
    })

if TUNE_MLP_2:
    model_tuning_2.update({
        "MLP": {
            "estimator": Pipeline([
                ("Preprocess", make_preprocessor()),
                ("pca", PCA(random_state=RANDOM_STATE)),
                ("model", MLPClassifier(
                    max_iter=1000,
                    early_stopping=True,
                    validation_fraction=0.15,
                    n_iter_no_change=20,
                    random_state=RANDOM_STATE,
                )),
            ]),
            "params": {
                SELECTOR_K_PARAM_2: K_VALUES_2,
                "pca__n_components": PCA_COMPONENTS_2,
                "model__hidden_layer_sizes": [(32,), (64,), (64, 32)],
                "model__activation": ["relu", "tanh"],
                "model__alpha": [0.001, 0.01, 0.1],
                "model__learning_rate_init": [0.0001, 0.001],
            },
            "search_type": "random",
        }
    })

print("Models selected for tuning:")
print(list(model_tuning_2.keys()))

In [ ]:
# Run hyperparameter tuning on SmartSeq_QCd datasets.
RUN_TUNING_2 = True
DATASETS_TO_TUNE_2 = ["HCC1806_QCd", "MCF7_QCd","Combined_QCd"] 
RANDOM_SEARCH_ITER_2 = 30
OPTUNA_TRIALS_2 = 30

all_tuning_summaries_2 = []
all_tuning_searches_2 = {}
all_best_estimators_2 = {}

if RUN_TUNING_2:
    for dataset_name in DATASETS_TO_TUNE_2:
        print(f"\n########## TUNING ON {dataset_name} ##########")
        X_train = splits[dataset_name]["X_train"]
        y_train = splits[dataset_name]["y_train"]

        tuning_summary, tuning_searches, best_estimators = run_tuning_suite(
            model_tuning_2,
            X_train,
            y_train,
            dataset_name=dataset_name,
            default_search_type="random",
            n_iter=RANDOM_SEARCH_ITER_2,
            n_trials=OPTUNA_TRIALS_2,
        )

        all_tuning_summaries_2.append(tuning_summary)
        all_tuning_searches_2[dataset_name] = tuning_searches
        all_best_estimators_2[dataset_name] = best_estimators

    all_tuning_summary_2 = pd.concat(all_tuning_summaries_2, axis=0).reset_index(drop=True)
    display(all_tuning_summary_2)
else:
    all_tuning_summary_2 = pd.DataFrame()
    print("Tuning was not run.")

### DropSeq_normalised tuning

The DropSeq tuning section applies the same supervised learning framework to the larger DropSeq dataset. Quick benchmarking is skipped here to avoid unnecessary repetition: once the workflow has been validated on SmartSeq, DropSeq moves directly to hyperparameter tuning, five-scenario evaluation and stacking.

The purpose of this section is to understand whether the same model families remain effective when trained on a larger number of cells obtained with a different single-cell sequencing technology.

In [ ]:
# Hyperparameter tuning on DropSeq filtered/normalised data

# RUN_DROPSEQ_TUNING = False
# DATASETS_TO_TUNE_DROPSEQ = ["DropSeq_HCC1806", "DropSeq_MCF7", "DropSeq_Combined"]
# RANDOM_SEARCH_ITER_DROPSEQ = 30
# OPTUNA_TRIALS_DROPSEQ = 30

# all_tuning_summaries_dropseq = []
# all_tuning_searches_dropseq = {}
# all_best_estimators_dropseq = {}

# if RUN_DROPSEQ_TUNING:
#     for dataset_name in DATASETS_TO_TUNE_DROPSEQ:
#         print(f"\n########## TUNING ON {dataset_name} ##########")
#         X_train = splits[dataset_name]["X_train"]
#         y_train = splits[dataset_name]["y_train"]

#         tuning_summary, tuning_searches, best_estimators = run_tuning_suite(
#             model_tuning,
#             X_train,
#             y_train,
#             dataset_name=dataset_name,
#             default_search_type="random",
#             n_iter=RANDOM_SEARCH_ITER_DROPSEQ,
#             n_trials=OPTUNA_TRIALS_DROPSEQ,
#         )

#         all_tuning_summaries_dropseq.append(tuning_summary)
#         all_tuning_searches_dropseq[dataset_name] = tuning_searches
#         all_best_estimators_dropseq[dataset_name] = best_estimators

#     all_tuning_summary_dropseq = pd.concat(
#         all_tuning_summaries_dropseq,
#         axis=0,
#     ).reset_index(drop=True)

#     display(all_tuning_summary_dropseq)
# else:
#     all_tuning_summary_dropseq = pd.DataFrame()
#     print("DropSeq tuning was not run.")

This should have been the hyperparameter tuning section for the DropSeq dataset but it's computational infeasible, so we have decided to fit only models with speciific parameters (taking the ones used for the first quick evaluation for the SmartSeq normalised dataset)

In [ ]:
# Run the first quick benchmark on DropSeq_normalised.
RUN_EVALUATION = True
DATASETS_EVAL = ["DropSeq_HCC1806", "DropSeq_MCF7", "DropSeq_Combined",]

DropSeq_cv_results = []
DropSeq_holdout_results = []
DropSeq_fitted_models = {}
DropSeq_predictions = {}
DropSeq_scores = {}

if RUN_EVALUATION:
    for dataset_name in DATASETS_EVAL:
        X_train = splits[dataset_name]["X_train"]
        y_train = splits[dataset_name]["y_train"]
        X_test = splits[dataset_name]["X_test"]
        y_test = splits[dataset_name]["y_test"]

        DropSeq_fitted_models[dataset_name] = {}
        DropSeq_predictions[dataset_name] = {}
        DropSeq_scores[dataset_name] = {}

        for model_name, model in models.items():
            print(f"\nQuick evaluation | {dataset_name} | {model_name}")
            cv_row, _ = evaluation_model_cv(
                model,
                X_train,
                y_train,
                dataset_name=dataset_name,
                model_name=model_name,
            )
            DropSeq_cv_results.append(cv_row)

            holdout_row, fitted_model, y_pred, y_score = evaluation_model(
                model,
                X_train,
                y_train,
                X_test,
                y_test,
                model_name=model_name,
                dataset_name=dataset_name,
            )
            DropSeq_holdout_results.append(holdout_row)
            DropSeq_fitted_models[dataset_name][model_name] = fitted_model
            DropSeq_predictions[dataset_name][model_name] = y_pred
            DropSeq_scores[dataset_name][model_name] = y_score

    DropSeq_cv_results = pd.DataFrame(DropSeq_cv_results)
    DropSeq_holdout_results = pd.DataFrame(DropSeq_holdout_results)


    DropSeq_cv_results = DropSeq_cv_results.sort_values(
        ["dataset", "cv_balanced_accuracy_mean"],
        ascending=[True, False],
    ).reset_index(drop=True)

    
    DropSeq_holdout_results = DropSeq_holdout_results.sort_values(
        ["dataset", "balanced_accuracy"],
        ascending=[True, False],
    ).reset_index(drop=True)

    print("Cross-validation results:")
    display(DropSeq_cv_results)

    print("Hold-out results:")
    display(DropSeq_holdout_results)
    
else:
    DropSeq_cv_results = pd.DataFrame()
    DropSeq_holdout_results = pd.DataFrame()
    print("Quick evaluation was not run.")

Here we can see a big difference with respect to the previous cases, indeed, non-linear models performed better than linear models.
On the SmartSeq dataset non-linear models tends to overfitt too much resulting in a slighlty worse balanced_accuracy, however, using more and more data they generalise very well obtaining a better score (we can see in particular MLP model that has moved from the bottom to the top of the ranking)

However, we can already see that on the combined dataset scores are slightly less than on the speficic dataset but this point will be develped better in the next section

In [ ]:
# Convert DropSeq quick evaluation outputs into tuning-like objects.

RUN_DROPSEQ_TUNING = True

DATASETS_TO_TUNE_DROPSEQ = [
    "DropSeq_HCC1806",
    "DropSeq_MCF7",
    "DropSeq_Combined",
]

all_tuning_summaries_dropseq = []
all_tuning_searches_dropseq = {}
all_best_estimators_dropseq = {}

for dataset_name in DATASETS_TO_TUNE_DROPSEQ:
    cv_results_dataset = (
        DropSeq_cv_results[DropSeq_cv_results["dataset"] == dataset_name]
        .sort_values("cv_balanced_accuracy_mean", ascending=False)
        .reset_index(drop=True)
    )

    all_tuning_searches_dropseq[dataset_name] = {}
    all_best_estimators_dropseq[dataset_name] = {}

    rows = []

    for _, row in cv_results_dataset.iterrows():
        model_name = row["model"]

        if model_name not in DropSeq_fitted_models[dataset_name]:
            continue

        all_tuning_searches_dropseq[dataset_name][model_name] = None
        all_best_estimators_dropseq[dataset_name][model_name] = clone(
            DropSeq_fitted_models[dataset_name][model_name]
        )

        rows.append({
            "dataset": dataset_name,
            "model": model_name,
            "search_method": "QuickCV_NoTuning",
            "best_params": {},
            "best_score_refit": row["cv_balanced_accuracy_mean"],

            "mean_test_accuracy": row.get("cv_accuracy_mean", np.nan),
            "std_test_accuracy": row.get("cv_accuracy_std", np.nan),
            "mean_train_accuracy": np.nan,

            "mean_test_balanced_accuracy": row.get("cv_balanced_accuracy_mean", np.nan),
            "std_test_balanced_accuracy": row.get("cv_balanced_accuracy_std", np.nan),
            "mean_train_balanced_accuracy": row.get("train_balanced_accuracy_mean", np.nan),

            "mean_test_f1_macro": row.get("cv_f1_macro_mean", np.nan),
            "std_test_f1_macro": row.get("cv_f1_macro_std", np.nan),
            "mean_train_f1_macro": np.nan,

            "mean_test_roc_auc": row.get("cv_roc_auc_mean", np.nan),
            "std_test_roc_auc": row.get("cv_roc_auc_std", np.nan),
            "mean_train_roc_auc": np.nan,
        })

    all_tuning_summaries_dropseq.append(pd.DataFrame(rows))

all_tuning_summary_dropseq = pd.concat(
    all_tuning_summaries_dropseq,
    axis=0,
).reset_index(drop=True)

display(all_tuning_summary_dropseq)

## Five evaluation scenarios

After tuning, each dataset family is evaluated in five scenarios:

- **MCF7 → MCF7**: within-cell-line performance for MCF7;
- **HCC1806 → HCC1806**: within-cell-line performance for HCC1806;
- **Combined → Combined**: performance of a general model trained on both cell lines;
- **MCF7 → HCC1806**: transfer from MCF7 to HCC1806;
- **HCC1806 → MCF7**: transfer from HCC1806 to MCF7.

The first three scenarios measure classification performance when the training and test data come from the same distribution or a mixed distribution. The last two scenarios are biologically more demanding: they test whether the learned hypoxia signal generalises across cell lines.

### SmartSeq_normalised

The SmartSeq_normalised five-scenario evaluation uses the best tuned models from the professor-processed data. In the saved run, internal scenarios are extremely strong, with several models reaching near-perfect or perfect hold-out performance. Cross-cell-line transfer is also informative: the best models still perform well, although the direction of transfer and the model family matter. This suggests that the SmartSeq_normalised matrices contain both a shared hypoxia signal and some cell-line-specific structure.

In [ ]:
# Evaluate tuned SmartSeq_normalised models in five scenarios.

RUN_SMARTSEQ_FIVE_SCENARIOS = True

five_scenario_results = []
scenario_fitted_models = {}
scenario_predictions = {}
scenario_scores = {}


def align_train_test_features(X_train, X_test):
    """Align train and test matrices to common genes, preserving the training-column order."""
    common_genes = X_train.columns.intersection(X_test.columns)

    if len(common_genes) == 0:
        raise ValueError("No common genes between train and test matrices.")

    return X_train.loc[:, common_genes], X_test.loc[:, common_genes], common_genes


smartseq_scenario_specs = [
    {
        "scenario_name": "MCF7_to_MCF7",
        "tuning_dataset": "MCF7",
        "X_train": splits["MCF7"]["X_train"],
        "y_train": splits["MCF7"]["y_train"],
        "X_test": splits["MCF7"]["X_test"],
        "y_test": splits["MCF7"]["y_test"],
        "source_cell_line": "MCF7",
        "target_cell_line": "MCF7",
    },
    {
        "scenario_name": "HCC1806_to_HCC1806",
        "tuning_dataset": "HCC1806",
        "X_train": splits["HCC1806"]["X_train"],
        "y_train": splits["HCC1806"]["y_train"],
        "X_test": splits["HCC1806"]["X_test"],
        "y_test": splits["HCC1806"]["y_test"],
        "source_cell_line": "HCC1806",
        "target_cell_line": "HCC1806",
    },
    {
        "scenario_name": "Combined_to_Combined",
        "tuning_dataset": "Combined",
        "X_train": splits["Combined"]["X_train"],
        "y_train": splits["Combined"]["y_train"],
        "X_test": splits["Combined"]["X_test"],
        "y_test": splits["Combined"]["y_test"],
        "source_cell_line": "Combined",
        "target_cell_line": "Combined",
    },
    {
        "scenario_name": "MCF7_to_HCC1806",
        "tuning_dataset": "MCF7",
        "X_train": X_mcf,
        "y_train": y_mcf,
        "X_test": X_hcc,
        "y_test": y_hcc,
        "source_cell_line": "MCF7",
        "target_cell_line": "HCC1806",
    },
    {
        "scenario_name": "HCC1806_to_MCF7",
        "tuning_dataset": "HCC1806",
        "X_train": X_hcc,
        "y_train": y_hcc,
        "X_test": X_mcf,
        "y_test": y_mcf,
        "source_cell_line": "HCC1806",
        "target_cell_line": "MCF7",
    },
]


if RUN_SMARTSEQ_FIVE_SCENARIOS and RUN_TUNING:
    for spec in smartseq_scenario_specs:
        scenario_name = spec["scenario_name"]
        tuning_dataset = spec["tuning_dataset"]

        if tuning_dataset not in all_best_estimators:
            print(f"Skipping {scenario_name}: no tuned models available for {tuning_dataset}.")
            continue

        X_train_aligned, X_test_aligned, common_genes = align_train_test_features(
            spec["X_train"],
            spec["X_test"],
        )

        scenario_fitted_models[scenario_name] = {}
        scenario_predictions[scenario_name] = {}
        scenario_scores[scenario_name] = {}

        for model_name, estimator in all_best_estimators[tuning_dataset].items():
            print(f"SmartSeq_normalised scenario evaluation | {scenario_name} | {model_name}")

            result, fitted_model, y_pred, y_score = evaluation_model(
                estimator,
                X_train_aligned,
                spec["y_train"],
                X_test_aligned,
                spec["y_test"],
                model_name=model_name,
                dataset_name=scenario_name,
            )

            result.update({
                "scenario": scenario_name,
                "tuning_dataset": tuning_dataset,
                "source_cell_line": spec["source_cell_line"],
                "target_cell_line": spec["target_cell_line"],
                "n_train_cells": X_train_aligned.shape[0],
                "n_test_cells": X_test_aligned.shape[0],
                "n_common_genes": len(common_genes),
            })

            five_scenario_results.append(result)
            scenario_fitted_models[scenario_name][model_name] = fitted_model
            scenario_predictions[scenario_name][model_name] = y_pred
            scenario_scores[scenario_name][model_name] = y_score

    five_scenario_results = pd.DataFrame(five_scenario_results)

    if not five_scenario_results.empty:
        five_scenario_results = five_scenario_results.sort_values(
            ["scenario", "balanced_accuracy"],
            ascending=[True, False],
        ).reset_index(drop=True)

        internal_scenarios = [
            "MCF7_to_MCF7",
            "HCC1806_to_HCC1806",
            "Combined_to_Combined",
        ]

        cross_scenarios = [
            "MCF7_to_HCC1806",
            "HCC1806_to_MCF7",
        ]

        tuned_holdout_results = five_scenario_results[
            five_scenario_results["scenario"].isin(internal_scenarios)
        ].copy().reset_index(drop=True)

        cross_cell_line_tuned_results = five_scenario_results[
            five_scenario_results["scenario"].isin(cross_scenarios)
        ].copy().reset_index(drop=True)

        best_tuned_model_by_scenario = (
            five_scenario_results
            .sort_values(["scenario", "balanced_accuracy"], ascending=[True, False])
            .groupby("scenario", as_index=False)
            .head(1)
            .reset_index(drop=True)
        )

        print("All five SmartSeq_normalised scenario results:")
        display(five_scenario_results)

        print("Best tuned SmartSeq_normalised model in each scenario:")
        display(best_tuned_model_by_scenario)
    else:
        tuned_holdout_results = pd.DataFrame()
        cross_cell_line_tuned_results = pd.DataFrame()
        best_tuned_model_by_scenario = pd.DataFrame()
        print("No SmartSeq_normalised scenario results were produced.")
else:
    five_scenario_results = pd.DataFrame()
    tuned_holdout_results = pd.DataFrame()
    cross_cell_line_tuned_results = pd.DataFrame()
    best_tuned_model_by_scenario = pd.DataFrame()
    print("SmartSeq_normalised scenario evaluation was not run.")

### SmartSeq_QCd

The QCd five-scenario evaluation repeats the same experiment after applying our preprocessing pipeline. The goal is not only to maximise performance, but to test whether our preprocessing produces a representation that is as useful as the professor-provided filtered/normalised data. In the saved run, within-cell-line performance remains very high, while cross-cell-line transfer is somewhat more variable. This suggests that the custom preprocessing preserves the hypoxia signal but may not always improve generalisation across biological backgrounds.

In [ ]:
# Evaluate tuned SmartSeq_QCd models in five scenarios.

RUN_QCD_FIVE_SCENARIOS = True

qcd_five_scenario_results = []
qcd_scenario_fitted_models = {}
qcd_scenario_predictions = {}
qcd_scenario_scores = {}

qcd_scenario_specs = [
    {
        "scenario_name": "MCF7_QCd_to_MCF7_QCd",
        "tuning_dataset": "MCF7_QCd",
        "X_train": splits["MCF7_QCd"]["X_train"],
        "y_train": splits["MCF7_QCd"]["y_train"],
        "X_test": splits["MCF7_QCd"]["X_test"],
        "y_test": splits["MCF7_QCd"]["y_test"],
        "source_cell_line": "MCF7",
        "target_cell_line": "MCF7",
    },
    {
        "scenario_name": "HCC1806_QCd_to_HCC1806_QCd",
        "tuning_dataset": "HCC1806_QCd",
        "X_train": splits["HCC1806_QCd"]["X_train"],
        "y_train": splits["HCC1806_QCd"]["y_train"],
        "X_test": splits["HCC1806_QCd"]["X_test"],
        "y_test": splits["HCC1806_QCd"]["y_test"],
        "source_cell_line": "HCC1806",
        "target_cell_line": "HCC1806",
    },
    {
        "scenario_name": "Combined_QCd_to_Combined_QCd",
        "tuning_dataset": "Combined_QCd",
        "X_train": splits["Combined_QCd"]["X_train"],
        "y_train": splits["Combined_QCd"]["y_train"],
        "X_test": splits["Combined_QCd"]["X_test"],
        "y_test": splits["Combined_QCd"]["y_test"],
        "source_cell_line": "Combined",
        "target_cell_line": "Combined",
    },
    {
        "scenario_name": "MCF7_QCd_to_HCC1806_QCd",
        "tuning_dataset": "MCF7_QCd",
        "X_train": X_mcf_qcd,
        "y_train": y_mcf_qcd,
        "X_test": X_hcc_qcd,
        "y_test": y_hcc_qcd,
        "source_cell_line": "MCF7",
        "target_cell_line": "HCC1806",
    },
    {
        "scenario_name": "HCC1806_QCd_to_MCF7_QCd",
        "tuning_dataset": "HCC1806_QCd",
        "X_train": X_hcc_qcd,
        "y_train": y_hcc_qcd,
        "X_test": X_mcf_qcd,
        "y_test": y_mcf_qcd,
        "source_cell_line": "HCC1806",
        "target_cell_line": "MCF7",
    },
]

if RUN_QCD_FIVE_SCENARIOS and RUN_TUNING_2:
    for spec in qcd_scenario_specs:
        scenario_name = spec["scenario_name"]
        tuning_dataset = spec["tuning_dataset"]

        if tuning_dataset not in all_best_estimators_2:
            print(f"Skipping {scenario_name}: no tuned models available for {tuning_dataset}.")
            continue

        X_train_aligned, X_test_aligned, common_genes = align_train_test_features(
            spec["X_train"],
            spec["X_test"],
        )

        qcd_scenario_fitted_models[scenario_name] = {}
        qcd_scenario_predictions[scenario_name] = {}
        qcd_scenario_scores[scenario_name] = {}

        for model_name, estimator in all_best_estimators_2[tuning_dataset].items():
            print(f"QCd scenario evaluation | {scenario_name} | {model_name}")

            result, fitted_model, y_pred, y_score = evaluation_model(
                estimator,
                X_train_aligned,
                spec["y_train"],
                X_test_aligned,
                spec["y_test"],
                model_name=model_name,
                dataset_name=scenario_name,
            )

            result.update({
                "scenario": scenario_name,
                "tuning_dataset": tuning_dataset,
                "source_cell_line": spec["source_cell_line"],
                "target_cell_line": spec["target_cell_line"],
                "n_train_cells": X_train_aligned.shape[0],
                "n_test_cells": X_test_aligned.shape[0],
                "n_common_genes": len(common_genes),
            })

            qcd_five_scenario_results.append(result)
            qcd_scenario_fitted_models[scenario_name][model_name] = fitted_model
            qcd_scenario_predictions[scenario_name][model_name] = y_pred
            qcd_scenario_scores[scenario_name][model_name] = y_score

    qcd_five_scenario_results = pd.DataFrame(qcd_five_scenario_results)

    if not qcd_five_scenario_results.empty:
        qcd_five_scenario_results = qcd_five_scenario_results.sort_values(
            ["scenario", "balanced_accuracy"],
            ascending=[True, False],
        ).reset_index(drop=True)

        qcd_internal_scenarios = [
            "MCF7_QCd_to_MCF7_QCd",
            "HCC1806_QCd_to_HCC1806_QCd",
            "Combined_QCd_to_Combined_QCd",
        ]

        qcd_cross_scenarios = [
            "MCF7_QCd_to_HCC1806_QCd",
            "HCC1806_QCd_to_MCF7_QCd",
        ]

        qcd_tuned_holdout_results = qcd_five_scenario_results[
            qcd_five_scenario_results["scenario"].isin(qcd_internal_scenarios)
        ].copy().reset_index(drop=True)

        qcd_cross_cell_line_tuned_results = qcd_five_scenario_results[
            qcd_five_scenario_results["scenario"].isin(qcd_cross_scenarios)
        ].copy().reset_index(drop=True)

        qcd_best_tuned_model_by_scenario = (
            qcd_five_scenario_results
            .sort_values(["scenario", "balanced_accuracy"], ascending=[True, False])
            .groupby("scenario", as_index=False)
            .head(1)
            .reset_index(drop=True)
        )

        print("All five QCd scenario results:")
        display(qcd_five_scenario_results)

        print("Best tuned QCd model in each scenario:")
        display(qcd_best_tuned_model_by_scenario)
    else:
        qcd_tuned_holdout_results = pd.DataFrame()
        qcd_cross_cell_line_tuned_results = pd.DataFrame()
        qcd_best_tuned_model_by_scenario = pd.DataFrame()
        print("No QCd scenario results were produced.")
else:
    qcd_five_scenario_results = pd.DataFrame()
    qcd_tuned_holdout_results = pd.DataFrame()
    qcd_cross_cell_line_tuned_results = pd.DataFrame()
    qcd_best_tuned_model_by_scenario = pd.DataFrame()
    print("QCd scenario evaluation was not run.")

### DropSeq_normalised

The DropSeq five-scenario evaluation is especially important because DropSeq contains more cells. Larger sample size should make the estimates more stable and may help models learn a more robust hypoxia response. The same five scenarios are used so that DropSeq can be compared directly with the two SmartSeq-based analyses.

In [ ]:
# Evaluate tuned DropSeq_normalised models in five scenarios.

RUN_DROPSEQ_FIVE_SCENARIOS = True

dropseq_five_scenario_results = []
dropseq_scenario_fitted_models = {}
dropseq_scenario_predictions = {}
dropseq_scenario_scores = {}

dropseq_scenario_specs = [
    {
        "scenario_name": "DropSeq_MCF7_to_DropSeq_MCF7",
        "tuning_dataset": "DropSeq_MCF7",
        "X_train": splits["DropSeq_MCF7"]["X_train"],
        "y_train": splits["DropSeq_MCF7"]["y_train"],
        "X_test": splits["DropSeq_MCF7"]["X_test"],
        "y_test": splits["DropSeq_MCF7"]["y_test"],
        "source_cell_line": "MCF7",
        "target_cell_line": "MCF7",
    },
    {
        "scenario_name": "DropSeq_HCC1806_to_DropSeq_HCC1806",
        "tuning_dataset": "DropSeq_HCC1806",
        "X_train": splits["DropSeq_HCC1806"]["X_train"],
        "y_train": splits["DropSeq_HCC1806"]["y_train"],
        "X_test": splits["DropSeq_HCC1806"]["X_test"],
        "y_test": splits["DropSeq_HCC1806"]["y_test"],
        "source_cell_line": "HCC1806",
        "target_cell_line": "HCC1806",
    },
    {
        "scenario_name": "DropSeq_Combined_to_DropSeq_Combined",
        "tuning_dataset": "DropSeq_Combined",
        "X_train": splits["DropSeq_Combined"]["X_train"],
        "y_train": splits["DropSeq_Combined"]["y_train"],
        "X_test": splits["DropSeq_Combined"]["X_test"],
        "y_test": splits["DropSeq_Combined"]["y_test"],
        "source_cell_line": "Combined",
        "target_cell_line": "Combined",
    },
    {
        "scenario_name": "DropSeq_MCF7_to_DropSeq_HCC1806",
        "tuning_dataset": "DropSeq_MCF7",
        "X_train": X_mcf_dropseq,
        "y_train": y_mcf_dropseq,
        "X_test": X_hcc_dropseq,
        "y_test": y_hcc_dropseq,
        "source_cell_line": "MCF7",
        "target_cell_line": "HCC1806",
    },
    {
        "scenario_name": "DropSeq_HCC1806_to_DropSeq_MCF7",
        "tuning_dataset": "DropSeq_HCC1806",
        "X_train": X_hcc_dropseq,
        "y_train": y_hcc_dropseq,
        "X_test": X_mcf_dropseq,
        "y_test": y_mcf_dropseq,
        "source_cell_line": "HCC1806",
        "target_cell_line": "MCF7",
    },
]


if RUN_DROPSEQ_FIVE_SCENARIOS and RUN_DROPSEQ_TUNING:
    for spec in dropseq_scenario_specs:
        scenario_name = spec["scenario_name"]
        tuning_dataset = spec["tuning_dataset"]

        if tuning_dataset not in all_best_estimators_dropseq:
            print(f"Skipping {scenario_name}: no tuned models available for {tuning_dataset}.")
            continue

        X_train_aligned, X_test_aligned, common_genes = align_train_test_features(
            spec["X_train"],
            spec["X_test"],
        )

        dropseq_scenario_fitted_models[scenario_name] = {}
        dropseq_scenario_predictions[scenario_name] = {}
        dropseq_scenario_scores[scenario_name] = {}

        for model_name, estimator in all_best_estimators_dropseq[tuning_dataset].items():
            print(f"DropSeq_normalised scenario evaluation | {scenario_name} | {model_name}")

            result, fitted_model, y_pred, y_score = evaluation_model(
                estimator,
                X_train_aligned,
                spec["y_train"],
                X_test_aligned,
                spec["y_test"],
                model_name=model_name,
                dataset_name=scenario_name,
            )

            result.update({
                "scenario": scenario_name,
                "tuning_dataset": tuning_dataset,
                "source_cell_line": spec["source_cell_line"],
                "target_cell_line": spec["target_cell_line"],
                "n_train_cells": X_train_aligned.shape[0],
                "n_test_cells": X_test_aligned.shape[0],
                "n_common_genes": len(common_genes),
            })

            dropseq_five_scenario_results.append(result)
            dropseq_scenario_fitted_models[scenario_name][model_name] = fitted_model
            dropseq_scenario_predictions[scenario_name][model_name] = y_pred
            dropseq_scenario_scores[scenario_name][model_name] = y_score

    dropseq_five_scenario_results = pd.DataFrame(dropseq_five_scenario_results)

    if not dropseq_five_scenario_results.empty:
        dropseq_five_scenario_results = dropseq_five_scenario_results.sort_values(
            ["scenario", "balanced_accuracy"],
            ascending=[True, False],
        ).reset_index(drop=True)

        dropseq_internal_scenarios = [
            "DropSeq_MCF7_to_DropSeq_MCF7",
            "DropSeq_HCC1806_to_DropSeq_HCC1806",
            "DropSeq_Combined_to_DropSeq_Combined",
        ]

        dropseq_cross_scenarios = [
            "DropSeq_MCF7_to_DropSeq_HCC1806",
            "DropSeq_HCC1806_to_DropSeq_MCF7",
        ]

        dropseq_tuned_holdout_results = dropseq_five_scenario_results[
            dropseq_five_scenario_results["scenario"].isin(dropseq_internal_scenarios)
        ].copy().reset_index(drop=True)

        dropseq_cross_cell_line_tuned_results = dropseq_five_scenario_results[
            dropseq_five_scenario_results["scenario"].isin(dropseq_cross_scenarios)
        ].copy().reset_index(drop=True)

        dropseq_best_tuned_model_by_scenario = (
            dropseq_five_scenario_results
            .sort_values(["scenario", "balanced_accuracy"], ascending=[True, False])
            .groupby("scenario", as_index=False)
            .head(1)
            .reset_index(drop=True)
        )

        print("All five DropSeq_normalised scenario results:")
        display(dropseq_five_scenario_results)

        print("Best tuned DropSeq_normalised model in each scenario:")
        display(dropseq_best_tuned_model_by_scenario)
    else:
        dropseq_tuned_holdout_results = pd.DataFrame()
        dropseq_cross_cell_line_tuned_results = pd.DataFrame()
        dropseq_best_tuned_model_by_scenario = pd.DataFrame()
        print("No DropSeq_normalised scenario results were produced.")
else:
    dropseq_five_scenario_results = pd.DataFrame()
    dropseq_tuned_holdout_results = pd.DataFrame()
    dropseq_cross_cell_line_tuned_results = pd.DataFrame()
    dropseq_best_tuned_model_by_scenario = pd.DataFrame()
    print("DropSeq_normalised scenario evaluation was not run.")

### Interpretation of the five-scenario evaluation

The five-scenario evaluation is one of the most important parts of the supervised learning analysis, because it measures not only whether hypoxia can be predicted within a single dataset, but also whether the learned hypoxia signal generalises across different biological backgrounds.

Overall, the results show three different behaviours for the three dataset versions.

First, the **SmartSeq_normalised** data provide the clearest and most stable supervised signal. In the within-cell-line settings, the best models reach perfect performance: `MCF7 → MCF7`, `HCC1806 → HCC1806` and `Combined → Combined` all obtain a balanced accuracy of `1.00`. More importantly, the cross-cell-line evaluations also remain very strong. When training on HCC1806 and testing on MCF7, the best model reaches a balanced accuracy of approximately `0.992`; when training on MCF7 and testing on HCC1806, the best model still reaches approximately `0.906`. This means that the professor-provided filtered and normalised SmartSeq data contain a strong hypoxia-related transcriptional signal that is not limited to one cell line only. The decrease in the `MCF7 → HCC1806` direction suggests that part of the hypoxia response is still cell-line-specific, but the overall performance remains high enough to support the presence of a shared hypoxia signature.

Second, the **SmartSeq_QCd** dataset, obtained using our own preprocessing pipeline, also preserves a very strong hypoxia signal. The within-cell-line and combined scenarios again reach a balanced accuracy of `1.00`, showing that our preprocessing is able to retain the main discriminative information between normoxic and hypoxic cells. However, the cross-cell-line evaluations are slightly weaker than in the normalised version. In particular, `HCC1806_QCd → MCF7_QCd` reaches a balanced accuracy of about `0.931`, while `MCF7_QCd → HCC1806_QCd` reaches about `0.887`. This suggests that our preprocessing pipeline is valid and biologically meaningful, but it does not clearly improve cross-cell-line generalisation compared with the professor-provided normalised matrices.

Third, the **DropSeq_normalised** results show a different pattern. Within-cell-line performance is still high, especially for `DropSeq_MCF7 → DropSeq_MCF7`, where the best model reaches a balanced accuracy of about `0.982`, and for `DropSeq_HCC1806 → DropSeq_HCC1806`, where the best model reaches about `0.956`. The combined DropSeq model also performs well, with a balanced accuracy of about `0.939`. Therefore, hypoxia is still predictable in DropSeq when training and testing are performed within the same cell line or within the same combined distribution. However, the cross-cell-line transfer is much weaker: `DropSeq_HCC1806 → DropSeq_MCF7` reaches about `0.733`, while `DropSeq_MCF7 → DropSeq_HCC1806` drops to about `0.576`. This indicates that, in DropSeq, the model does not learn a hypoxia signal that transfers robustly between MCF7 and HCC1806. The signal appears to be much more dependent on the specific cell line used for training.

This difference is biologically important. If a model trained on one cell line performs well on another cell line, this suggests that the two cell lines share a similar hypoxia-induced transcriptional programme. In contrast, poor cross-cell-line performance suggests that the gene-expression response to hypoxia is partly cell-line-specific, or that technical and biological differences between the two cell lines dominate over the shared hypoxia signal. The DropSeq results therefore suggest that, despite the larger number of cells, the transcriptional structure of MCF7 and HCC1806 remains substantially different when evaluated in a transfer-learning setting.

The comparison between linear and non-linear models also helps interpret these results. In smaller datasets such as SmartSeq, simpler linear models often perform extremely well. For example, Logistic Regression reaches perfect balanced accuracy in several SmartSeq_normalised and SmartSeq_QCd within-cell-line scenarios. This suggests that, when the dataset is smaller and cleaner, the hypoxia signal can often be separated by a relatively simple linear decision boundary. In this setting, more flexible non-linear models may not add much and can even risk overfitting, because they have enough flexibility to model noise or cell-line-specific patterns.

In contrast, in the larger DropSeq dataset, non-linear models become more useful. The best within-cell-line and combined DropSeq results are obtained by models such as MLP, SVM with RBF kernel. This is reasonable because DropSeq contains many more cells, giving complex models more data from which to learn stable non-linear patterns. For example, MLP obtains the best performance in the `DropSeq_Combined → DropSeq_Combined` scenario, with a balanced accuracy of about `0.939`, and also in the `DropSeq_HCC1806 → DropSeq_HCC1806` scenario, with a balanced accuracy of about `0.956`. However, the same flexibility does not automatically solve the cross-cell-line problem: even with many cells, the DropSeq transfer scenarios remain weak. This means that the limitation is not only statistical, but also biological and technological. The model can learn hypoxia well within each DropSeq distribution, but it struggles when the target cell line has a different expression structure.

In summary, the five-scenario evaluation supports three main conclusions. The normalised SmartSeq data provide the strongest and most transferable hypoxia signal. Our QCd preprocessing pipeline is valid and preserves high classification performance, but it is slightly less effective in cross-cell-line generalisation. DropSeq provides strong within-cell-line performance thanks to the larger number of cells, especially for non-linear models, but it shows the weakest transfer between MCF7 and HCC1806. Therefore, hypoxia is clearly predictable from single-cell gene expression, but the degree to which the learned signal generalises depends strongly on the dataset version, sequencing technology and biological distance between cell lines.

## Stacking ensemble

Stacking is an ensemble method that combines several base classifiers through a second-level model called a meta-model. Each base model first produces predictions or probability scores. These outputs are then used as inputs for the meta-model, which learns how to weight the base models.

In this notebook, the meta-model is **Logistic Regression**. This keeps the ensemble interpretable and reduces the risk of overfitting compared with a more complex meta-learner. The base models are selected from the best tuned models according to cross-validation balanced accuracy. Stacking is tested separately for SmartSeq_normalised, SmartSeq_QCd and DropSeq_normalised.

The key question is whether stacking improves over the best individual model. If the base models make very similar predictions, stacking may add little. If the base models capture complementary patterns, the meta-model can combine them to improve robustness.

### SmartSeq_normalised

The SmartSeq_normalised stacking model combines the best tuned classifiers for HCC1806, MCF7 and the combined dataset. This gives a compact ensemble built from the strongest individual models in the professor-processed data.

In [ ]:
# Build and evaluate SmartSeq_normalised stacking ensembles.
def build_stacking_from_best(
    dataset_name,
    top_n=3,
    tuning_summary_table=None,
    best_estimators_by_dataset=None,
):
    if tuning_summary_table is None:
        tuning_summary_table = all_tuning_summary

    if best_estimators_by_dataset is None:
        best_estimators_by_dataset = all_best_estimators

    dataset_summary = tuning_summary_table[tuning_summary_table["dataset"] == dataset_name].copy()
    dataset_summary = dataset_summary.sort_values("mean_test_balanced_accuracy", ascending=False)
    selected_names = dataset_summary["model"].head(top_n).tolist()

    estimators = []
    for name in selected_names:
        estimators.append((name, clone(best_estimators_by_dataset[dataset_name][name])))

    stack = StackingClassifier(
        estimators=estimators,
        final_estimator=LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        stack_method="auto",
        cv=3,
        n_jobs=-1,
    )

    return stack, selected_names


RUN_STACKING = True
STACKING_TOP_N = 3
stacking_results = []
stacking_models = {}

if RUN_STACKING and RUN_TUNING:
    for dataset_name in DATASETS_TO_TUNE:
        print(f"\nStacking on {dataset_name}")
        stack_model, selected_names = build_stacking_from_best(dataset_name, top_n=STACKING_TOP_N)
        stacking_models[dataset_name] = stack_model
        print("Base models:", selected_names)

        X_train = splits[dataset_name]["X_train"]
        y_train = splits[dataset_name]["y_train"]
        X_test = splits[dataset_name]["X_test"]
        y_test = splits[dataset_name]["y_test"]

        cv_row, _ = evaluation_model_cv(stack_model, X_train, y_train, dataset_name=dataset_name, model_name="Stacking")
        display(pd.DataFrame([cv_row]))
        holdout_result, fitted_stack, y_pred_stack, y_score_stack = evaluation_model(
            stack_model,
            X_train,
            y_train,
            X_test,
            y_test,
            model_name="Stacking",
            dataset_name=dataset_name,
        )
        holdout_result["base_models"] = selected_names
        stacking_results.append(holdout_result)

    stacking_results = pd.DataFrame(stacking_results).sort_values("balanced_accuracy", ascending=False)
    display(stacking_results)
else:
    stacking_results = pd.DataFrame()
    print("Stacking was not run.")

### SmartSeq_QCd

The QCd stacking model repeats the same approach using the best models trained with our preprocessing pipeline. This tests whether the custom preprocessing provides complementary information that can be exploited by an ensemble.

In [ ]:
# Build and evaluate SmartSeq_QCd stacking ensembles.

RUN_QCD_STACKING = True

qcd_stacking_results = []
qcd_stacking_models = {}

if RUN_QCD_STACKING and RUN_TUNING_2:
    for dataset_name in DATASETS_TO_TUNE_2:
        print(f"\nQCd stacking on {dataset_name}")

        if dataset_name not in all_best_estimators_2:
            print(f"Skipping {dataset_name}: no tuned models available.")
            continue

        stack_model, selected_names = build_stacking_from_best(
            dataset_name,
            top_n=STACKING_TOP_N,
            tuning_summary_table=all_tuning_summary_2,
            best_estimators_by_dataset=all_best_estimators_2,
        )

        qcd_stacking_models[dataset_name] = stack_model
        print("Base models:", selected_names)

        X_train = splits[dataset_name]["X_train"]
        y_train = splits[dataset_name]["y_train"]
        X_test = splits[dataset_name]["X_test"]
        y_test = splits[dataset_name]["y_test"]

        cv_row, _ = evaluation_model_cv(
            stack_model,
            X_train,
            y_train,
            dataset_name=dataset_name,
            model_name="Stacking_QCd",
        )

        display(pd.DataFrame([cv_row]))

        holdout_result, fitted_stack, y_pred_stack, y_score_stack = evaluation_model(
            stack_model,
            X_train,
            y_train,
            X_test,
            y_test,
            model_name="Stacking_QCd",
            dataset_name=dataset_name,
        )

        holdout_result["base_models"] = selected_names
        qcd_stacking_results.append(holdout_result)

        qcd_stacking_models[dataset_name] = fitted_stack

    qcd_stacking_results = pd.DataFrame(qcd_stacking_results)

    if not qcd_stacking_results.empty:
        qcd_stacking_results = qcd_stacking_results.sort_values(
            "balanced_accuracy",
            ascending=False,
        ).reset_index(drop=True)

        display(qcd_stacking_results)
    else:
        print("No QCd stacking results were produced.")
else:
    qcd_stacking_results = pd.DataFrame()
    print("QCd stacking was not run.")

### DropSeq_normalised

The DropSeq stacking model is built from the best tuned DropSeq classifiers. Since DropSeq contains more cells, this ensemble may be more stable than the SmartSeq stackers, but it still needs to be checked with the diagnostic analyses below.

In [ ]:
# Build and evaluate DropSeq_normalised stacking models using the same function used for SmartSeq.

RUN_DROPSEQ_STACKING = True

dropseq_stacking_results = []
dropseq_stacking_models = {}
dropseq_stacking_predictions = {}
dropseq_stacking_scores = {}

if RUN_DROPSEQ_STACKING and RUN_DROPSEQ_TUNING:
    for dataset_name in DATASETS_TO_TUNE_DROPSEQ:
        if dataset_name not in all_best_estimators_dropseq:
            print(f"Skipping DropSeq stacking for {dataset_name}: no tuned models available.")
            continue

        print(f"DropSeq_normalised stacking | {dataset_name}")

        stack_model, selected_base_models = build_stacking_from_best(
            dataset_name=dataset_name,
            top_n=STACKING_TOP_N,
            tuning_summary_table=all_tuning_summary_dropseq,
            best_estimators_by_dataset=all_best_estimators_dropseq,
        )

        X_train = splits[dataset_name]["X_train"]
        y_train = splits[dataset_name]["y_train"]
        X_test = splits[dataset_name]["X_test"]
        y_test = splits[dataset_name]["y_test"]

        result, fitted_stack, y_pred, y_score = evaluation_model(
            stack_model,
            X_train,
            y_train,
            X_test,
            y_test,
            model_name="Stacking",
            dataset_name=dataset_name,
        )

        result.update({
            "dataset_family": "DropSeq_normalised",
            "dataset": dataset_name,
            "base_models": ", ".join(selected_base_models),
            "n_base_models": len(selected_base_models),
        })

        dropseq_stacking_results.append(result)
        dropseq_stacking_models[dataset_name] = fitted_stack
        dropseq_stacking_predictions[dataset_name] = y_pred
        dropseq_stacking_scores[dataset_name] = y_score

    dropseq_stacking_results = pd.DataFrame(dropseq_stacking_results)

    if not dropseq_stacking_results.empty:
        dropseq_stacking_results = dropseq_stacking_results.sort_values(
            "balanced_accuracy",
            ascending=False,
        ).reset_index(drop=True)

        display(dropseq_stacking_results)
    else:
        print("No DropSeq stacking results were produced.")
else:
    dropseq_stacking_results = pd.DataFrame()
    print("DropSeq stacking was not run.")

## Stacking diagnostic

The stacking diagnostic checks whether the ensemble is actually learning something useful from the base models. Two plots are used for each dataset family.

First, the correlation heatmap compares the hypoxia scores produced by the base models. If the correlations are very high, the base models are making almost the same predictions, so stacking is unlikely to add much information. If the correlations are lower, the models may be complementary.

Second, the logistic meta-model coefficients show which base models the stacker uses most. A large positive coefficient means that the meta-model relies strongly on that base model. A coefficient close to zero suggests that the base model adds little after accounting for the others. In the previous SmartSeq diagnostics, the base models were highly correlated, which explains why stacking improved little or not at all: the ensemble had limited new information to learn.

In [ ]:
# Plot base-model score correlations for each stacking family.

STACKING_DIAGNOSTIC_CONFIGS = [
    {
        "label": "SmartSeq_normalised",
        "dataset": "Combined",
        "models": stacking_models,
        "available": RUN_STACKING and "Combined" in stacking_models,
    },
    {
        "label": "SmartSeq_QCd",
        "dataset": "Combined_QCd",
        "models": qcd_stacking_models,
        "available": RUN_QCD_STACKING and "Combined_QCd" in qcd_stacking_models,
    },
    {
        "label": "DropSeq_normalised",
        "dataset": "DropSeq_Combined",
        "models": dropseq_stacking_models if "dropseq_stacking_models" in globals() else {},
        "available": "dropseq_stacking_models" in globals() and "DropSeq_Combined" in dropseq_stacking_models,
    },
]

stacking_diag_fitted = {}
stacking_diag_correlations = {}

for config in STACKING_DIAGNOSTIC_CONFIGS:
    if not config["available"]:
        print(f"Skipping {config['label']}: fitted stacking model not available.")
        continue

    dataset_name = config["dataset"]
    X_train = splits[dataset_name]["X_train"]
    y_train = splits[dataset_name]["y_train"]
    X_test = splits[dataset_name]["X_test"]

    # SmartSeq_normalised stacking models are stored before fitting, while QCd and DropSeq may already be fitted.
    stack = clone(config["models"][dataset_name])
    stack.fit(X_train, y_train)
    stacking_diag_fitted[config["label"]] = stack

    base_scores = {}
    for model_name, fitted_model in stack.named_estimators_.items():
        if hasattr(fitted_model, "predict_proba"):
            base_scores[model_name] = fitted_model.predict_proba(X_test)[:, 1]
        elif hasattr(fitted_model, "decision_function"):
            base_scores[model_name] = fitted_model.decision_function(X_test)

    base_scores_df = pd.DataFrame(base_scores, index=X_test.index)
    corr = base_scores_df.corr()
    stacking_diag_correlations[config["label"]] = corr

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(corr, vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.index)))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right")
    ax.set_yticklabels(corr.index)

    for i in range(corr.shape[0]):
        for j in range(corr.shape[1]):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")

    plt.colorbar(im, ax=ax)
    ax.set_title(f"{config['label']} - base-model score correlation")
    plt.tight_layout()
    plt.show()

    display(corr)

In [ ]:
# Plot logistic meta-model coefficients for each stacking family.

stacking_diag_coefficients = {}

for label, stack in stacking_diag_fitted.items():
    meta_model = stack.final_estimator_

    if not hasattr(meta_model, "coef_"):
        print(f"Skipping {label}: the meta-model has no coefficients.")
        continue

    coefs = meta_model.coef_.ravel()

    try:
        names = stack.get_feature_names_out()
    except Exception:
        names = list(stack.named_estimators_.keys())

    if len(names) != len(coefs):
        names = [f"meta_feature_{i}" for i in range(len(coefs))]

    coef_df = pd.DataFrame({
        "meta_feature": names,
        "coefficient": coefs,
    }).sort_values("coefficient", ascending=False)

    stacking_diag_coefficients[label] = coef_df

    plt.figure(figsize=(7, 4))
    plt.barh(coef_df["meta_feature"], coef_df["coefficient"])
    plt.axvline(0, linestyle="--", linewidth=1)
    plt.xlabel("Coefficient")
    plt.title(f"{label} - stacking meta-model coefficients")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

    display(coef_df)

### Stacking interpretation

For the **SmartSeq_normalised** dataset, stacking does not provide a meaningful improvement over the best single models. This is consistent with the previous five-scenario evaluation, where several individual classifiers already reached extremely high or perfect performance. The stacking diagnostics confirm this interpretation: the predicted hypoxia scores of the base models are highly correlated, meaning that the classifiers are essentially learning the same decision rule. In this case, the logistic meta-model has very little new information to combine. Therefore, the lack of improvement is not surprising: when Logistic Regression, SVM and tree-based models already separate hypoxia and normoxia almost perfectly, the ensemble cannot substantially improve the result.

The situation is slightly different for the **SmartSeq_QCd** dataset. Here, stacking gives a small improvement in some scenarios. This suggests that the models trained on the QCd version are not completely redundant. The diagnostic correlation heatmap supports this idea: Random Forest shows a slightly lower correlation with the other models, indicating that it captures a somewhat different signal compared with the more linear classifiers. This makes the ensemble more useful, because the meta-model can exploit this partial complementarity between models. The improvement is still limited, but it suggests that the QCd preprocessing creates a setting where combining different model families can provide a small advantage.

For **DropSeq_normalised**, stacking is slightly more useful than in the SmartSeq_normalised case. The best individual combined DropSeq model, SVM RBF, reaches a balanced accuracy of about `0.937`, while the stacking model improves it slightly to about `0.949`. This suggests that the ensemble adds some useful information, although the improvement is limited.

The diagnostics explain this result. SVM RBF and MLP are highly correlated, with a correlation of about `0.987`, so they produce very similar hypoxia scores. Random Forest, however, is less correlated with both of them, around `0.893–0.895`, meaning that it captures a somewhat different structure in the data. This is confirmed by the logistic meta-model coefficients: Random Forest receives the largest weight, approximately `8.80`, compared with about `2.34` for MLP and `2.00` for SVM RBF. Therefore, in DropSeq the stacking model mainly benefits from the complementary contribution of Random Forest.

Overall, the DropSeq stacking results suggest that ensemble learning is more meaningful in the larger DropSeq dataset than in the SmartSeq_normalised one. The larger number of cells allows more complex models to learn stable patterns, and the lower correlation of Random Forest with the other base models gives the stacker something useful to combine. However, this improvement mainly concerns within-distribution prediction, especially the combined DropSeq hold-out setting. It does not solve the biological limitation observed in the five-scenario evaluation, where cross-cell-line transfer in DropSeq remains weak. Therefore, stacking improves the technical predictive performance of DropSeq models, but it does not fully recover a shared hypoxia signal between MCF7 and HCC1806.

## Diagnostic plots for selected combined models

This diagnostic plots are generated for the best tuned **combined** model in each dataset family: SmartSeq_normalised, SmartSeq_QCd and DropSeq_normalised. The combined setting is selected because it represents the most general model within each technology or preprocessing strategy.

Before plotting, the notebook selects the best combined model according to cross-validation balanced accuracy, fits it on the internal training split and evaluates it on the internal test split. The same plots are then produced in the fixed order SmartSeq_normalised, SmartSeq_QCd and DropSeq_normalised.

In [ ]:
# Define diagnostic plotting functions.
def plot_cv_summary(results_df, dataset_name=None, metric="cv_balanced_accuracy_mean"):
    df = results_df.copy()
    if dataset_name is not None:
        df = df[df["dataset"] == dataset_name]
    if df.empty:
        print("No results to plot.")
        return

    df = df.sort_values(metric, ascending=False)
    err_col = metric.replace("_mean", "_std")

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(df))
    y = df[metric].values
    yerr = df[err_col].values if err_col in df.columns else None
    ax.bar(x, y, yerr=yerr, capsize=4)
    ax.set_xticks(x)
    ax.set_xticklabels(df["model"], rotation=45, ha="right")
    ax.set_ylabel(metric)
    title = "Cross-validation model comparison"
    if dataset_name is not None:
        title += f" - {dataset_name}"
    ax.set_title(title)
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.show()


def plot_confusion(y_true, y_pred, title="Confusion matrix"):
    labels = [0, 1]
    display_labels = [INV_LABEL_MAP[0], INV_LABEL_MAP[1]]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=display_labels)
    disp.plot(values_format="d")
    plt.title(title)
    plt.show()


def plot_roc(y_true, y_score, title="ROC curve"):
    if y_score is None:
        print("No probability or decision score available for ROC curve.")
        return
    RocCurveDisplay.from_predictions(y_true, y_score)
    plt.title(title)
    plt.show()


def plot_probability_boxplot(y_true, y_score, title="Predicted probability distribution"):
    if y_score is None:
        print("No probability score available.")
        return
    y_true = np.asarray(y_true)
    norm_scores = y_score[y_true == 0]
    hypo_scores = y_score[y_true == 1]

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.boxplot([norm_scores, hypo_scores], labels=["True Normoxia", "True Hypoxia"])
    ax.set_ylabel("Predicted probability / score for Hypoxia")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def plot_pca_prediction_errors(X_train, X_test, y_test, y_pred, title="PCA projection of prediction errors"):
    scaler = StandardScaler()
    pca = PCA(n_components=2, random_state=RANDOM_STATE)

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    pca.fit(X_train_scaled)
    coords = pca.transform(X_test_scaled)

    y_test = np.asarray(y_test)
    y_pred = np.asarray(y_pred)
    correct = y_test == y_pred

    fig, ax = plt.subplots(figsize=(7, 6))
    for cls in [0, 1]:
        idx_correct = (y_test == cls) & correct
        idx_wrong = (y_test == cls) & (~correct)
        ax.scatter(
            coords[idx_correct, 0],
            coords[idx_correct, 1],
            marker="o",
            label=f"{INV_LABEL_MAP[cls]} correct",
            alpha=0.8,
        )
        ax.scatter(
            coords[idx_wrong, 0],
            coords[idx_wrong, 1],
            marker="x",
            s=90,
            label=f"{INV_LABEL_MAP[cls]} wrong",
            alpha=0.9,
        )

    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_learning_curve_model(model, X, y, title="Learning curve"):
    train_sizes, train_scores, val_scores = learning_curve(
        model,
        X,
        y,
        cv=stratified_cv,
        scoring="balanced_accuracy",
        n_jobs=-1,
        train_sizes=np.linspace(0.3, 1.0, 5),
    )

    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(train_sizes, train_mean, marker="o", label="Train")
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2)
    ax.plot(train_sizes, val_mean, marker="o", label="Validation")
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.2)
    ax.set_xlabel("Training set size")
    ax.set_ylabel("Balanced accuracy")
    ax.set_title(title)
    ax.set_ylim(0, 1.05)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Select the best combined tuned model for each dataset family and compute hold-out predictions.

DIAGNOSTIC_TARGETS = [
    {
        "label": "SmartSeq_normalised",
        "dataset": "Combined",
        "summary": all_tuning_summary,
        "estimators": all_best_estimators,
        "available": RUN_TUNING and "Combined" in all_best_estimators,
    },
    {
        "label": "SmartSeq_QCd",
        "dataset": "Combined_QCd",
        "summary": all_tuning_summary_2,
        "estimators": all_best_estimators_2,
        "available": RUN_TUNING_2 and "Combined_QCd" in all_best_estimators_2,
    },
    {
        "label": "DropSeq_normalised",
        "dataset": "DropSeq_Combined",
        "summary": all_tuning_summary_dropseq if "all_tuning_summary_dropseq" in globals() else pd.DataFrame(),
        "estimators": all_best_estimators_dropseq if "all_best_estimators_dropseq" in globals() else {},
        "available": "all_best_estimators_dropseq" in globals() and "DropSeq_Combined" in all_best_estimators_dropseq,
    },
]

diagnostic_results = {}

for target in DIAGNOSTIC_TARGETS:
    if not target["available"]:
        print(f"Skipping {target['label']}: tuned combined model not available.")
        continue

    dataset_name = target["dataset"]
    summary = target["summary"]
    best_row = summary[summary["dataset"] == dataset_name].sort_values(
        "mean_test_balanced_accuracy",
        ascending=False,
    ).iloc[0]

    model_name = best_row["model"]
    model = target["estimators"][dataset_name][model_name]

    X_train = splits[dataset_name]["X_train"]
    y_train = splits[dataset_name]["y_train"]
    X_test = splits[dataset_name]["X_test"]
    y_test = splits[dataset_name]["y_test"]

    result, fitted_model, y_pred, y_score = evaluation_model(
        model,
        X_train,
        y_train,
        X_test,
        y_test,
        model_name=model_name,
        dataset_name=dataset_name,
    )

    diagnostic_results[target["label"]] = {
        "dataset": dataset_name,
        "model_name": model_name,
        "summary": summary,
        "result": result,
        "model": fitted_model,
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "y_pred": y_pred,
        "y_score": y_score,
    }

    print(f"{target['label']} | selected model: {model_name}")
    display(pd.DataFrame([result]))

### Cross-validation score comparison

This plot is used to compare the tuned models within each combined dataset using cross-validation balanced accuracy. It is useful for seeing whether one model clearly dominates or whether several models perform similarly. When many models have nearly identical scores, model choice should also consider simplicity, stability and interpretability.

In [ ]:
# Plot cross-validation balanced accuracy for the three selected combined settings.

for label, item in diagnostic_results.items():
    plot_cv_summary(
        item["summary"].rename(columns={
            "mean_test_balanced_accuracy": "cv_balanced_accuracy_mean",
            "std_test_balanced_accuracy": "cv_balanced_accuracy_std",
        }),
        dataset_name=item["dataset"],
        metric="cv_balanced_accuracy_mean",
    )

### Confusion matrix

The confusion matrix shows the number of correctly and incorrectly classified cells in the internal test split. It makes errors more interpretable than a single score: false negatives correspond to hypoxic cells predicted as normoxic, while false positives correspond to normoxic cells predicted as hypoxic.

In [ ]:
# Plot confusion matrices for the selected combined models.

for label, item in diagnostic_results.items():
    plot_confusion(
        item["y_test"],
        item["y_pred"],
        title=f"{label} - {item['model_name']} confusion matrix",
    )

### ROC curve

The ROC curve evaluates the ability of the model to rank hypoxic cells above normoxic cells across all possible thresholds. A curve close to the upper-left corner indicates strong separation between the two classes.

In [ ]:
# Plot ROC curves for the selected combined models.

for label, item in diagnostic_results.items():
    plot_roc(
        item["y_test"],
        item["y_score"],
        title=f"{label} - {item['model_name']} ROC curve",
    )

### Predicted hypoxia-score distribution

The probability boxplot compares the predicted hypoxia scores for true normoxic and true hypoxic cells. A good classifier should assign low hypoxia scores to normoxic cells and high hypoxia scores to hypoxic cells, with little overlap between the two distributions.

In [ ]:
# Plot probability distributions for Normoxia and Hypoxia.

for label, item in diagnostic_results.items():
    plot_probability_boxplot(
        item["y_test"],
        item["y_score"],
        title=f"{label} - {item['model_name']} hypoxia score distribution",
    )

### PCA projection of prediction errors

This plot projects the internal test cells onto the first two principal components fitted on the training split. Correctly and incorrectly classified cells are shown with different markers. If errors cluster in a specific region of the PCA space, this may indicate ambiguous cells, batch effects or biological subpopulations that are harder to classify.

In [ ]:
# Plot PCA projections of prediction errors for the selected combined models.

for label, item in diagnostic_results.items():
    plot_pca_prediction_errors(
        item["X_train"],
        item["X_test"],
        item["y_test"],
        item["y_pred"],
        title=f"{label} - PCA projection of prediction errors",
    )

### Interpretation of the diagnostic plots for the combined models

The diagnostic plots for the combined models confirm the main results obtained in the evaluation tables. For the SmartSeq-based datasets, the predictions are generally very confident and the confusion matrices show very few errors, meaning that the models separate normoxic and hypoxic cells clearly. The PCA error plots also suggest that the few misclassified cells tend to lie in more ambiguous regions of the expression space, rather than being random mistakes. In DropSeq, the model still performs well in the combined setting, but the probability distributions are slightly less clean, suggesting that the classification problem is more complex and that the boundary between hypoxia and normoxia is less sharp than in SmartSeq. This gives the reasons to why non-linear classifiers perform better.

## Cross-technology transfer

As an exploratory analysis, the best model tuned on the SmartSeq_normalised combined dataset is trained on SmartSeq and tested on DropSeq, and the best DropSeq combined model is trained on DropSeq and tested on SmartSeq. This is not a standard final evaluation, but a curiosity analysis about technology transfer.

The biological question is whether the hypoxia signal is strong enough to survive a change in sequencing technology. A strong performance in both directions would suggest that the learned signature is robust across protocols. A performance drop would not necessarily mean that the biology is different; it could also reflect technical differences between SmartSeq and DropSeq, such as read depth, dropout, gene detection rate and normalisation effects.

In [ ]:
# Cross-technology transfer: SmartSeq combined <-> DropSeq combined

RUN_CROSS_TECHNOLOGY_TRANSFER = True
cross_technology_results = []


def choose_best_model_from_summary(summary_table, estimators_by_dataset, dataset_name):
    rows = summary_table[summary_table["dataset"] == dataset_name].copy()
    if rows.empty:
        raise ValueError(f"No tuning summary available for {dataset_name}.")

    best_model_name = rows.sort_values(
        "mean_test_balanced_accuracy",
        ascending=False,
    ).iloc[0]["model"]

    return best_model_name, estimators_by_dataset[dataset_name][best_model_name]


if RUN_CROSS_TECHNOLOGY_TRANSFER and RUN_TUNING and RUN_DROPSEQ_TUNING:
    smartseq_combined_best_name, smartseq_combined_best_model = choose_best_model_from_summary(
        all_tuning_summary,
        all_best_estimators,
        "Combined",
    )

    dropseq_combined_best_name, dropseq_combined_best_model = choose_best_model_from_summary(
        all_tuning_summary_dropseq,
        all_best_estimators_dropseq,
        "DropSeq_Combined",
    )

    cross_technology_specs = [
        {
            "scenario_name": "SmartSeq_Combined_to_DropSeq_Combined",
            "model_name": smartseq_combined_best_name,
            "estimator": smartseq_combined_best_model,
            "X_train": X_combined,
            "y_train": y_combined,
            "X_test": X_dropseq_combined,
            "y_test": y_dropseq_combined,
            "source_technology": "SmartSeq_normalised",
            "target_technology": "DropSeq_normalised",
        },
        {
            "scenario_name": "DropSeq_Combined_to_SmartSeq_Combined",
            "model_name": dropseq_combined_best_name,
            "estimator": dropseq_combined_best_model,
            "X_train": X_dropseq_combined,
            "y_train": y_dropseq_combined,
            "X_test": X_combined,
            "y_test": y_combined,
            "source_technology": "DropSeq_normalised",
            "target_technology": "SmartSeq_normalised",
        },
    ]

    for spec in cross_technology_specs:
        X_train_aligned, X_test_aligned, common_genes = align_train_test_features(
            spec["X_train"],
            spec["X_test"],
        )

        print(f"Cross-technology evaluation | {spec['scenario_name']} | {spec['model_name']}")

        result, fitted_model, y_pred, y_score = evaluation_model(
            spec["estimator"],
            X_train_aligned,
            spec["y_train"],
            X_test_aligned,
            spec["y_test"],
            model_name=spec["model_name"],
            dataset_name=spec["scenario_name"],
        )

        result.update({
            "scenario": spec["scenario_name"],
            "source_technology": spec["source_technology"],
            "target_technology": spec["target_technology"],
            "n_train_cells": X_train_aligned.shape[0],
            "n_test_cells": X_test_aligned.shape[0],
            "n_common_genes": len(common_genes),
        })

        cross_technology_results.append(result)

    cross_technology_results = pd.DataFrame(cross_technology_results)
    display(cross_technology_results)
else:
    cross_technology_results = pd.DataFrame()
    print("Cross-technology transfer was not run.")

### Interpretation of the cross-technology transfer

The cross-technology transfer experiment shows that models trained on one sequencing technology do not generalise well to the other. In practice, the performance is close to what we would expect from a very weak baseline such as a DummyClassifier. This does not necessarily mean that hypoxia is absent from one dataset, but rather that SmartSeq and DropSeq have very different technical structures, expression distributions and sources of noise. As a result, the model learns patterns that are specific to the technology used for training and cannot directly transfer them to the other technology. This confirms that sequencing technology is a major source of domain shift and that cross-technology prediction would require a dedicated integration or batch-correction strategy.

## Pathway analysis

The supervised learning results show that hypoxia can be predicted from gene-expression profiles, but predictive performance alone does not tell us whether the model is using biologically meaningful genes. For this reason, the most interpretable supervised models are used to extract ranked gene lists and test whether they recover known hypoxia biology.

### Relationship with the unsupervised NMF pathway analysis

The NMF pathway analysis and the supervised pathway analysis answer related but different questions. NMF identifies co-expression programs that emerge without using the condition labels, so it is useful for discovering latent biological structure. Logistic Regression feature interpretation, instead, identifies genes that are directly useful for predicting hypoxia. If both analyses point to similar pathways, the biological interpretation is stronger because the signal is both spontaneously present in the data and predictive of the label. If they differ, the difference is still informative: it may indicate that some genes are good classifiers without belonging to the dominant co-expression program, or that NMF is capturing broader cell-line-specific biology around the hypoxic response.

### Extracting top genes from Logistic Regression

We start from the combined datasets because they are the most appropriate setting for identifying a general hypoxia signal across both MCF7 and HCC1806. For each dataset family, we select the Logistic Regression model and refit it on the corresponding internal training split. This keeps the interpretation consistent with the rest of the supervised workflow and avoids using the anonymous test set.

For a binary Logistic Regression model, the predicted probability of Hypoxia is modelled as:

$$
\log\left(\frac{P(\mathrm{Hypoxia}\mid x)}{1-P(\mathrm{Hypoxia}\mid x)}\right)
= \beta_0 + \sum_{j=1}^{p} \beta_j x_j.
$$

A large positive coefficient means that higher expression of gene pushes the model towards Hypoxia, while a large negative coefficient pushes it towards Normoxia. Therefore, the coefficient ranking gives us a direct and interpretable gene-level signature.

In [ ]:
# Define helper functions for Logistic Regression gene-level interpretation.

def get_logistic_estimator(estimator_dict, dataset_name):
    """Return the Logistic Regression estimator for one dataset."""
    if dataset_name not in estimator_dict:
        raise KeyError(f"Dataset {dataset_name} not found in estimator dictionary.")
    if "LogisticRegression" not in estimator_dict[dataset_name]:
        raise KeyError(f"LogisticRegression not found for dataset {dataset_name}.")
    return estimator_dict[dataset_name]["LogisticRegression"]


def extract_logreg_gene_coefficients(label, dataset_name, estimator_dict, X_train, y_train):
    """Fit a Logistic Regression pipeline and return gene coefficients."""
    estimator = clone(get_logistic_estimator(estimator_dict, dataset_name))
    estimator.fit(X_train, y_train)

    final_model = estimator.named_steps["model"]
    coefficients = final_model.coef_.ravel()

    if "selector" in estimator.named_steps:
        selector = estimator.named_steps["selector"]
        selected_genes = np.asarray(X_train.columns)[selector.get_support()]
        gene_coefficients = coefficients

    elif "Preprocess" in estimator.named_steps:
        preprocess = estimator.named_steps["Preprocess"]
        gene_branch = dict(preprocess.transformer_list)["genes"]
        gene_selector = gene_branch.named_steps["select_gene_columns"]
        kbest_selector = gene_branch.named_steps["select_k_best_genes"]
        all_gene_columns = np.asarray(gene_selector.gene_columns_)
        selected_genes = all_gene_columns[kbest_selector.get_support()]

        # In the QCd pipeline the FeatureUnion concatenates selected genes first,
        # followed by engineered features. For the pathway analysis we keep only
        # the gene coefficients.
        gene_coefficients = coefficients[:len(selected_genes)]

    else:
        raise ValueError("Unsupported Logistic Regression pipeline structure.")

    coef_df = pd.DataFrame({
        "dataset_family": label,
        "dataset": dataset_name,
        "gene": selected_genes,
        "coefficient": gene_coefficients,
        "abs_coefficient": np.abs(gene_coefficients),
    })

    coef_df["direction"] = np.where(
        coef_df["coefficient"] >= 0,
        "Hypoxia-associated",
        "Normoxia-associated",
    )

    coef_df = coef_df.sort_values("coefficient", ascending=False).reset_index(drop=True)
    coef_df["rank_positive"] = np.arange(1, len(coef_df) + 1)
    coef_df["rank_absolute"] = coef_df["abs_coefficient"].rank(
        ascending=False,
        method="min",
    ).astype(int)

    return coef_df, estimator


PATHWAY_MODEL_SPECS = {
    "SmartSeq_normalised": {
        "dataset_name": "Combined",
        "estimators": all_best_estimators,
        "X_train": splits["Combined"]["X_train"],
        "y_train": splits["Combined"]["y_train"],
    },
    "SmartSeq_QCd": {
        "dataset_name": "Combined_QCd",
        "estimators": all_best_estimators_2,
        "X_train": splits["Combined_QCd"]["X_train"],
        "y_train": splits["Combined_QCd"]["y_train"],
    },
    "DropSeq_normalised": {
        "dataset_name": "DropSeq_Combined",
        "estimators": all_best_estimators_dropseq,
        "X_train": splits["DropSeq_Combined"]["X_train"],
        "y_train": splits["DropSeq_Combined"]["y_train"],
    },
}

pathway_gene_coefficients = {}
pathway_fitted_logreg_models = {}

#### SmartSeq_normalised Combined

We first interpret the Logistic Regression model trained on the SmartSeq_normalised combined dataset. This is the main reference dataset because it showed the strongest and most transferable hypoxia signal in the previous supervised evaluations. The table below reports the genes with the largest positive coefficients, which are the genes most strongly associated with Hypoxia according to the model.

In [ ]:
# Extract top Logistic Regression genes for SmartSeq_normalised Combined.
label = "SmartSeq_normalised"
spec = PATHWAY_MODEL_SPECS[label]

coef_df, fitted_model = extract_logreg_gene_coefficients(
    label=label,
    dataset_name=spec["dataset_name"],
    estimator_dict=spec["estimators"],
    X_train=spec["X_train"],
    y_train=spec["y_train"],
)

pathway_gene_coefficients[label] = coef_df
pathway_fitted_logreg_models[label] = fitted_model

print(f"{label}: {coef_df.shape[0]} selected genes interpreted.")
print("Top Hypoxia-associated genes:")
display(coef_df.head(20))

print("Top Normoxia-associated genes:")
display(coef_df.tail(20).sort_values("coefficient"))

#### SmartSeq_QCd Combined

We then repeat the same interpretation on the SmartSeq_QCd combined dataset. This allows us to check whether our own preprocessing pipeline preserves a similar biological signal. The QCd pipeline also includes engineered cell-level features, but for pathway analysis we focus only on the gene-expression branch and extract the genes selected by `SelectKBest` before the Logistic Regression model.

In [ ]:
# Extract top Logistic Regression genes for SmartSeq_QCd Combined.
label = "SmartSeq_QCd"
spec = PATHWAY_MODEL_SPECS[label]

coef_df, fitted_model = extract_logreg_gene_coefficients(
    label=label,
    dataset_name=spec["dataset_name"],
    estimator_dict=spec["estimators"],
    X_train=spec["X_train"],
    y_train=spec["y_train"],
)

pathway_gene_coefficients[label] = coef_df
pathway_fitted_logreg_models[label] = fitted_model

print(f"{label}: {coef_df.shape[0]} selected genes interpreted.")
print("Top Hypoxia-associated genes:")
display(coef_df.head(20))

print("Top Normoxia-associated genes:")
display(coef_df.tail(20).sort_values("coefficient"))

#### DropSeq_normalised Combined

Finally, we interpret the Logistic Regression model trained on the DropSeq_normalised combined dataset. DropSeq contains many more cells and was generated with a different sequencing technology. Therefore, this comparison helps us understand whether the genes associated with hypoxia are stable across technologies, or whether DropSeq relies on a partially different expression structure.

In [ ]:
# Extract top Logistic Regression genes for DropSeq_normalised Combined.
label = "DropSeq_normalised"
spec = PATHWAY_MODEL_SPECS[label]

coef_df, fitted_model = extract_logreg_gene_coefficients(
    label=label,
    dataset_name=spec["dataset_name"],
    estimator_dict=spec["estimators"],
    X_train=spec["X_train"],
    y_train=spec["y_train"],
)

pathway_gene_coefficients[label] = coef_df
pathway_fitted_logreg_models[label] = fitted_model

print(f"{label}: {coef_df.shape[0]} selected genes interpreted.")
print("Top Hypoxia-associated genes:")
display(coef_df.head(20))

print("Top Normoxia-associated genes:")
display(coef_df.tail(20).sort_values("coefficient"))

### Comparison of the extracted top genes across the three datasets

The Logistic Regression top genes show that the three dataset versions capture the hypoxia signal with different levels of biological clarity. **SmartSeq_normalised** provides a very coherent gene-level signature: many of the strongest positive coefficients correspond to well-known hypoxia-related genes such as `FAM162A`, `PGK1`, `DDIT4`, `EGLN3`, `P4HA1`, `ANGPTL4`, `BNIP3L`, `CA9`, `BNIP3`, `SLC2A1`, `ALDOA` and `LDHA`. This suggests that the model learns a clean and biologically meaningful hypoxia programme.

**SmartSeq_QCd** shows a very similar pattern, and in some cases even more explicitly centered on canonical hypoxia markers. Genes such as `CA9`, `ANGPTL4`, `P4HA1`, `EGLN3`, `PGAM1`, `DDIT4`, `FAM162A`, `PGK1` and `NDRG1` appear among the most positive coefficients. This is important because it confirms that our custom preprocessing pipeline preserves the same core biological signal found in the professor-normalised SmartSeq data.

The **DropSeq_normalised** top genes are different. Some genes, such as `PGK1`, `ENO1`, `GAPDH`, `TPI1` and `PKM`, are still compatible with hypoxia-related metabolic adaptation, especially glycolysis. However, the ranking also includes genes that are less directly canonical for hypoxia, suggesting that the DropSeq model relies on a broader and more technology- or cell-state-dependent expression structure. Therefore, at the single-gene level, SmartSeq_normalised and SmartSeq_QCd show a cleaner hypoxia signature, while DropSeq appears less specific but still retains a metabolic component consistent with hypoxic adaptation.

### Overlap with known hypoxia and HIF target genes

After extracting the top Logistic Regression genes, we compare them with a curated list of known hypoxia-related and HIF target genes. This is a direct biological sanity check. The list includes genes involved in glycolysis, angiogenesis, pH regulation, survival/apoptosis, erythropoiesis, invasion, and cell-cycle adaptation under low oxygen. If these genes appear among the most important model coefficients, it supports the idea that the classifier is learning a real hypoxia-related signal.

In [ ]:
# Compare model-selected genes with known hypoxia and HIF target genes.
known_hypoxia_genes = {
    "SLC2A1", "SLC2A3", "HK1", "HK2", "PFKL", "ALDOA",
    "PGAM1", "ENO1", "LDHA", "PDK1",
    "VEGFA", "VEGFC", "ANGPT2", "FLT1", "NRP1",
    "CA9", "CA12", "SLC9A1",
    "BNIP3", "BNIP3L", "BCL2L1",
    "EPO", "TFRC",
    "LOX", "MMP2", "CXCR4",
    "CDKN1A", "CCND1",
}


def compute_hypoxia_gene_overlap(coef_df, top_n=100):
    """Compute overlap between top model genes and known hypoxia genes."""
    top_abs_genes = set(
        coef_df.sort_values("abs_coefficient", ascending=False)
        .head(top_n)["gene"]
    )

    top_positive_genes = set(
        coef_df.sort_values("coefficient", ascending=False)
        .head(top_n)["gene"]
    )

    overlap_abs = sorted(top_abs_genes.intersection(known_hypoxia_genes))
    overlap_positive = sorted(top_positive_genes.intersection(known_hypoxia_genes))

    return {
        "top_n": top_n,
        "n_overlap_abs": len(overlap_abs),
        "overlap_abs": ", ".join(overlap_abs),
        "n_overlap_hypoxia_direction": len(overlap_positive),
        "overlap_hypoxia_direction": ", ".join(overlap_positive),
    }


overlap_rows = []
hypoxia_known_gene_rankings = {}

for label, coef_df in pathway_gene_coefficients.items():
    overlap_info = compute_hypoxia_gene_overlap(coef_df, top_n=100)
    overlap_info["dataset_family"] = label
    overlap_rows.append(overlap_info)

    known_rank_df = coef_df[coef_df["gene"].isin(known_hypoxia_genes)].copy()
    known_rank_df = known_rank_df.sort_values("abs_coefficient", ascending=False)
    hypoxia_known_gene_rankings[label] = known_rank_df

hypoxia_overlap_summary = pd.DataFrame(overlap_rows)[[
    "dataset_family",
    "top_n",
    "n_overlap_abs",
    "overlap_abs",
    "n_overlap_hypoxia_direction",
    "overlap_hypoxia_direction",
]]

display(hypoxia_overlap_summary)

for label, known_rank_df in hypoxia_known_gene_rankings.items():
    print(f"\nKnown hypoxia/HIF genes present in {label} model coefficients:")
    display(known_rank_df[["gene", "coefficient", "abs_coefficient", "direction", "rank_absolute"]].head(20))

### Interpretation of the overlap with known hypoxia genes

The overlap analysis confirms that the genes selected by the Logistic Regression models are biologically meaningful, especially in the two SmartSeq datasets. Here we compared the model-derived top genes with a curated list of known hypoxia and HIF-related genes. Two overlaps were considered: the overlap with the most important genes by absolute coefficient, and the overlap with the genes that have positive coefficients, therefore directly pushing the model toward the Hypoxia class.

The strongest results are observed for **SmartSeq_normalised** and **SmartSeq_QCd**. In both cases, several known hypoxia genes are recovered among the most relevant model features, including genes related to glycolysis, oxygen response and HIF-regulated adaptation. In SmartSeq_normalised, genes such as `BNIP3L`, `CA9`, `BNIP3`, `SLC2A1`, `ALDOA`, `LDHA`, `HK2` and `VEGFA` appear in the overlap. In SmartSeq_QCd, the overlap includes genes such as `CA9`, `PGAM1`, `ENO1`, `SLC2A1`, `BNIP3L`, `BNIP3`, `LDHA`, `ALDOA`, `PDK1` and `HK2`. This supports the conclusion that both SmartSeq representations recover a canonical hypoxia programme.

The distinction between absolute overlap and Hypoxia-direction overlap is important. The absolute overlap tells us whether known hypoxia genes are important for the model in general, regardless of whether they push the prediction toward Hypoxia or Normoxia. The Hypoxia-direction overlap is more biologically informative, because it checks whether these known hypoxia genes contribute in the expected direction. The fact that many known hypoxia genes appear among the positive coefficients in SmartSeq_normalised and SmartSeq_QCd strengthens the interpretation that the models are learning a real hypoxia-related signal.

For **DropSeq_normalised**, the overlap with the restricted list of known hypoxia genes is much weaker, with mainly `ENO1` appearing among the selected genes. This does not mean that DropSeq lacks a hypoxia signal; rather, it suggests that the DropSeq model relies less on the small set of canonical HIF-related markers and more on broader transcriptional patterns. This interpretation is consistent with the following ORA and GSEA analyses, where DropSeq still shows pathway-level enrichment for hypoxia despite the weaker single-gene overlap.

### Over-Representation Analysis on top model genes

Over-Representation Analysis (ORA) tests whether a selected list of genes is enriched in known biological gene sets. Here, the input list is composed of the top Hypoxia-associated genes according to the Logistic Regression coefficients. For each dataset family, we test whether these genes are over-represented in curated databases such as MSigDB Hallmark, GO Biological Process, KEGG, and Reactome.

Conceptually, ORA asks whether the overlap between the model-selected genes and a pathway is larger than expected by chance. This is usually tested with a hypergeometric or Fisher exact test. If \(N\) is the number of measured genes, \(K\) the number of genes in a pathway, \(n\) the number of selected model genes, and \(k\) the observed overlap, the enrichment probability can be written as:
$$
P(X \geq k) = \sum_{i=k}^{\min(K,n)} \frac{\binom{K}{i}\binom{N-K}{n-i}}{\binom{N}{n}}.
$$

ORA is easy to interpret, but it depends on the arbitrary choice of how many top genes to include. For this reason, we use it as a first pathway-level summary and complement it with GSEA in the following section.

In [ ]:
# Run ORA with GSEApy/Enrichr on top Hypoxia-associated genes.

ORA_TOP_N_GENES = 150
ORA_GENE_SETS = [
    "MSigDB_Hallmark_2020",
    "GO_Biological_Process_2023",
    "KEGG_2021_Human",
    "Reactome_2022",
]

PATHWAY_KEYWORDS = [
    "HYPOXIA", "HIF", "OXYGEN", "GLYCOLYSIS", "ANGIOGENESIS",
    "VEGF", "LACTATE", "APOPTOSIS", "PH", "CARBONIC",
]


def run_ora_for_dataset(label, coef_df, top_n=ORA_TOP_N_GENES):
    """Run ORA on the top positive Logistic Regression genes."""
    top_genes = (
        coef_df.sort_values("coefficient", ascending=False)
        .head(top_n)["gene"]
        .astype(str)
        .tolist()
    )

    enr = gp.enrichr(
        gene_list=top_genes,
        gene_sets=ORA_GENE_SETS,
        organism="human",
        outdir=None,
        verbose=False,
    )

    results = enr.results.copy()
    results.insert(0, "dataset_family", label)
    results["input_top_n"] = top_n
    return results


def filter_relevant_pathway_terms(results_df):
    """Return rows containing hypoxia-related pathway keywords."""
    term_col = "Term"
    mask = results_df[term_col].astype(str).str.upper().apply(
        lambda term: any(keyword in term for keyword in PATHWAY_KEYWORDS)
    )
    return results_df[mask].copy()


ora_results_by_dataset = {}
ora_relevant_terms_by_dataset = {}

for label, coef_df in pathway_gene_coefficients.items():
    print(f"Running ORA for {label}...")
    ora_results = run_ora_for_dataset(label, coef_df)
    ora_results_by_dataset[label] = ora_results
    ora_relevant_terms_by_dataset[label] = filter_relevant_pathway_terms(ora_results)

    print(f"Top enriched terms for {label}:")
    display(
        ora_results.sort_values("Adjusted P-value")
        .head(15)[["dataset_family", "Gene_set", "Term", "Adjusted P-value", "Overlap", "Combined Score"]]
    )

    print(f"Hypoxia-related enriched terms for {label}:")
    display(
        ora_relevant_terms_by_dataset[label]
        .sort_values("Adjusted P-value")
        .head(15)[["dataset_family", "Gene_set", "Term", "Adjusted P-value", "Overlap", "Combined Score"]]
    )

ora_results_all = pd.concat(ora_results_by_dataset.values(), axis=0).reset_index(drop=True)
ora_relevant_terms_all = pd.concat(ora_relevant_terms_by_dataset.values(), axis=0).reset_index(drop=True)

### Interpretation of ORA results

The ORA results provide a pathway-level validation of the Hypoxia-associated genes selected by Logistic Regression. Overall, the two SmartSeq datasets show the clearest and most biologically coherent enrichment patterns. In **SmartSeq_normalised**, the top enriched terms include `HALLMARK_HYPOXIA`, `Glycolysis`, `HIF-1 signaling pathway`, `Cellular Response To Hypoxia` and oxygen-response processes. This indicates that the genes selected by the model are not random predictive markers, but belong to biological pathways directly related to hypoxic adaptation.

The **SmartSeq_QCd** results are very similar and confirm that our custom preprocessing pipeline preserves the core hypoxia signal. The enrichment of `HALLMARK_HYPOXIA`, `Glycolysis`, `mTORC1 Signaling`, `HIF-1 signaling pathway` and oxygen-response terms shows that the QCd model also relies on biologically meaningful genes. This supports the idea that the QCd preprocessing strategy maintains both predictive and biological information.

The **DropSeq_normalised** ORA results are more complex. The top enriched terms are dominated by translation and protein-synthesis-related processes, suggesting that the strongest positive genes in DropSeq also capture broader cellular or technology-related signals. However, `HALLMARK_HYPOXIA` and glycolysis-related terms are still significantly enriched. Therefore, DropSeq does contain a hypoxia-related biological signal, but this signal is less dominant and less cleanly separated from other cellular programmes than in the SmartSeq datasets.

Overall, ORA confirms that the Logistic Regression models, especially in SmartSeq_normalised and SmartSeq_QCd, recover pathways that are highly consistent with hypoxia biology. DropSeq shows a weaker and more mixed enrichment profile, but still retains evidence of hypoxia-related transcriptional activity.

### Gene Set Enrichment Analysis on ranked Logistic Regression genes

GSEA avoids choosing a hard cutoff for the top genes. Instead, it uses the full ranked gene list and tests whether the genes belonging to a pathway tend to accumulate at the top or bottom of the ranking. Here, genes are ranked by their Logistic Regression coefficient, from the most Hypoxia-associated genes to the most Normoxia-associated genes.

For a gene set \(S\), GSEA walks down the ranked list and increases a running sum when a gene belongs to \(S\), while decreasing it when it does not. The maximum deviation from zero gives the enrichment score \(ES\). The normalized enrichment score \(NES\) adjusts for gene set size, and the FDR q-value controls for multiple testing. Positive NES values indicate pathways enriched among Hypoxia-associated genes, while negative NES values indicate enrichment among Normoxia-associated genes.

In [ ]:
# Run preranked GSEA using Logistic Regression coefficients.
GSEA_GENE_SETS = "MSigDB_Hallmark_2020"
GSEA_PERMUTATIONS = 100


def prepare_prerank_table(coef_df):
    """Prepare a two-column ranked gene table for GSEApy prerank."""
    ranking = (
        coef_df[["gene", "coefficient"]]
        .dropna()
        .groupby("gene", as_index=False)["coefficient"]
        .mean()
        .sort_values("coefficient", ascending=False)
    )
    return ranking


def run_gsea_prerank(label, coef_df):
    """Run preranked GSEA on Logistic Regression coefficients."""
    ranking = prepare_prerank_table(coef_df)
    pre_res = gp.prerank(
        rnk=ranking,
        gene_sets=GSEA_GENE_SETS,
        permutation_num=GSEA_PERMUTATIONS,
        outdir=None,
        seed=RANDOM_STATE,
        min_size=5,
        max_size=500,
        verbose=False,
    )

    results = pre_res.res2d.copy()
    results.insert(0, "dataset_family", label)
    return results


gsea_results_by_dataset = {}

for label, coef_df in pathway_gene_coefficients.items():
    print(f"Running preranked GSEA for {label}...")
    gsea_results = run_gsea_prerank(label, coef_df)
    gsea_results_by_dataset[label] = gsea_results

    pval_col = "FDR q-val" if "FDR q-val" in gsea_results.columns else "FDR qval"
    print(f"Top GSEA terms for {label}:")
    display(gsea_results.sort_values(pval_col).head(15))

    term_column = "Term" if "Term" in gsea_results.columns else "Name"
    hypoxia_rows = gsea_results[
        gsea_results[term_column].astype(str).str.upper().str.contains("HYPOXIA", na=False)
    ]
    print(f"Hypoxia-specific GSEA result for {label}:")
    display(hypoxia_rows)

gsea_results_all = pd.concat(gsea_results_by_dataset.values(), axis=0).reset_index(drop=True)

### Interpretation of GSEA results

GSEA confirms the main biological message of the pathway analysis, using the full ranked list of Logistic Regression coefficients instead of only a fixed set of top genes. A positive NES indicates that a pathway is enriched among genes with positive coefficients, meaning that it is associated with the Hypoxia class.

The results show that **HALLMARK_HYPOXIA** is significantly enriched in all three dataset families. This is a strong result because it means that hypoxia-related genes are not only present among a few top features, but are systematically shifted toward the Hypoxia-associated side of the ranking. Therefore, the models are capturing a coordinated hypoxia programme rather than isolated gene effects.

For **SmartSeq_normalised**, GSEA confirms a clean and biologically coherent signal. `HALLMARK_HYPOXIA`, `Glycolysis` and `mTORC1 Signaling` are enriched in the Hypoxia direction, supporting the idea that the model learns a canonical hypoxic response involving metabolic adaptation and oxygen-response pathways.

For **SmartSeq_QCd**, the enrichment is also strong. The hypoxia gene set is concentrated very early in the ranked list, meaning that many of the most positive coefficients belong to hypoxia-related genes. This confirms that our preprocessing pipeline preserves the biological structure of the hypoxia response.

For **DropSeq_normalised**, GSEA is particularly informative. Although the overlap with individual known hypoxia genes was weaker, `HALLMARK_HYPOXIA` is still significantly enriched and has a strong positive NES. This shows that DropSeq does contain a real hypoxia-related signal, but this signal is more distributed and less evident when looking only at single canonical genes.

Overall, GSEA provides the strongest pathway-level confirmation that the supervised models are biologically meaningful. The SmartSeq datasets show the clearest and most canonical hypoxia programme, while DropSeq shows a more complex but still significant hypoxia-associated transcriptional pattern.

### Summary and limitations of the pathway analysis

This pathway analysis connects the supervised classification results to biological interpretation. The main idea was to use Logistic Regression as an interpretable model and analyse the genes that most strongly push the prediction towards Hypoxia. This was done for the three combined datasets: SmartSeq_normalised, SmartSeq_QCd, and DropSeq_normalised. The comparison allows us to evaluate whether the hypoxia-related signal is stable across preprocessing strategies and sequencing technologies.

The first level of interpretation is gene-based. By comparing top Logistic Regression genes with known hypoxia and HIF target genes, we can check whether the model recovers canonical markers of hypoxic adaptation, such as genes involved in glycolysis, angiogenesis, pH regulation, cell survival and invasion. The second level is pathway-based. ORA tests whether the top model-selected genes are enriched in known biological pathways, while GSEA tests whether hypoxia-related pathways are systematically shifted across the full ranked list of coefficients.

There are some important limitations. First, we analysed only the combined datasets. This keeps the section concise and focuses on the most general models, but it does not capture possible cell-line-specific pathway differences between MCF7 and HCC1806. Second, we used Logistic Regression because it is directly interpretable at the gene level. More complex models such as MLP, SVM RBF or stacking may perform very well, but their predictions are harder to map back to individual genes without additional interpretability methods. Third, ORA depends on the choice of the number of top genes, while GSEA depends on the ranking metric and on the gene-set databases used. Therefore, pathway analysis should be interpreted as biological support for the supervised results, not as an independent proof of mechanism.

Overall, this section provides a biological sanity check for the supervised learning pipeline. If the extracted genes and enriched pathways are consistent with known hypoxia biology, then the model is not only accurate, but also biologically meaningful. If the three dataset families highlight different genes or pathways, this can help explain why some models generalise well across cell lines and technologies while others remain more dataset-specific.

## Final anonymous-test predictions

The final prediction step uses stacking models where they are available. For SmartSeq_normalised and DropSeq_normalised, the stackers are retrained on all labelled cells of each cell line and then applied to the corresponding anonymous test files. The anonymous files are not used during model selection or evaluation; they are used only here to generate the final predicted labels.

For SmartSeq_QCd, anonymous predictions can be generated only if matching anonymous QCd count matrices are available. If they are not provided, the notebook skips that part and reports it explicitly. This avoids applying a preprocessing pipeline to an incompatible anonymous matrix.

In [ ]:
# Generate anonymous-test predictions using the best stacking models where available.

FINAL_OUTPUT_DIR = Path(DATA_DIR) / "supervised_predictions"
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DROPSEQ_FINAL_OUTPUT_DIR = Path(DROPSEQ_DATA_DIR) / "supervised_predictions" if "DROPSEQ_DATA_DIR" in globals() else None
if DROPSEQ_FINAL_OUTPUT_DIR is not None:
    DROPSEQ_FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def fit_stack_and_predict_anonymous(stack_model, X_labelled, y_labelled, X_anon, output_path):
    """Fit a stacking model on all labelled cells and predict anonymous labels."""
    common = X_labelled.columns.intersection(X_anon.columns)
    if len(common) == 0:
        raise ValueError("No common genes between labelled and anonymous data.")

    fitted_stack = clone(stack_model)
    fitted_stack.fit(X_labelled[common], y_labelled)

    y_pred_num = fitted_stack.predict(X_anon[common])
    y_pred_label = pd.Series(y_pred_num).map(INV_LABEL_MAP).values
    y_score = get_prediction_scores(fitted_stack, X_anon[common])

    predictions = pd.DataFrame({
        "Cell_ID": X_anon.index,
        "Predicted_Condition": y_pred_label,
    })

    if y_score is not None:
        predictions["Hypoxia_score"] = y_score

    predictions.to_csv(output_path, sep="	", index=False)
    return predictions, fitted_stack


final_stacking_predictions = {}

# SmartSeq_normalised anonymous predictions.
if RUN_STACKING and "HCC1806" in stacking_models and "MCF7" in stacking_models:
    smartseq_prediction_specs = [
        ("SmartSeq_normalised_HCC1806", stacking_models["HCC1806"], X_hcc, y_hcc, X_hcc_anon, FINAL_OUTPUT_DIR / "SmartSeq_HCC1806_stacking_predictions.tsv"),
        ("SmartSeq_normalised_MCF7", stacking_models["MCF7"], X_mcf, y_mcf, X_mcf_anon, FINAL_OUTPUT_DIR / "SmartSeq_MCF7_stacking_predictions.tsv"),
    ]

    for label, stack_model, X_lab, y_lab, X_anon, out_path in smartseq_prediction_specs:
        predictions, fitted_stack = fit_stack_and_predict_anonymous(stack_model, X_lab, y_lab, X_anon, out_path)
        final_stacking_predictions[label] = predictions
        print(f"Saved {label} predictions:", out_path)
        display(predictions.head())
else:
    print("SmartSeq stacking predictions skipped: stacking models are not available.")

# SmartSeq_QCd predictions are possible only if QCd anonymous count matrices are provided.
if (
    "qcd_stacking_models" in globals()
    and "HCC1806_QCd" in qcd_stacking_models
    and "MCF7_QCd" in qcd_stacking_models
    and "X_hcc_qcd_anon" in globals()
    and "X_mcf_qcd_anon" in globals()
):
    qcd_prediction_specs = [
        ("SmartSeq_QCd_HCC1806", qcd_stacking_models["HCC1806_QCd"], X_hcc_qcd, y_hcc_qcd, X_hcc_qcd_anon, FINAL_OUTPUT_DIR / "SmartSeq_QCd_HCC1806_stacking_predictions.tsv"),
        ("SmartSeq_QCd_MCF7", qcd_stacking_models["MCF7_QCd"], X_mcf_qcd, y_mcf_qcd, X_mcf_qcd_anon, FINAL_OUTPUT_DIR / "SmartSeq_QCd_MCF7_stacking_predictions.tsv"),
    ]

    for label, stack_model, X_lab, y_lab, X_anon, out_path in qcd_prediction_specs:
        predictions, fitted_stack = fit_stack_and_predict_anonymous(stack_model, X_lab, y_lab, X_anon, out_path)
        final_stacking_predictions[label] = predictions
        print(f"Saved {label} predictions:", out_path)
        display(predictions.head())
else:
    print("SmartSeq_QCd anonymous predictions skipped: QCd anonymous count matrices are not available in this notebook.")

# DropSeq_normalised anonymous predictions.
if (
    "dropseq_stacking_models" in globals()
    and "DropSeq_HCC1806" in dropseq_stacking_models
    and "DropSeq_MCF7" in dropseq_stacking_models
):
    dropseq_prediction_specs = [
        ("DropSeq_normalised_HCC1806", dropseq_stacking_models["DropSeq_HCC1806"], X_hcc_dropseq, y_hcc_dropseq, X_hcc_dropseq_anon, DROPSEQ_FINAL_OUTPUT_DIR / "DropSeq_HCC1806_stacking_predictions.tsv"),
        ("DropSeq_normalised_MCF7", dropseq_stacking_models["DropSeq_MCF7"], X_mcf_dropseq, y_mcf_dropseq, X_mcf_dropseq_anon, DROPSEQ_FINAL_OUTPUT_DIR / "DropSeq_MCF7_stacking_predictions.tsv"),
    ]

    for label, stack_model, X_lab, y_lab, X_anon, out_path in dropseq_prediction_specs:
        predictions, fitted_stack = fit_stack_and_predict_anonymous(stack_model, X_lab, y_lab, X_anon, out_path)
        final_stacking_predictions[label] = predictions
        print(f"Saved {label} predictions:", out_path)
        display(predictions.head())
else:
    print("DropSeq stacking predictions skipped: DropSeq stacking models are not available.")

## Final conclusions

The supervised learning section evaluates whether normoxic and hypoxic cancer cells can be distinguished from single-cell gene-expression profiles, and whether the learned signal generalises across cell lines and sequencing technologies. To do this, we compared three main dataset versions: the professor-provided **SmartSeq_normalised** data, our own **SmartSeq_QCd** preprocessing pipeline, and the larger **DropSeq_normalised** dataset. For each version, we evaluated models within the same cell line, on the combined dataset, and across cell lines.

Overall, the results show that hypoxia is clearly predictable from gene-expression data. The strongest and most stable results were obtained on **SmartSeq_normalised**, where the models performed extremely well not only within each cell line, but also in cross-cell-line evaluation. This suggests that the professor-normalised SmartSeq data contain a robust hypoxia transcriptional signal that is shared, at least in part, between MCF7 and HCC1806. The **SmartSeq_QCd** pipeline also preserved the hypoxia signal very well, reaching excellent within-cell-line performance, although its cross-cell-line results were slightly weaker. This indicates that our preprocessing strategy is valid, but the professor-provided filtering and normalisation appear to produce a more transferable representation.

The **DropSeq_normalised** dataset showed a different behaviour. Within the same cell line, and especially in the combined DropSeq setting, the models still reached strong performance. However, cross-cell-line evaluation was much weaker, indicating that in DropSeq the hypoxia signal does not transfer well between MCF7 and HCC1806. This suggests that the larger number of cells helps models learn strong within-distribution patterns, but it does not automatically solve the biological differences between cell lines. In this context, non-linear models such as SVM RBF, MLP and ensemble methods were more useful than in SmartSeq, likely because the larger dataset provides enough observations to support more flexible decision boundaries. In contrast, in the smaller SmartSeq datasets, simpler linear models were already sufficient and less prone to overfitting.

The stacking analysis confirmed that ensemble learning is not always beneficial. In **SmartSeq_normalised**, stacking added almost nothing because the base models were already highly accurate and strongly correlated, meaning that they were learning nearly the same decision rule. In **SmartSeq_QCd**, stacking produced a small improvement, probably because Random Forest contributed slightly less redundant predictions. In **DropSeq_normalised**, stacking was more useful: the combined DropSeq stacker slightly improved over the best individual model, mainly because Random Forest provided complementary information compared with SVM RBF and MLP. However, stacking improved mainly within-distribution performance and did not resolve the weak cross-cell-line transfer observed in DropSeq.

The cross-technology transfer experiment further showed that models trained on SmartSeq do not generalise well to DropSeq, and vice versa. The performance was close to a weak baseline, suggesting that the two sequencing technologies generate substantially different expression distributions and technical noise structures. This does not mean that hypoxia is not present in both technologies, but rather that a model trained on one technology learns technology-specific patterns that cannot be directly transferred to the other without dedicated integration or batch-correction methods.

The pathway analysis adds biological support to the supervised learning results. By interpreting Logistic Regression coefficients on the three combined datasets, we found that the SmartSeq models select several canonical hypoxia-related genes, including genes involved in HIF response, glycolysis and oxygen-level adaptation. ORA and GSEA further confirmed that `HALLMARK_HYPOXIA`, `Glycolysis` and related oxygen-response pathways are enriched among the model-associated genes.

The DropSeq results were less clear at the single-gene level, with fewer canonical hypoxia genes appearing in the direct overlap analysis. However, GSEA still showed significant enrichment of `HALLMARK_HYPOXIA`, suggesting that DropSeq also contains a coordinated hypoxia signal, although it is more distributed and mixed with other biological or technology-related effects. Overall, this confirms that the models are not only predictive, but also biologically meaningful.

In conclusion, the supervised learning results support the main biological hypothesis: hypoxic cancer cells can be identified from single-cell gene expression. The most reliable performance was obtained when training and testing within the same dataset structure, especially with the professor-normalised SmartSeq data. Cross-cell-line and cross-technology evaluations were more challenging and revealed important biological and technical limitations. Therefore, while the models are highly effective for hypoxia classification within controlled settings, true generalisation across cell lines and sequencing technologies remains more difficult and would require further preprocessing, domain adaptation or biological harmonisation strategies.

---

## Final Conclusions

The complete workflow moves from data understanding to unsupervised discovery and finally to supervised prediction. The EDA confirms that the datasets have the typical structure of single-cell RNA-seq data: many zeros, highly skewed gene-expression distributions and quality-control issues that must be handled before modelling. The comparison with the professor-provided filtered data is important because it shows whether our filtering preserves the same gene-expression structure while applying our own criteria.

From a biological point of view, the unsupervised analysis is especially informative. Hypoxia is not only learned by classifiers after labels are provided: it also appears as a relevant axis of variation in PCA, clustering, Buffa metagene scoring and NMF. The signal is shared across HCC1806 and MCF7, but it is not identical. MCF7 tends to show cleaner separation between normoxia and hypoxia, while HCC1806 may emphasize a more canonical HIF/glycolytic hypoxia program. This suggests a conserved hypoxic core with cell-line-specific transcriptional responses.

The supervised analysis then tests this signal predictively. Within-line and combined models evaluate whether hypoxic cells can be recognized from expression profiles, while cross-line and cross-technology scenarios test whether the learned signature generalizes. The pathway analysis of Logistic Regression genes connects model decisions to biology, and should be interpreted together with the NMF pathway analysis: agreement between the two supports a robust hypoxia interpretation, while differences highlight the distinction between co-expression programs and predictive biomarkers.

The main limitations are the relatively small number of SmartSeq cells compared with the dimensionality of the gene-expression space, the strong difference between cell lines, and the technology-specific behaviour expected when comparing SmartSeq and DropSeq. A stronger future version of the analysis could include explicit batch/cell-line correction, stability analysis of selected genes, external validation on independent hypoxia datasets, and a deeper pathway-level comparison of shared versus cell-line-specific hypoxia programs.

Overall, the project supports the central biological conclusion that hypoxia leaves a measurable transcriptional footprint in cancer single cells. This footprint is strong enough to be detected by unsupervised methods and exploited by supervised classifiers, but its exact gene-level form depends on the cellular context and on the sequencing/preprocessing pipeline.